In [ ]:
# Cell 1 — Imports, Configuration & System Resource Verification
import copy, gc, hashlib, json, os, platform, random, shutil, sys, time, warnings, psutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                              f1_score, precision_score, recall_score,
                              roc_auc_score, matthews_corrcoef)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
import shap
from art.attacks.evasion import DeepFool

warnings.filterwarnings("once")

CONFIG = {
    "experiment_name"             : "V9_FINAL_PRODUCTION",
    "data_dir"                    : "data",
    "output_dir"                  : "outputs/v9_final",
    "seeds"                       : [42, 123, 2024],
    "data_seed"                   : 42,
    "test_size"                   : 0.20,
    "dev_fraction_of_train"       : 0.10,
    "mi_variance_threshold"       : 1e-5,
    "mi_sample_size"              : 100_000,
    "paper_mi_top_k"              : 40,
    "paper_shap_top_k"            : 15,
    "extension_shap_cumulative"   : 0.80,
    "shap_background_samples"     : 200,
    "shap_eval_samples"           : 500,
    "smote_strategy"              : 1.0,
    "hidden_layers"               : [128, 64, 32],
    "wider_layers"                : [256, 128, 64],
    "dropout"                     : 0.30,
    "learning_rate"               : 0.001,
    "batch_size"                  : 2048,   # Conservative batch size for RX 7600 8GB
    "baseline_epochs"             : 50,
    "paper_epochs"                : 50,
    "v9_clean_epochs"             : 10,
    "v9_online_epochs"            : 40,
    "attack_epsilon"              : 0.05,
    "pgd_steps_eval"              : 100,
    "pgd_steps_online"            : 5,
    "pgd_step_ratio"              : 0.10,
    "pgd_restarts"                : 1,
    "deepfool_steps"              : 50,
    "deepfool_eval_n"             : 500,
    "eval_n"                      : 2000,
    "square_eval_n"               : 1000,
    "square_iterations"           : 100,    # GPU-vectorized fast iteration budget
    "square_init_frac"            : 0.30,
    "paper_fgsm_n"                : 10_000, # Static FGSM augmentation count
    "paper_pgd_n"                 : 10_000, # Static PGD augmentation count
    "paper_deepfool_n"            : 1_000,  # Static DeepFool augmentation count
    "online_adv_weight"           : 0.50,
    "online_clean_weight"         : 0.50,
    "transfer_n"                  : 1000,
    "surrogate_epochs"            : 30,
    "pgd_sweep_steps"             : [20, 50, 100, 200],
    "eps_sweep"                   : [0.01, 0.03, 0.05, 0.10],
    "boundary_n"                  : 300,
    "boundary_init_trials"        : 30,     # Bounded random search trials
    "boundary_refine_steps"       : 30,     # Bounded binary search steps
    "min_disk_free_gb"            : 5.0,
    "min_ram_free_gb"             : 2.0,
    "immutable_cols"              : [],
}

print("=" * 70)
print(f"  EXPERIMENT: {CONFIG['experiment_name']}")
print(f"  OUTPUT DIR: {CONFIG['output_dir']}")
print(f"  SEEDS     : {CONFIG['seeds']}")
print(f"  BATCH SIZE: {CONFIG['batch_size']} (Safe for RX 7600 8GB)")
print("=" * 70)

In [ ]:
# Cell 2 — Environment & GPU Verification (ROCm/CUDA Device)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def check_resources(stage_name=""):
    """Monitors RAM, VRAM, and Disk space before expensive operations."""
    vm = psutil.virtual_memory()
    disk = psutil.disk_usage(os.path.abspath(CONFIG["output_dir"] if os.path.exists(CONFIG["output_dir"]) else "."))
    ram_free_gb = vm.available / (1024**3)
    disk_free_gb = disk.free / (1024**3)
    
    print(f"[{stage_name or 'RESOURCE CHECK'}] RAM Free: {ram_free_gb:.1f} GB | Disk Free: {disk_free_gb:.1f} GB", end="")
    if DEVICE.type == "cuda":
        vram_alloc = torch.cuda.memory_allocated(0) / (1024**3)
        vram_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f" | VRAM: {vram_alloc:.2f}/{vram_total:.1f} GB used")
    else:
        print()
        
    if disk_free_gb < CONFIG["min_disk_free_gb"]:
        raise RuntimeError(f"LOW DISK SPACE: {disk_free_gb:.1f} GB available (minimum {CONFIG['min_disk_free_gb']} GB required).")
    if ram_free_gb < CONFIG["min_ram_free_gb"]:
        print(f"  [WARN] Low RAM: {ram_free_gb:.1f} GB free.")

print("=" * 60)
print("  ENVIRONMENT VERIFICATION")
print("=" * 60)
print(f"  Python           : {sys.version.split()[0]}")
print(f"  PyTorch          : {torch.__version__}")
try:
    print(f"  ROCm / HIP       : {torch.version.hip}")
except Exception:
    print(f"  CUDA             : {torch.version.cuda}")
print(f"  CUDA Available   : {torch.cuda.is_available()}")
print(f"  Selected Device  : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU Device Name  : {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"  Total VRAM       : {props.total_memory / 1024**3:.2f} GB")
else:
    print("  [WARN] GPU not available. Running on CPU.")
print("=" * 60)

def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(CONFIG["data_seed"])
check_resources("STARTUP")
print(f"[OK] Global seed set to {CONFIG['data_seed']}")

In [ ]:
# Cell 3 — Directory Setup, Stage Manager & Atomic Save Utilities
OUTPUT_DIR   = Path(CONFIG["output_dir"])
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
STATUS_DIR   = OUTPUT_DIR / "status"
CONFIG_DIR   = OUTPUT_DIR / "configs"
for d in [OUTPUT_DIR, ARTIFACT_DIR, STATUS_DIR, CONFIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Atomic filesystem save utilities (guaranteed power-cut safety)
def atomic_save_numpy(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp.npz")
    np.savez_compressed(tmp, **arrays)
    tmp.replace(path)

def atomic_save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp.json")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)
    tmp.replace(path)

def atomic_save_checkpoint(ckpt_dict, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp.pt")
    torch.save(ckpt_dict, tmp)
    tmp.replace(path)

def atomic_save_df(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp.csv")
    df.to_csv(tmp, index=False)
    tmp.replace(path)

def validate_checkpoint(path, required_keys, expected_dim=None, expected_seed=None):
    path = Path(path)
    if not path.exists():
        print(f"  [FAIL] Checkpoint not found: {path}")
        return False
    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        for k in required_keys:
            if k not in ckpt:
                print(f"  [FAIL] Missing key '{k}' in {path.name}")
                return False
        if expected_seed is not None and ckpt.get("seed") != expected_seed:
            print(f"  [FAIL] Seed mismatch in {path.name}: expected {expected_seed}, got {ckpt.get('seed')}")
            return False
        if expected_dim is not None and ckpt.get("feature_count") != expected_dim:
            print(f"  [FAIL] Feature count mismatch in {path.name}: expected {expected_dim}, got {ckpt.get('feature_count')}")
            return False
        print(f"  [LOAD] Checkpoint valid: {path.name}")
        return True
    except Exception as e:
        print(f"  [FAIL] Checkpoint corrupted ({path.name}): {e}")
        return False

class StageManager:
    def __init__(self, base_dir):
        self.status_dir = Path(base_dir) / "status"
        self.status_dir.mkdir(parents=True, exist_ok=True)

    def is_done(self, name):
        p = self.status_dir / f"{name}.json"
        if not p.exists():
            return False
        try:
            with open(p) as f:
                d = json.load(f)
            return d.get("status") == "complete"
        except Exception:
            return False

    def mark_done(self, name, meta=None):
        atomic_save_json({"status": "complete", "timestamp": time.time(), **(meta or {})},
                         self.status_dir / f"{name}.json")

    def report(self, name):
        if self.is_done(name):
            print(f"[SKIP] Stage '{name}' already complete — loading from disk")
            return True
        print(f"[RUN]  Stage '{name}' executing...")
        return False

SM = StageManager(OUTPUT_DIR)

# Save initial config snapshot
atomic_save_json(CONFIG, CONFIG_DIR / "config.json")
atomic_save_json(CONFIG, OUTPUT_DIR / "config.json")
_env_init = {
    "python": sys.version,
    "torch": torch.__version__,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu",
    "os": platform.platform(),
}
atomic_save_json(_env_init, CONFIG_DIR / "environment.json")
atomic_save_json(_env_init, OUTPUT_DIR / "environment.json")

print("[OK] Directories, atomic utilities, and StageManager initialized.")

In [ ]:
# Cell 4 — Dataset Discovery (Scans CSE-CIC-IDS2018)
DATA_DIR = Path(CONFIG["data_dir"])
csv_files = sorted(list(DATA_DIR.rglob("*.csv")) + list(DATA_DIR.rglob("*.parquet")))
print(f"[INFO] Scanning dataset directory: {DATA_DIR.resolve()}")
print(f"[INFO] Found {len(csv_files)} data file(s):")
for f in csv_files:
    size_mb = f.stat().st_size / 1024**2
    print(f"  {f.name}  ({size_mb:.1f} MB)")

if len(csv_files) == 0:
    raise FileNotFoundError(f"No CSV or Parquet files found in {DATA_DIR}. Check CONFIG['data_dir'].")

CSV_FILES = csv_files
print(f"\n[OK] Dataset discovery complete: {len(CSV_FILES)} file(s) queued for loading.")

In [ ]:
# Cell 5 — Dataset Loading & Disk Caching
pq_path  = ARTIFACT_DIR / "raw_data.parquet"
raw_meta = ARTIFACT_DIR / "raw_data_meta.json"

check_resources("PRE-LOADING")

if SM.report("raw_loading") and pq_path.exists() and raw_meta.exists():
    df_raw = pd.read_parquet(pq_path)
    with open(raw_meta) as f:
        raw_info = json.load(f)
    print(f"[LOAD] Raw data from cache: {len(df_raw):,} rows x {len(df_raw.columns)} cols")
else:
    t0 = time.perf_counter()
    chunks = []
    for fp in CSV_FILES:
        print(f"  Reading {fp.name} ...", end=" ")
        try:
            if fp.suffix == ".parquet":
                chunk = pd.read_parquet(fp)
            else:
                chunk = pd.read_csv(fp, low_memory=False)
            chunks.append(chunk)
            print(f"{len(chunk):,} rows")
        except Exception as e:
            print(f"ERROR reading {fp.name}: {e}")

    df_raw = pd.concat(chunks, ignore_index=True)
    elapsed = time.perf_counter() - t0
    print(f"\n[OK] Loaded {len(df_raw):,} rows x {len(df_raw.columns)} cols in {elapsed:.1f}s")

    df_raw.to_parquet(pq_path, index=False, compression="snappy")
    raw_info = {"rows": len(df_raw), "cols": len(df_raw.columns),
                "columns": list(df_raw.columns), "files": [str(f) for f in CSV_FILES]}
    atomic_save_json(raw_info, raw_meta)
    SM.mark_done("raw_loading", {"rows": len(df_raw)})
    print(f"[SAVE] Cached raw_data.parquet ({pq_path.stat().st_size/1024**2:.1f} MB)")

check_resources("POST-LOADING")

In [ ]:
# Cell 6 — Data Cleaning & Deduplication (With Safe Numeric Coercion)
clean_path = ARTIFACT_DIR / "data_clean.parquet"
clean_meta = ARTIFACT_DIR / "data_clean_meta.json"

check_resources("PRE-CLEANING")

if SM.report("cleaning") and clean_path.exists() and clean_meta.exists():
    df_clean = pd.read_parquet(clean_path)
    with open(clean_meta) as f:
        clean_info = json.load(f)
    print(f"[LOAD] Cleaned data from cache: {len(df_clean):,} rows x {len(df_clean.columns)} cols")
else:
    t0 = time.perf_counter()
    df = df_raw.copy()

    # Drop identifier / metadata columns (prevent artificial leakage)
    id_cols = [c for c in df.columns if c.lower().strip() in
               ("flow id", "flow_id", "src ip", "src_ip", "dst ip", "dst_ip",
                "source ip", "source_ip", "destination ip", "destination_ip",
                "src port", "src_port", "dst port", "dst_port",
                "timestamp", "flowid", "srcip", "dstip", "srcport", "dstport")]
    df.drop(columns=id_cols, inplace=True, errors="ignore")
    print(f"[INFO] Dropped {len(id_cols)} identifier cols")

    # Coerce numeric columns and convert corrupted strings to NaN
    for col in df.columns:
        if col.lower().strip() not in ("label", "class", "attack type", "attack_type", "protocol"):
            if df[col].dtype == object:
                df[col] = pd.to_numeric(df[col], errors="coerce")

    # Replace inf with nan, drop NaNs
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    rows_before = len(df)
    df.dropna(inplace=True)
    print(f"[INFO] Dropped {rows_before - len(df):,} rows with NaN/Inf")

    # Drop constant columns (std == 0 or nunique <= 1)
    const_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                  if (df[c].std() == 0) or (df[c].nunique() <= 1)]
    df.drop(columns=const_cols, inplace=True)
    print(f"[INFO] Dropped {len(const_cols)} constant columns")

    # Drop duplicates
    rows_before = len(df)
    df.drop_duplicates(inplace=True)
    print(f"[INFO] Dropped {rows_before - len(df):,} duplicate rows")

    df_clean = df.reset_index(drop=True)
    elapsed = time.perf_counter() - t0

    df_clean.to_parquet(clean_path, index=False, compression="snappy")
    clean_info = {"rows_after_cleaning": len(df_clean),
                  "cols_after_cleaning": len(df_clean.columns),
                  "rows_raw": len(df_raw)}
    atomic_save_json(clean_info, clean_meta)
    SM.mark_done("cleaning", clean_info)
    print(f"[OK] Cleaning done in {elapsed:.1f}s: {len(df_clean):,} rows remain")

CLEAN_ROWS = len(df_clean)
if "df_raw" in dir():
    del df_raw
    gc.collect()

In [ ]:
# Cell 7 — Target Column Creation (Binary Classification)
label_col_candidates = ["label", "Label", "class", "Class",
                         "attack type", "Attack Type", "attack_type", " Label"]
label_col = None
for c in label_col_candidates:
    if c in df_clean.columns:
        label_col = c
        break
if label_col is None:
    raise RuntimeError(f"Cannot find label column. Columns: {list(df_clean.columns[-5:])}")

print(f"[INFO] Label column detected: '{label_col}'")
print(f"[INFO] Raw class distribution:\n{df_clean[label_col].value_counts().head(10)}")

df_clean["target"] = (df_clean[label_col].astype(str).str.strip().str.lower() != "benign").astype(np.int8)
print(f"\n[OK] Binary target created:")
print(f"  Benign (0): {(df_clean['target']==0).sum():,}")
print(f"  Attack (1): {(df_clean['target']==1).sum():,}")

LABEL_COL = label_col

In [ ]:
# Cell 8 — Stratified Train / Test Split (Strict Test Isolation)
split_path = ARTIFACT_DIR / "train_test_split.npz"

if SM.report("train_test_split") and split_path.exists():
    loaded = np.load(split_path)
    train_idx = loaded["train_idx"]
    test_idx  = loaded["test_idx"]
    print(f"[LOAD] Train: {len(train_idx):,}  Test: {len(test_idx):,}")
else:
    y_all = df_clean["target"].values
    train_idx, test_idx = train_test_split(
        np.arange(len(df_clean)),
        test_size=CONFIG["test_size"],
        random_state=CONFIG["data_seed"],
        stratify=y_all
    )
    atomic_save_numpy(split_path, train_idx=train_idx, test_idx=test_idx)
    SM.mark_done("train_test_split",
                 {"train_n": len(train_idx), "test_n": len(test_idx)})
    print(f"[OK] Train: {len(train_idx):,}  Test: {len(test_idx):,}")

FEATURE_COLS = [c for c in df_clean.columns if c not in (LABEL_COL, "target")]
df_train_raw = df_clean.iloc[train_idx].reset_index(drop=True)
df_test_raw  = df_clean.iloc[test_idx].reset_index(drop=True)
print(f"[INFO] Raw features count: {len(FEATURE_COLS)}")

In [ ]:
# Cell 9 — Train / Dev Split (Dev Set for Model Validation Only)
devsp_path = ARTIFACT_DIR / "dev_split.npz"

if SM.report("dev_split") and devsp_path.exists():
    loaded = np.load(devsp_path)
    tr_idx = loaded["tr_idx"]
    dv_idx = loaded["dv_idx"]
    print(f"[LOAD] Core-train: {len(tr_idx):,}  Dev: {len(dv_idx):,}")
else:
    y_tr = df_train_raw["target"].values
    tr_idx, dv_idx = train_test_split(
        np.arange(len(df_train_raw)),
        test_size=CONFIG["dev_fraction_of_train"],
        random_state=CONFIG["data_seed"],
        stratify=y_tr
    )
    atomic_save_numpy(devsp_path, tr_idx=tr_idx, dv_idx=dv_idx)
    SM.mark_done("dev_split", {"train_n": len(tr_idx), "dev_n": len(dv_idx)})
    print(f"[OK] Core-train: {len(tr_idx):,}  Dev: {len(dv_idx):,}")

df_tr  = df_train_raw.iloc[tr_idx].reset_index(drop=True)
df_dev = df_train_raw.iloc[dv_idx].reset_index(drop=True)
y_train_raw = df_tr["target"].values.astype(np.int64)
y_dev_raw   = df_dev["target"].values.astype(np.int64)
y_test_raw  = df_test_raw["target"].values.astype(np.int64)

In [ ]:
# Cell 10 — One-Hot Encoding (Fitted on Train Only)
enc_path = ARTIFACT_DIR / "encoded.npz"

if SM.report("encoding") and enc_path.exists():
    loaded = np.load(enc_path, allow_pickle=True)
    X_train_enc = loaded["X_train"]
    X_dev_enc   = loaded["X_dev"]
    X_test_enc  = loaded["X_test"]
    enc_feat_names = list(loaded["feature_names"])
    print(f"[LOAD] Encoded feature count: {X_train_enc.shape[1]}")
else:
    t0 = time.perf_counter()
    # Robust partitioning into numeric vs non-numeric
    num_cols = df_tr[FEATURE_COLS].select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df_tr[FEATURE_COLS].select_dtypes(exclude=[np.number]).columns.tolist()
    print(f"[INFO] Numeric cols: {len(num_cols)}  Categorical cols: {len(cat_cols)}")

    X_tr_num  = df_tr[num_cols].values.astype(np.float32)
    X_dev_num = df_dev[num_cols].values.astype(np.float32)
    X_te_num  = df_test_raw[num_cols].values.astype(np.float32)
    feat_names = num_cols[:]

    if cat_cols:
        ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore", dtype=np.float32)
        ohe.fit(df_tr[cat_cols])
        X_tr_cat  = ohe.transform(df_tr[cat_cols])
        X_dev_cat = ohe.transform(df_dev[cat_cols])
        X_te_cat  = ohe.transform(df_test_raw[cat_cols])
        ohe_names = list(ohe.get_feature_names_out(cat_cols))
        X_train_enc = np.hstack([X_tr_num, X_tr_cat])
        X_dev_enc   = np.hstack([X_dev_num, X_dev_cat])
        X_test_enc  = np.hstack([X_te_num, X_te_cat])
        feat_names  = feat_names + ohe_names
    else:
        X_train_enc, X_dev_enc, X_test_enc = X_tr_num, X_dev_num, X_te_num

    enc_feat_names = feat_names
    atomic_save_numpy(enc_path,
                      X_train=X_train_enc, X_dev=X_dev_enc, X_test=X_test_enc,
                      feature_names=np.array(enc_feat_names, dtype=object))
    SM.mark_done("encoding", {"n_features": len(enc_feat_names)})
    print(f"[OK] Encoding completed in {time.perf_counter()-t0:.1f}s: {X_train_enc.shape}")

ALL_FEATURE_NAMES = enc_feat_names
print(f"[INFO] Total features after encoding: {len(ALL_FEATURE_NAMES)}")

In [ ]:
# Cell 11 — Mutual Information Feature Selection (Top 40 from Train)
mi_path     = ARTIFACT_DIR / "mi_features.json"
mi_idx_path = ARTIFACT_DIR / "mi_indices.npz"

if SM.report("mi_selection") and mi_path.exists() and mi_idx_path.exists():
    with open(mi_path) as f:
        mi_info = json.load(f)
    paper_mi_features = mi_info["paper_mi_features"]
    mi_idx = list(np.load(mi_idx_path)["mi_idx"])
    print(f"[LOAD] Top MI features: {len(paper_mi_features)}")
else:
    t0 = time.perf_counter()
    k = CONFIG["paper_mi_top_k"]
    n = len(X_train_enc)
    if n > CONFIG["mi_sample_size"]:
        rng = np.random.RandomState(CONFIG["data_seed"])
        idx_sub = rng.choice(n, CONFIG["mi_sample_size"], replace=False)
        X_mi = X_train_enc[idx_sub]
        y_mi = y_train_raw[idx_sub]
        print(f"[INFO] MI computed on subsample of {CONFIG['mi_sample_size']:,} rows")
    else:
        X_mi, y_mi = X_train_enc, y_train_raw

    mi_scores = mutual_info_classif(X_mi, y_mi, random_state=CONFIG["data_seed"])
    mi_order  = np.argsort(mi_scores)[::-1]
    mi_idx    = mi_order[:k].tolist()
    paper_mi_features = [ALL_FEATURE_NAMES[i] for i in mi_idx]

    mi_info = {"paper_mi_features": paper_mi_features,
               "mi_idx": mi_idx,
               "mi_scores": {ALL_FEATURE_NAMES[i]: float(mi_scores[i]) for i in mi_order[:k]}}
    atomic_save_json(mi_info, mi_path)
    atomic_save_numpy(mi_idx_path, mi_idx=np.array(mi_idx))
    SM.mark_done("mi_selection", {"k": k})
    print(f"[OK] MI selection done in {time.perf_counter()-t0:.1f}s. Top feature: {paper_mi_features[0]}")

mi_idx = [int(i) for i in mi_idx]
print(f"[INFO] Top-{len(mi_idx)} MI features ready.")

In [ ]:
# Cell 12 — Feature Scaling & Attack Domain Bound Computation
scale_path = ARTIFACT_DIR / "scaled.npz"

if SM.report("scaling") and scale_path.exists():
    loaded = np.load(scale_path)
    X_train_mi  = loaded["X_train_mi"].astype(np.float32)
    X_dev_mi    = loaded["X_dev_mi"].astype(np.float32)
    X_test_mi   = loaded["X_test_mi"].astype(np.float32)
    X_train_all = loaded["X_train_all"].astype(np.float32)
    X_dev_all   = loaded["X_dev_all"].astype(np.float32)
    X_test_all  = loaded["X_test_all"].astype(np.float32)
    print(f"[LOAD] Scaled data ready. MI features shape: {X_train_mi.shape}")
else:
    t0 = time.perf_counter()
    scaler = StandardScaler()
    X_train_all = scaler.fit_transform(X_train_enc).astype(np.float32)
    X_dev_all   = scaler.transform(X_dev_enc).astype(np.float32)
    X_test_all  = scaler.transform(X_test_enc).astype(np.float32)

    X_train_mi = X_train_all[:, mi_idx]
    X_dev_mi   = X_dev_all[:, mi_idx]
    X_test_mi  = X_test_all[:, mi_idx]

    atomic_save_numpy(scale_path,
                      X_train_mi=X_train_mi, X_dev_mi=X_dev_mi, X_test_mi=X_test_mi,
                      X_train_all=X_train_all, X_dev_all=X_dev_all, X_test_all=X_test_all)
    SM.mark_done("scaling")
    print(f"[OK] Scaling done in {time.perf_counter()-t0:.1f}s")

y_train = y_train_raw.astype(np.int64)
y_dev   = y_dev_raw.astype(np.int64)
y_test  = y_test_raw.astype(np.int64)

# Feature min/max bounds for attack clipping
clip_min = X_train_mi.min(axis=0).astype(np.float32)
clip_max = X_train_mi.max(axis=0).astype(np.float32)
bounds   = (clip_min, clip_max)

# Structural mask: all features mutable (1.0)
structural_mask = np.ones(len(mi_idx), dtype=np.float32)
print(f"[INFO] Domain bounds & structural mask initialized (|mask|={structural_mask.sum():.0f}).")

In [ ]:
# Cell 13 — SMOTE Class Balancing (Fitted Strictly on Train)
smote_path = ARTIFACT_DIR / "smote.npz"

check_resources("PRE-SMOTE")
print(f"[INFO] Original train rows: {len(X_train_mi):,} (Benign: {(y_train==0).sum():,}, Attack: {(y_train==1).sum():,})")
print("[METHODOLOGY NOTE] SMOTE is applied after one-hot encoding; synthetic samples may contain interpolated fractional values.")

if SM.report("smote") and smote_path.exists():
    loaded = np.load(smote_path)
    X_train_bal = loaded["X_train_bal"].astype(np.float32)
    y_train_bal = loaded["y_train_bal"].astype(np.int64)
    print(f"[LOAD] SMOTE balanced train set: {X_train_bal.shape}")
else:
    t0 = time.perf_counter()
    sm = SMOTE(sampling_strategy=CONFIG["smote_strategy"],
               random_state=CONFIG["data_seed"])
    X_train_bal, y_train_bal = sm.fit_resample(X_train_mi, y_train)
    X_train_bal = X_train_bal.astype(np.float32)
    y_train_bal = y_train_bal.astype(np.int64)
    atomic_save_numpy(smote_path, X_train_bal=X_train_bal, y_train_bal=y_train_bal)
    SM.mark_done("smote", {"n_balanced": len(X_train_bal)})
    print(f"[OK] SMOTE done in {time.perf_counter()-t0:.1f}s")
    print(f"     Before: {len(X_train_mi):,}  After: {len(X_train_bal):,}")

# Save data summary JSON
data_summary = {
    "raw_rows":          int(CLEAN_ROWS) if "CLEAN_ROWS" in dir() else int(len(X_train_mi) + len(X_test_mi)),
    "rows_after_clean":  int(CLEAN_ROWS) if "CLEAN_ROWS" in dir() else int(len(X_train_mi) + len(X_test_mi)),
    "train_rows":        int(len(X_train_mi)),
    "dev_rows":          int(len(X_dev_mi)),
    "test_rows":         int(len(X_test_mi)),
    "train_bal_rows":    int(len(X_train_bal)),
    "mi_features":       int(len(mi_idx)),
    "all_features":      int(len(ALL_FEATURE_NAMES)),
}
atomic_save_json(data_summary, ARTIFACT_DIR / "data_summary.json")
atomic_save_json(data_summary, OUTPUT_DIR / "data_summary.json")
print("[SAVE] data_summary.json")
print(f"[INFO] Class balance after SMOTE: {np.bincount(y_train_bal)}")
check_resources("POST-SMOTE")

In [ ]:
# Cell 14 — Model Architectures, Pure-GPU Training Engine & OOM Recovery
import gc

class IDSNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        h1, h2, h3 = CONFIG["hidden_layers"]
        self.network = nn.Sequential(
            nn.Linear(input_dim, h1), nn.BatchNorm1d(h1), nn.ReLU(), nn.Dropout(CONFIG["dropout"]),
            nn.Linear(h1, h2),        nn.BatchNorm1d(h2), nn.ReLU(), nn.Dropout(CONFIG["dropout"]),
            nn.Linear(h2, h3),        nn.BatchNorm1d(h3), nn.ReLU(), nn.Dropout(CONFIG["dropout"]),
            nn.Linear(h3, 2)
        )
    def forward(self, x):
        return self.network(x)

class WiderNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        h1, h2, h3 = CONFIG["wider_layers"]
        self.network = nn.Sequential(
            nn.Linear(input_dim, h1), nn.BatchNorm1d(h1), nn.ReLU(), nn.Dropout(CONFIG["dropout"]),
            nn.Linear(h1, h2),        nn.BatchNorm1d(h2), nn.ReLU(), nn.Dropout(CONFIG["dropout"]),
            nn.Linear(h2, h3),        nn.BatchNorm1d(h3), nn.ReLU(), nn.Dropout(CONFIG["dropout"]),
            nn.Linear(h3, 2)
        )
    def forward(self, x):
        return self.network(x)

def make_loader_safe(X, y, batch_size, shuffle=True):
    """Create DataLoader; auto-reduces batch size on memory limit."""
    fallback = [2048, 1024, 512, 256, 128]
    bs = batch_size
    while True:
        try:
            ds = TensorDataset(
                torch.from_numpy(np.ascontiguousarray(X)).float(),
                torch.from_numpy(np.ascontiguousarray(y)).long()
            )
            return DataLoader(ds, batch_size=bs, shuffle=shuffle,
                              pin_memory=(DEVICE.type == "cuda"), num_workers=0)
        except (RuntimeError, MemoryError):
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
            idx = fallback.index(bs) + 1 if bs in fallback else len(fallback)
            if idx >= len(fallback):
                raise RuntimeError("OOM even at batch_size=128")
            bs = fallback[idx]
            print(f"  [OOM] Reducing batch size to {bs}")

def compute_metrics(y_true, y_pred, y_prob=None):
    """Compute classification metrics with safe error guards."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    m = {
        "accuracy":          float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "f1_weighted":       float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "f1_macro":          float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision":         float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "recall":            float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "mcc":               float(matthews_corrcoef(y_true, y_pred)),
    }
    if y_prob is not None and len(np.unique(y_true)) == 2:
        try:
            m["roc_auc"] = float(roc_auc_score(y_true, y_prob[:, 1]))
        except Exception:
            pass
    return m

def predict_model(model, X, batch_size=None):
    """Batch inference on numpy array. Returns (preds, probs)."""
    bs = batch_size or CONFIG["batch_size"]
    model.eval()
    all_logits = []
    with torch.no_grad():
        for start in range(0, len(X), bs):
            xb = torch.from_numpy(np.ascontiguousarray(X[start:start+bs])).float().to(DEVICE)
            all_logits.append(model(xb).cpu())
    logits = torch.cat(all_logits)
    probs  = torch.softmax(logits, dim=1).numpy()
    preds  = logits.argmax(1).numpy()
    return preds, probs

def capture_rng_state():
    """Captures all Python, NumPy, and PyTorch RNG states."""
    state = {
        "py": random.getstate(),
        "np": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state

def restore_rng_state(state):
    """Restores all RNG states for exact deterministic resumption."""
    if not state:
        return
    try:
        if "py" in state: random.setstate(state["py"])
        if "np" in state: np.random.set_state(state["np"])
        if "torch" in state: torch.set_rng_state(state["torch"])
        if "cuda" in state and torch.cuda.is_available():
            torch.cuda.set_rng_state_all(state["cuda"])
    except Exception as e:
        print(f"  [WARN] Could not restore RNG state: {e}")

def train_with_resume(model, ckpt_dir, X_tr, y_tr, X_val, y_val,
                      total_epochs, seed, name, cfg):
    """Train model with epoch-level checkpointing, RNG resumption, and optimal model load."""
    ckpt_dir = Path(ckpt_dir)
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    latest = ckpt_dir / "latest.pt"
    best   = ckpt_dir / "best.pt"

    # Move model to device FIRST before creating optimizer
    model.to(DEVICE)
    optimizer   = torch.optim.Adam(model.parameters(), lr=cfg["learning_rate"])
    criterion   = nn.CrossEntropyLoss()
    history     = []
    start_epoch = 0
    best_f1     = 0.0

    if latest.exists():
        try:
            ckpt = torch.load(latest, map_location=DEVICE, weights_only=False)
            model.load_state_dict(ckpt["model_state_dict"])
            if "optimizer_state_dict" in ckpt:
                optimizer.load_state_dict(ckpt["optimizer_state_dict"])
                for state in optimizer.state.values():
                    for k, v in state.items():
                        if isinstance(v, torch.Tensor):
                            state[k] = v.to(DEVICE)
            start_epoch = ckpt["epoch"] + 1
            history     = ckpt.get("history", [])
            best_f1     = ckpt.get("best_val_f1", 0.0)
            restore_rng_state(ckpt.get("rng_state"))
            print(f"  [INFO] Resuming {name} from epoch {start_epoch}/{total_epochs}")
        except Exception as e:
            print(f"  [WARN] Could not load checkpoint ({e}), starting fresh.")
            start_epoch = 0

    if start_epoch >= total_epochs:
        print(f"  [SKIP] {name} already trained ({total_epochs} epochs done)")
        if best.exists():
            ckpt_b = torch.load(best, map_location=DEVICE, weights_only=False)
            model.load_state_dict(ckpt_b["model_state_dict"])
        model.to(DEVICE)
        return model, pd.DataFrame(history)

    model.to(DEVICE)
    current_bs = cfg["batch_size"]
    loader = make_loader_safe(X_tr, y_tr, current_bs)

    for epoch in range(start_epoch, total_epochs):
        model.train()
        ep_loss = 0.0
        
        try:
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
                optimizer.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                optimizer.step()
                ep_loss += loss.item() * len(yb)
        except (torch.cuda.OutOfMemoryError, RuntimeError) as oom_err:
            if "out of memory" in str(oom_err).lower() or "hip" in str(oom_err).lower():
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                gc.collect()
                current_bs = max(128, current_bs // 2)
                print(f"  [OOM RECOVERY] Halving batch size to {current_bs} and retrying epoch {epoch+1}")
                loader = make_loader_safe(X_tr, y_tr, current_bs)
                optimizer.zero_grad()
                ep_loss = 0.0
                continue
            else:
                raise oom_err

        model.eval()
        val_preds, val_probs = predict_model(model, X_val)
        val_f1   = float(f1_score(y_val, val_preds, average="weighted", zero_division=0))
        tr_loss  = ep_loss / len(y_tr)
        history.append({"epoch": epoch, "train_loss": tr_loss, "val_f1": val_f1})

        if (epoch + 1) % 5 == 0 or epoch == total_epochs - 1:
            print(f"  Epoch {epoch+1:3d}/{total_epochs}  loss={tr_loss:.4f}  val_F1={val_f1:.4f}")

        ckpt_dict = {"model_state_dict": model.state_dict(),
                     "optimizer_state_dict": optimizer.state_dict(),
                     "epoch": epoch, "seed": seed, "history": history,
                     "best_val_f1": best_f1, "feature_count": X_tr.shape[1],
                     "rng_state": capture_rng_state()}
        atomic_save_checkpoint(ckpt_dict, latest)
        if val_f1 > best_f1:
            best_f1 = val_f1
            ckpt_dict["best_val_f1"] = best_f1
            atomic_save_checkpoint(ckpt_dict, best)

    # Load guaranteed optimal checkpoint weights
    if best.exists():
        ckpt_b = torch.load(best, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt_b["model_state_dict"])
    model.to(DEVICE)
    return model, pd.DataFrame(history)

def pgd_attack_gpu_tensor(model, x0, yt, eps, steps, bounds, mask, seed, random_start=True):
    """Pure-GPU tensor-native PGD generation for maximum ROCm acceleration."""
    model.eval()
    lo, hi = bounds
    mt = mask
    alpha = eps * CONFIG["pgd_step_ratio"]
    
    if random_start:
        gen = torch.Generator(device=DEVICE)
        gen.manual_seed(int(seed % (2**31 - 1)))
        delta = torch.empty_like(x0).uniform_(-eps, eps, generator=gen) * mt
        x = torch.clamp(x0 + delta, lo, hi).detach()
    else:
        x = x0.clone()
        
    for _ in range(steps):
        x.requires_grad_(True)
        losses = F.cross_entropy(model(x), yt, reduction="none")
        grad = torch.autograd.grad(losses.sum(), x)[0]
        x = x.detach() + alpha * grad.sign() * mt
        x = torch.clamp(x, x0 - eps, x0 + eps)
        x = torch.clamp(x, lo, hi)
        x = (x0 + (x - x0) * mt).detach()
        
    return x.detach()

def train_online_pgd_with_resume(model, ckpt_dir, X_tr, y_tr, X_val, y_val,
                                  total_epochs, seed, name, cfg, bounds, mask):
    """Pure-GPU V9 online PGD adversarial training with epoch-level resume and OOM protection."""
    ckpt_dir = Path(ckpt_dir)
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    latest = ckpt_dir / "latest.pt"
    best   = ckpt_dir / "best.pt"

    # Move model to device FIRST before creating optimizer
    model.to(DEVICE)
    optimizer   = torch.optim.Adam(model.parameters(), lr=cfg["learning_rate"])
    criterion   = nn.CrossEntropyLoss()
    history     = []
    start_epoch = 0
    best_f1     = 0.0

    bounds_gpu = (torch.as_tensor(bounds[0], dtype=torch.float32, device=DEVICE),
                  torch.as_tensor(bounds[1], dtype=torch.float32, device=DEVICE))
    mask_gpu   = torch.as_tensor(mask, dtype=torch.float32, device=DEVICE)

    if latest.exists():
        try:
            ckpt = torch.load(latest, map_location=DEVICE, weights_only=False)
            model.load_state_dict(ckpt["model_state_dict"])
            if "optimizer_state_dict" in ckpt:
                optimizer.load_state_dict(ckpt["optimizer_state_dict"])
                for state in optimizer.state.values():
                    for k, v in state.items():
                        if isinstance(v, torch.Tensor):
                            state[k] = v.to(DEVICE)
            start_epoch = ckpt["epoch"] + 1
            history     = ckpt.get("history", [])
            best_f1     = ckpt.get("best_val_f1", 0.0)
            restore_rng_state(ckpt.get("rng_state"))
            print(f"  [INFO] Resuming V9 training ({name}) from epoch {start_epoch}/{total_epochs}")
        except Exception as e:
            print(f"  [WARN] Could not load V9 checkpoint ({e}), starting fresh.")
            start_epoch = 0

    if start_epoch >= total_epochs:
        print(f"  [SKIP] {name} already trained ({total_epochs} epochs done)")
        if best.exists():
            ckpt_b = torch.load(best, map_location=DEVICE, weights_only=False)
            model.load_state_dict(ckpt_b["model_state_dict"])
        model.to(DEVICE)
        return model, pd.DataFrame(history)

    model.to(DEVICE)
    current_bs = cfg["batch_size"]
    loader = make_loader_safe(X_tr, y_tr, current_bs)
    eps    = cfg["attack_epsilon"]
    steps  = cfg["pgd_steps_online"]

    for epoch in range(start_epoch, total_epochs):
        model.train()
        ep_loss = 0.0
        
        try:
            for b_idx, (xb_t, yb_t) in enumerate(loader):
                xb = xb_t.to(DEVICE, non_blocking=True)
                yb = yb_t.to(DEVICE, non_blocking=True)

                # Pure-GPU PGD generation on current weights
                xb_adv = pgd_attack_gpu_tensor(model, xb, yb, eps, steps, bounds_gpu, mask_gpu,
                                               seed=(seed + epoch * 1000 + b_idx) % (2**31 - 1))
                
                model.train()
                optimizer.zero_grad()
                loss_clean = criterion(model(xb), yb)
                loss_adv   = criterion(model(xb_adv), yb)
                loss = cfg["online_clean_weight"] * loss_clean + cfg["online_adv_weight"] * loss_adv
                loss.backward()
                optimizer.step()
                ep_loss += loss.item() * len(yb)
        except (torch.cuda.OutOfMemoryError, RuntimeError) as oom_err:
            if "out of memory" in str(oom_err).lower() or "hip" in str(oom_err).lower():
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                gc.collect()
                current_bs = max(128, current_bs // 2)
                print(f"  [OOM RECOVERY] Halving batch size to {current_bs} and retrying online epoch {epoch+1}")
                loader = make_loader_safe(X_tr, y_tr, current_bs)
                optimizer.zero_grad()
                ep_loss = 0.0
                continue
            else:
                raise oom_err

        model.eval()
        val_preds, _ = predict_model(model, X_val)
        val_f1  = float(f1_score(y_val, val_preds, average="weighted", zero_division=0))
        tr_loss = ep_loss / len(y_tr)
        history.append({"epoch": epoch, "train_loss": tr_loss, "val_f1": val_f1})

        if (epoch + 1) % 5 == 0 or epoch == total_epochs - 1:
            print(f"  Epoch {epoch+1:3d}/{total_epochs}  loss={tr_loss:.4f}  val_F1={val_f1:.4f}")

        ckpt_dict = {"model_state_dict": model.state_dict(),
                     "optimizer_state_dict": optimizer.state_dict(),
                     "epoch": epoch, "seed": seed, "history": history,
                     "best_val_f1": best_f1, "feature_count": X_tr.shape[1],
                     "rng_state": capture_rng_state()}
        atomic_save_checkpoint(ckpt_dict, latest)
        if val_f1 > best_f1:
            best_f1 = val_f1
            ckpt_dict["best_val_f1"] = best_f1
            atomic_save_checkpoint(ckpt_dict, best)

    # Load guaranteed optimal checkpoint weights
    if best.exists():
        ckpt_b = torch.load(best, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt_b["model_state_dict"])
    model.to(DEVICE)
    return model, pd.DataFrame(history)

print("[OK] Pure-GPU model architectures, training engine and dynamic OOM handlers ready.")

In [ ]:
# Cell 15 — Fast Vectorized Attacks & Strictly-Within-Epsilon Boundary Search
def _to_gpu_float(x):
    return torch.as_tensor(np.ascontiguousarray(x), dtype=torch.float32, device=DEVICE)

def fgsm_attack(model, X, y, eps, bounds, mask):
    """Single-step FGSM attack. Returns float32 numpy array."""
    model.eval()
    x0 = _to_gpu_float(X)
    yt = torch.as_tensor(y, dtype=torch.long, device=DEVICE)
    mt = _to_gpu_float(mask)
    lo = _to_gpu_float(bounds[0])
    hi = _to_gpu_float(bounds[1])

    x = x0.clone().requires_grad_(True)
    loss = F.cross_entropy(model(x), yt)
    grad = torch.autograd.grad(loss, x)[0]
    adv = x0 + eps * grad.sign() * mt
    adv = torch.clamp(adv, lo, hi)
    return adv.detach().cpu().numpy().astype(np.float32)

def pgd_attack(model, X, y, eps, steps, restarts, bounds, mask, seed, random_start=True):
    """PGD with L-inf constraint and feature bounds."""
    model.eval()
    x0  = _to_gpu_float(X)
    yt  = torch.as_tensor(y, dtype=torch.long, device=DEVICE)
    mt  = _to_gpu_float(mask)
    lo  = _to_gpu_float(bounds[0])
    hi  = _to_gpu_float(bounds[1])
    alpha = eps * CONFIG["pgd_step_ratio"]
    best      = x0.clone()
    best_loss = torch.full((len(X),), -float("inf"), device=DEVICE)

    for r in range(restarts):
        gen = torch.Generator(device=DEVICE)
        gen.manual_seed(int((seed + r) % (2**31 - 1)))
        delta = (torch.empty_like(x0).uniform_(-eps, eps, generator=gen) * mt
                 if random_start else torch.zeros_like(x0))
        x = torch.clamp(x0 + delta, lo, hi).detach()

        for _ in range(steps):
            x.requires_grad_(True)
            losses = F.cross_entropy(model(x), yt, reduction="none")
            grad   = torch.autograd.grad(losses.sum(), x)[0]
            x = x.detach() + alpha * grad.sign() * mt
            x = torch.clamp(x, x0 - eps, x0 + eps)  # L-inf projection
            x = torch.clamp(x, lo, hi)               # feature bounds
            x = (x0 + (x - x0) * mt).detach()        # mask immutable features

        with torch.no_grad():
            losses = F.cross_entropy(model(x), yt, reduction="none")
        better = losses > best_loss
        best[better] = x[better]
        best_loss[better] = losses[better]

    result = best.cpu().numpy().astype(np.float32)
    max_pert = np.abs(result - X.astype(np.float32)).max()
    assert max_pert <= eps + 1e-5, f"PGD L-inf constraint violated: {max_pert:.6f} > {eps}"
    return result

def deepfool_via_art(model, X, n_classes=2):
    """DeepFool unconstrained minimum-perturbation attack via ART."""
    from art.estimators.classification import PyTorchClassifier
    model.eval()
    art_clf = PyTorchClassifier(
        model=model,
        loss=nn.CrossEntropyLoss(),
        optimizer=torch.optim.Adam(model.parameters()),
        input_shape=(X.shape[1],),
        nb_classes=n_classes,
        device_type="gpu" if DEVICE.type == "cuda" else "cpu",
        clip_values=(float(X.min()), float(X.max())),
    )
    attack = DeepFool(classifier=art_clf, max_iter=CONFIG["deepfool_steps"], verbose=False)
    X_adv = attack.generate(x=np.ascontiguousarray(X, dtype=np.float32))
    return X_adv.astype(np.float32)

def square_style_attack(model, X, eps, bounds=None, n_iters=None, seed=42):
    """
    Fast Vectorized Square-style Attack:
    Evaluates candidate noise blocks across all samples simultaneously in GPU batches.
    """
    rng = np.random.RandomState(seed)
    n_iters = n_iters or CONFIG["square_iterations"]
    n, d = X.shape
    init_w = max(1, int(d * CONFIG["square_init_frac"]))
    
    X_curr = torch.as_tensor(np.ascontiguousarray(X), dtype=torch.float32, device=DEVICE)
    X_orig = X_curr.clone()
    
    lo = torch.as_tensor(bounds[0], dtype=torch.float32, device=DEVICE) if bounds is not None else None
    hi = torch.as_tensor(bounds[1], dtype=torch.float32, device=DEVICE) if bounds is not None else None
    
    model.eval()
    with torch.no_grad():
        orig_logits = model(X_curr)
        orig_cls = orig_logits.argmax(dim=1)
        orig_probs = F.softmax(orig_logits, dim=1)
        best_target_conf = orig_probs[torch.arange(n, device=DEVICE), orig_cls]
        
    for it in range(n_iters):
        w = max(1, int(init_w * (1 - it / n_iters)))
        start_indices = rng.randint(0, max(1, d - w + 1), size=n)
        noise = rng.uniform(-eps, eps, size=(n, w)).astype(np.float32)
        
        X_cand = X_curr.clone()
        for i in range(n):
            s = start_indices[i]
            X_cand[i, s:s+w] += torch.as_tensor(noise[i], device=DEVICE)
            
        X_cand = torch.max(torch.min(X_cand, X_orig + eps), X_orig - eps)
        if lo is not None and hi is not None:
            X_cand = torch.clamp(X_cand, lo, hi)
            
        with torch.no_grad():
            cand_logits = model(X_cand)
            cand_probs = F.softmax(cand_logits, dim=1)
            cand_target_conf = cand_probs[torch.arange(n, device=DEVICE), orig_cls]
            
        improved = cand_target_conf < best_target_conf
        X_curr[improved] = X_cand[improved]
        best_target_conf[improved] = cand_target_conf[improved]
        
    return X_curr.detach().cpu().numpy().astype(np.float32)

def label_only_boundary_search_fixed(model, X_clean, y_clean, eps, bounds, mask, seed,
                                     init_trials=None, refine_steps=None, X_ref_pool=None, y_ref_pool=None):
    """
    Corrected Strictly-Within-Epsilon Label-Only Boundary Search Diagnostic.
    - Operates on clean-correct samples (y_pred_clean == y_true).
    - Guarantees ALL candidate initializations and binary search steps are within [x0 - eps, x0 + eps].
    - Evaluates whether a valid label-transition point exists INSIDE the epsilon ball.
    """
    rng = np.random.RandomState(seed)
    n_total = len(X_clean)
    X_clean = np.asarray(X_clean, dtype=np.float32)
    y_clean = np.asarray(y_clean, dtype=np.int64)
    d = X_clean.shape[1]
    
    lo = bounds[0].astype(np.float32)
    hi = bounds[1].astype(np.float32)
    mask = mask.astype(np.float32)
    init_trials = init_trials or CONFIG.get("boundary_init_trials", 30)
    refine_steps = refine_steps or CONFIG.get("boundary_refine_steps", 30)
    
    # Step 1: Clean start on correctly classified test samples
    model.eval()
    with torch.no_grad():
        x_t = torch.from_numpy(np.ascontiguousarray(X_clean)).float().to(DEVICE)
        clean_preds = model(x_t).argmax(dim=1).cpu().numpy()
        
    clean_correct_mask = (clean_preds == y_clean)
    n_clean_correct = int(clean_correct_mask.sum())
    
    ref_by_class = {}
    if X_ref_pool is not None and y_ref_pool is not None:
        X_ref_np = np.asarray(X_ref_pool, dtype=np.float32)
        y_ref_np = np.asarray(y_ref_pool, dtype=np.int64)
        for c in np.unique(y_ref_np):
            ref_by_class[int(c)] = X_ref_np[y_ref_np == c]
            
    records = []
    
    for i in range(n_total):
        if not clean_correct_mask[i]:
            records.append({
                "idx": i, "clean_correct": False,
                "boundary_found": False, "boundary_within_epsilon": False,
                "linf_distance": None, "l2_distance": None,
                "query_count": 1
            })
            continue
            
        x0 = X_clean[i]
        y0 = int(y_clean[i])
        queries = 1
        
        # Step 2: Bounded Random Initialization inside [x0 - eps, x0 + eps]
        x_adv = None
        init_success = False
        
        for _ in range(init_trials):
            delta = rng.uniform(-eps, eps, size=d).astype(np.float32) * mask
            x_cand = np.clip(x0 + delta, lo, hi)
            assert np.max(np.abs(x_cand - x0)) <= eps + 1e-5
            
            with torch.no_grad():
                pred = int(model(torch.from_numpy(x_cand[None]).float().to(DEVICE)).argmax(1).cpu().numpy()[0])
            queries += 1
            
            if pred != y0:
                x_adv = x_cand
                init_success = True
                break
                
        # Step 3: Bounded Reference Interpolation (smart bounded initialization)
        if not init_success and (1 - y0) in ref_by_class and len(ref_by_class[1 - y0]) > 0:
            opp_pool = ref_by_class[1 - y0]
            n_refs = min(5, len(opp_pool))
            ref_picks = rng.choice(len(opp_pool), n_refs, replace=False)
            for r_idx in ref_picks:
                x_ref = opp_pool[r_idx]
                diff = (x_ref - x0) * mask
                max_diff = np.max(np.abs(diff))
                if max_diff > 1e-7:
                    max_lambda = min(1.0, float(eps / max_diff))
                    for lam in np.linspace(max_lambda, 0.1 * max_lambda, 5):
                        x_cand = np.clip(x0 + lam * diff, lo, hi)
                        assert np.max(np.abs(x_cand - x0)) <= eps + 1e-5
                        with torch.no_grad():
                            pred = int(model(torch.from_numpy(x_cand[None]).float().to(DEVICE)).argmax(1).cpu().numpy()[0])
                        queries += 1
                        if pred != y0:
                            x_adv = x_cand
                            init_success = True
                            break
                if init_success:
                    break
                    
        # Step 4: Binary Search ONLY inside epsilon
        if init_success and x_adv is not None:
            low = 0.0
            high = 1.0
            for _ in range(refine_steps):
                mid = (low + high) / 2.0
                x_mid = np.clip(x0 + mid * (x_adv - x0), lo, hi)
                assert np.max(np.abs(x_mid - x0)) <= eps + 1e-5
                
                with torch.no_grad():
                    pred = int(model(torch.from_numpy(x_mid[None]).float().to(DEVICE)).argmax(1).cpu().numpy()[0])
                queries += 1
                
                if pred != y0:
                    high = mid
                else:
                    low = mid
                    
            x_boundary = np.clip(x0 + high * (x_adv - x0), lo, hi)
            delta_final = x_boundary - x0
            linf_dist = float(np.max(np.abs(delta_final)))
            l2_dist = float(np.linalg.norm(delta_final))
            
            # Step 5: Final validation
            is_valid_eps = (linf_dist <= eps + 1e-5)
            
            records.append({
                "idx": i, "clean_correct": True,
                "boundary_found": True,
                "boundary_within_epsilon": is_valid_eps,
                "linf_distance": linf_dist if is_valid_eps else None,
                "l2_distance": l2_dist if is_valid_eps else None,
                "query_count": queries
            })
        else:
            records.append({
                "idx": i, "clean_correct": True,
                "boundary_found": False,
                "boundary_within_epsilon": False,
                "linf_distance": None,
                "l2_distance": None,
                "query_count": queries
            })
            
    rec_df = pd.DataFrame(records)
    
    # Step 7: Required Final Statistics
    n_init_success = int(rec_df["boundary_found"].sum())
    n_within_eps = int(rec_df["boundary_within_epsilon"].sum())
    
    l2_vals = rec_df[rec_df["boundary_within_epsilon"]]["l2_distance"].dropna().values
    linf_vals = rec_df[rec_df["boundary_within_epsilon"]]["linf_distance"].dropna().values
    queries_vals = rec_df["query_count"].values
    
    stats = {
        "n_total":                          n_total,
        "n_clean_correct":                  n_clean_correct,
        "n_initialization_success":         n_init_success,
        "initialization_success_rate":       float(n_init_success / max(1, n_clean_correct)),
        "n_boundary_refined":               n_within_eps,
        "boundary_refinement_success_rate":  float(n_within_eps / max(1, n_init_success)) if n_init_success > 0 else 0.0,
        "n_within_epsilon":                 n_within_eps,
        "within_epsilon_success_rate":       float(n_within_eps / max(1, n_clean_correct)),
        "mean_l2_within_epsilon":           float(np.mean(l2_vals)) if len(l2_vals) > 0 else None,
        "median_l2_within_epsilon":         float(np.median(l2_vals)) if len(l2_vals) > 0 else None,
        "p10_l2":                           float(np.percentile(l2_vals, 10)) if len(l2_vals) > 0 else None,
        "p90_l2":                           float(np.percentile(l2_vals, 90)) if len(l2_vals) > 0 else None,
        "mean_linf":                        float(np.mean(linf_vals)) if len(linf_vals) > 0 else None,
        "median_linf":                      float(np.median(linf_vals)) if len(linf_vals) > 0 else None,
        "mean_queries":                     float(np.mean(queries_vals)),
        "median_queries":                   float(np.median(queries_vals)),
    }
    
    X_adv_out = X_clean.copy()
    return X_adv_out, rec_df, stats

def attack_perturbation_stats(X_clean, X_adv, y_true, y_pred_clean, y_pred_adv):
    """
    Compute attack sanity and perturbation magnitude statistics.
    Standard ASR Definition: (clean_correct -> adv_wrong) / clean_correct.
    """
    X_clean = np.asarray(X_clean, dtype=np.float32)
    X_adv   = np.asarray(X_adv, dtype=np.float32)
    y_true  = np.asarray(y_true)
    y_pred_clean = np.asarray(y_pred_clean)
    y_pred_adv   = np.asarray(y_pred_adv)
    
    delta   = np.abs(X_adv - X_clean)
    linf    = delta.max(axis=1)
    l2      = np.linalg.norm(X_adv - X_clean, axis=1)
    n       = len(X_clean)
    
    # Rigorous scientific ASR definition
    clean_correct = (y_pred_clean == y_true)
    if clean_correct.sum() > 0:
        asr = float((y_pred_adv[clean_correct] != y_true[clean_correct]).mean())
    else:
        asr = 0.0
        
    zero_pct = float((linf < 1e-8).mean())
    return {
        "n_attempted":         n,
        "attack_success_rate": asr,
        "zero_perturbation_pct": zero_pct,
        "mean_Linf":           float(linf.mean()),
        "max_Linf":            float(linf.max()),
        "mean_L2":             float(l2.mean()),
    }

print("[OK] Vectorized attack implementations, strict L-inf Boundary Search and standard ASR defined.")

In [ ]:
# Cell 16 — SHAP Feature Selection (Train Calibration Pool, Zero Dev Leakage)
shap_path     = ARTIFACT_DIR / "shap_features.json"
shap_imp_path = ARTIFACT_DIR / "shap_importance.csv"

check_resources("PRE-SHAP")

if SM.report("shap_selection") and shap_path.exists() and shap_imp_path.exists():
    with open(shap_path) as f:
        shap_info = json.load(f)
    paper_shap_features     = shap_info["paper_shap_features"]
    extension_shap_features = shap_info["extension_shap_features"]
    paper_idx     = [int(i) for i in shap_info["paper_idx"]]
    extension_idx = [int(i) for i in shap_info["extension_idx"]]
    shap_imp_df = pd.read_csv(shap_imp_path)
    print(f"[LOAD] Paper SHAP features: {len(paper_shap_features)}")
    print(f"[LOAD] Extension SHAP features: {len(extension_shap_features)}")
else:
    t0   = time.perf_counter()
    n_mi = len(mi_idx)

    # Train calibration model on training pool
    set_seed(CONFIG["data_seed"])
    temp_model = IDSNet(n_mi).to(DEVICE)
    temp_opt   = torch.optim.Adam(temp_model.parameters(), lr=CONFIG["learning_rate"])
    temp_crit  = nn.CrossEntropyLoss()
    temp_loader = make_loader_safe(X_train_bal, y_train_bal, min(CONFIG["batch_size"], 2048))

    temp_model.train()
    for ep in range(20):
        for xb, yb in temp_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            temp_opt.zero_grad()
            temp_crit(temp_model(xb), yb).backward()
            temp_opt.step()
    temp_model.eval()

    # METHODOLOGY IMPROVEMENT: Carve SHAP evaluation samples strictly from the TRAIN pool
    # The Dev set remains 100% untouched for unbiased validation tracking during defense training.
    n_bg   = min(CONFIG["shap_background_samples"], len(X_train_mi))
    n_eval = min(CONFIG["shap_eval_samples"], len(X_train_mi))
    rng    = np.random.RandomState(CONFIG["data_seed"])
    bg_idx = rng.choice(len(X_train_mi), n_bg, replace=False)
    ev_idx = rng.choice(len(X_train_mi), n_eval, replace=False)

    X_bg   = torch.from_numpy(np.ascontiguousarray(X_train_mi[bg_idx])).float().to(DEVICE)
    X_eval = torch.from_numpy(np.ascontiguousarray(X_train_mi[ev_idx])).float().to(DEVICE)
    y_eval_calib = y_train[ev_idx]

    explainer = shap.GradientExplainer(temp_model, X_bg)

    def _extract_shap_importance(raw_shap):
        """Extracts 1D feature importance array robust to SHAP output dimensionality."""
        if isinstance(raw_shap, list):
            arr = np.array(raw_shap)
            return np.abs(arr).mean(axis=(0, 1))
        arr = np.asarray(raw_shap)
        if arr.ndim == 3:
            return np.abs(arr).mean(axis=(0, 2)) if arr.shape[2] == 2 else np.abs(arr).mean(axis=(0, 1))
        elif arr.ndim == 2:
            return np.abs(arr).mean(axis=0)
        else:
            return np.abs(arr).flatten()

    # Clean SHAP importance
    shap_clean = explainer.shap_values(X_eval)
    clean_imp = _extract_shap_importance(shap_clean)

    # Adversarial SHAP importance (PGD perturbations generated on calibration subset)
    X_eval_np  = X_train_mi[ev_idx]
    X_adv_eval = pgd_attack(temp_model, X_eval_np, y_eval_calib,
                             CONFIG["attack_epsilon"], 20, 1, bounds, structural_mask,
                             seed=CONFIG["data_seed"])
    X_adv_t  = torch.from_numpy(np.ascontiguousarray(X_adv_eval)).float().to(DEVICE)
    shap_adv = explainer.shap_values(X_adv_t)
    adv_imp = _extract_shap_importance(shap_adv)

    # V9 Stability formula: 1.0 - |clean - adv| / denom
    denom = np.maximum(clean_imp + adv_imp, 1e-10)
    robust_score = 1.0 - np.abs(clean_imp - adv_imp) / denom

    mi_feature_names = [ALL_FEATURE_NAMES[i] for i in mi_idx]
    order_clean = np.argsort(clean_imp)[::-1]
    paper_shap_idx_local = order_clean[:CONFIG["paper_shap_top_k"]].tolist()
    paper_shap_features  = [mi_feature_names[i] for i in paper_shap_idx_local]
    paper_idx            = [mi_idx[i] for i in paper_shap_idx_local]

    order_robust = np.argsort(robust_score)[::-1]
    rs_sorted    = robust_score[order_robust]
    rs_total     = rs_sorted.sum()
    cumsum       = np.cumsum(rs_sorted)
    cutoff       = np.searchsorted(cumsum, CONFIG["extension_shap_cumulative"] * rs_total)
    ext_local    = order_robust[:cutoff+1].tolist()
    extension_shap_features = [mi_feature_names[i] for i in ext_local]
    extension_idx           = [mi_idx[i] for i in ext_local]

    shap_info = {"paper_shap_features": paper_shap_features,
                 "extension_shap_features": extension_shap_features,
                 "paper_idx": paper_idx, "extension_idx": extension_idx}
    atomic_save_json(shap_info, shap_path)

    shap_imp_df = pd.DataFrame({
        "feature":     mi_feature_names,
        "clean_imp":   clean_imp,
        "adv_imp":     adv_imp,
        "robust_score": robust_score,
    }).sort_values("robust_score", ascending=False)
    atomic_save_df(shap_imp_df, shap_imp_path)

    del temp_model, explainer, X_bg, X_eval, X_adv_t
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    SM.mark_done("shap_selection", {"paper_k": len(paper_shap_features),
                                     "ext_k":   len(extension_shap_features)})
    print(f"[OK] SHAP selection completed in {time.perf_counter()-t0:.1f}s")

paper_idx_local     = [mi_idx.index(i) for i in paper_idx if i in mi_idx]
extension_idx_local = [mi_idx.index(i) for i in extension_idx if i in mi_idx]

# Feature subsets for Baseline, Paper Defense, and V9 Defense
X_train_paper = X_train_bal[:, paper_idx_local]
X_dev_paper   = X_dev_mi[:,   paper_idx_local]
X_test_paper  = X_test_mi[:,  paper_idx_local]
X_train_ext   = X_train_bal[:, extension_idx_local]
X_dev_ext     = X_dev_mi[:,    extension_idx_local]
X_test_ext    = X_test_mi[:,   extension_idx_local]

paper_bounds = (bounds[0][paper_idx_local], bounds[1][paper_idx_local])
ext_bounds   = (bounds[0][extension_idx_local], bounds[1][extension_idx_local])
paper_mask   = structural_mask[paper_idx_local]
ext_mask     = structural_mask[extension_idx_local]

# Feature dimension consistency assertions
assert X_train_paper.shape[1] == len(paper_idx_local), "Paper feature dimension mismatch"
assert X_train_ext.shape[1] == len(extension_idx_local), "Extension feature dimension mismatch"

print(f"[INFO] Paper feature space: {len(paper_idx_local)} features")
print(f"[INFO] Extension feature space: {len(extension_idx_local)} features")
check_resources("POST-SHAP")

In [ ]:
# ======================================================================
# SEED 42 — FULLY SELF-CONTAINED INDEPENDENT EXECUTION
# ======================================================================
RUN_SEED = 42
set_seed(RUN_SEED)
t_seed_start = time.perf_counter()
print(f"\n======================================================================")
print(f"  RUNNING SEED 42")
print(f"======================================================================\n")

SEED_DIR   = OUTPUT_DIR / "seeds" / str(RUN_SEED)
ATTACK_DIR = SEED_DIR / "attacks"
MODEL_DIR  = SEED_DIR / "models"
LOG_DIR    = SEED_DIR / "logs"
for _d in [SEED_DIR, ATTACK_DIR, MODEL_DIR, LOG_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

atomic_save_json({"seed": RUN_SEED, "status": "started", "start_time": time.time()}, SEED_DIR / "seed_start.json")
atomic_save_json(CONFIG, SEED_DIR / "config.json")
_env_seed = {
    "python": sys.version, "torch": torch.__version__,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type=="cuda" else "cpu",
    "os": platform.platform(),
}
atomic_save_json(_env_seed, SEED_DIR / "environment.json")
seed_SM = StageManager(SEED_DIR)

# ── 1. Baseline Training ──────────────────────────────────────────────────────
set_seed(RUN_SEED)
baseline_dir = MODEL_DIR / "baseline"
if seed_SM.report("baseline_train") and (baseline_dir / "best.pt").exists():
    baseline = IDSNet(len(paper_idx_local)).to(DEVICE)
    baseline.load_state_dict(
        torch.load(baseline_dir / "best.pt", map_location=DEVICE, weights_only=False)["model_state_dict"]
    )
else:
    t_b = time.perf_counter()
    baseline = IDSNet(len(paper_idx_local))
    baseline, hist_baseline = train_with_resume(
        baseline, baseline_dir,
        X_train_paper, y_train_bal,
        X_dev_paper,   y_dev,
        CONFIG["baseline_epochs"], RUN_SEED, "Baseline", CONFIG
    )
    hist_baseline.to_csv(baseline_dir / "history.csv", index=False)
    seed_SM.mark_done("baseline_train", {"seconds": time.perf_counter()-t_b})
    print(f"  [OK] Baseline trained in {time.perf_counter()-t_b:.1f}s")

baseline.eval()
bl_preds, bl_probs = predict_model(baseline, X_test_paper)
bl_metrics = compute_metrics(y_test, bl_preds, bl_probs)
print(f"  [RESULT] Baseline clean acc={bl_metrics['accuracy']:.4f}  F1={bl_metrics['f1_weighted']:.4f}")

# ── 2. Paper Defense Training (Static FGSM + PGD + DeepFool Augmentation) ────
set_seed(RUN_SEED + 10_000)
paper_dir = MODEL_DIR / "paper_defense"
if seed_SM.report("paper_defense_train") and (paper_dir / "best.pt").exists():
    paper_model = IDSNet(len(paper_idx_local)).to(DEVICE)
    paper_model.load_state_dict(
        torch.load(paper_dir / "best.pt", map_location=DEVICE, weights_only=False)["model_state_dict"]
    )
else:
    t_p = time.perf_counter()
    aug_fgsm_n = min(CONFIG["paper_fgsm_n"], len(X_train_paper))
    aug_pgd_n  = min(CONFIG["paper_pgd_n"], len(X_train_paper))
    aug_df_n   = min(CONFIG["paper_deepfool_n"], len(X_train_paper))
    
    aug_model = IDSNet(len(paper_idx_local)).to(DEVICE)
    _warm_loader = make_loader_safe(X_train_paper[:aug_fgsm_n], y_train_bal[:aug_fgsm_n], CONFIG["batch_size"])
    _warm_opt    = torch.optim.Adam(aug_model.parameters(), lr=CONFIG["learning_rate"])
    aug_model.train()
    for _ in range(5):
        for _xb, _yb in _warm_loader:
            _xb, _yb = _xb.to(DEVICE, non_blocking=True), _yb.to(DEVICE, non_blocking=True)
            _warm_opt.zero_grad()
            nn.CrossEntropyLoss()(aug_model(_xb), _yb).backward()
            _warm_opt.step()
    aug_model.eval()
    
    # Generate static triad: FGSM + PGD + DeepFool
    X_aug_fgsm = fgsm_attack(aug_model, X_train_paper[:aug_fgsm_n], y_train_bal[:aug_fgsm_n],
                             CONFIG["attack_epsilon"], paper_bounds, paper_mask)
    X_aug_pgd  = pgd_attack(aug_model, X_train_paper[:aug_pgd_n], y_train_bal[:aug_pgd_n],
                            CONFIG["attack_epsilon"], 10, 1, paper_bounds, paper_mask, seed=RUN_SEED+10_000)
    try:
        X_aug_df = deepfool_via_art(aug_model, X_train_paper[:aug_df_n])
        X_aug_df = np.clip(X_aug_df, paper_bounds[0], paper_bounds[1])
    except Exception as e:
        print(f"  [WARN] DeepFool static aug skipped: {e}")
        X_aug_df = X_train_paper[:0]
        aug_df_n = 0
    
    X_tr_aug = np.vstack([X_train_paper, X_aug_fgsm, X_aug_pgd, X_aug_df])
    y_tr_aug = np.concatenate([y_train_bal, y_train_bal[:aug_fgsm_n], y_train_bal[:aug_pgd_n], y_train_bal[:aug_df_n]])
    
    del aug_model, X_aug_fgsm, X_aug_pgd, X_aug_df
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    paper_model = IDSNet(len(paper_idx_local))
    paper_model, hist_paper = train_with_resume(
        paper_model, paper_dir,
        X_tr_aug, y_tr_aug,
        X_dev_paper, y_dev,
        CONFIG["paper_epochs"], RUN_SEED+10_000, "Paper Defense (FGSM+PGD+DF Aug)", CONFIG
    )
    hist_paper.to_csv(paper_dir / "history.csv", index=False)
    seed_SM.mark_done("paper_defense_train", {"seconds": time.perf_counter()-t_p})
    print(f"  [OK] Paper Defense trained in {time.perf_counter()-t_p:.1f}s")

paper_model.eval()
pp_preds, pp_probs = predict_model(paper_model, X_test_paper)
pp_metrics = compute_metrics(y_test, pp_preds, pp_probs)
print(f"  [RESULT] Paper Defense clean acc={pp_metrics['accuracy']:.4f}  F1={pp_metrics['f1_weighted']:.4f}")

# ── 3. V9 Online Adversarial Training (Dynamic Per-Batch PGD on GPU) ──────────
set_seed(RUN_SEED + 20_000)
v9_dir = MODEL_DIR / "v9_defense"
if seed_SM.report("v9_train") and (v9_dir / "best.pt").exists():
    v9_model = IDSNet(len(extension_idx_local)).to(DEVICE)
    v9_model.load_state_dict(
        torch.load(v9_dir / "best.pt", map_location=DEVICE, weights_only=False)["model_state_dict"]
    )
else:
    t_v = time.perf_counter()
    v9_model = IDSNet(len(extension_idx_local))
    # Phase 1: clean pretraining
    v9_model, hist_v9_clean = train_with_resume(
        v9_model, v9_dir / "phase1_clean",
        X_train_ext, y_train_bal,
        X_dev_ext, y_dev,
        CONFIG["v9_clean_epochs"], RUN_SEED+20_000, "V9 Phase1 Clean", CONFIG
    )
    # Phase 2: online PGD training (pure GPU acceleration)
    v9_model, hist_v9_online = train_online_pgd_with_resume(
        v9_model, v9_dir,
        X_train_ext, y_train_bal,
        X_dev_ext, y_dev,
        CONFIG["v9_online_epochs"], RUN_SEED+20_000, "V9 Online PGD", CONFIG,
        ext_bounds, ext_mask
    )
    hist_v9_online.to_csv(v9_dir / "history_online.csv", index=False)
    seed_SM.mark_done("v9_train", {"seconds": time.perf_counter()-t_v})
    print(f"  [OK] V9 Defense trained in {time.perf_counter()-t_v:.1f}s")

v9_model.eval()
v9_preds, v9_probs = predict_model(v9_model, X_test_ext)
v9_metrics = compute_metrics(y_test, v9_preds, v9_probs)
print(f"  [RESULT] V9 Defense clean acc={v9_metrics['accuracy']:.4f}  F1={v9_metrics['f1_weighted']:.4f}")

# ── 4. Fixed Evaluation Sample Selection (shared across all models) ───────────
eval_ids_path = SEED_DIR / "eval_ids.npz"
if eval_ids_path.exists():
    eval_ids = np.load(eval_ids_path)["eval_ids"]
else:
    rng = np.random.RandomState(RUN_SEED)
    eval_ids = rng.choice(len(X_test_mi), min(CONFIG["eval_n"], len(X_test_mi)), replace=False)
    atomic_save_numpy(eval_ids_path, eval_ids=eval_ids)

X_eval_paper = X_test_paper[eval_ids]
X_eval_ext   = X_test_ext[eval_ids]
y_eval       = y_test[eval_ids]

bl_clean_p, _ = predict_model(baseline, X_eval_paper)
pp_clean_p, _ = predict_model(paper_model, X_eval_paper)
v9_clean_p, _ = predict_model(v9_model, X_eval_ext)

# ── 5. FGSM Evaluation ───────────────────────────────────────────────────────
fgsm_csv = ATTACK_DIR / "fgsm.csv"
if seed_SM.report("fgsm_eval") and fgsm_csv.exists():
    fgsm_df = pd.read_csv(fgsm_csv)
else:
    rows = []
    for model_obj, name, X_ev, bnds, msk, clean_p in [
        (baseline,    "Baseline",     X_eval_paper, paper_bounds, paper_mask, bl_clean_p),
        (paper_model, "Paper Defense",X_eval_paper, paper_bounds, paper_mask, pp_clean_p),
        (v9_model,    "V9 Defense",   X_eval_ext,   ext_bounds,   ext_mask,   v9_clean_p),
    ]:
        model_obj.eval()
        X_adv = fgsm_attack(model_obj, X_ev, y_eval, CONFIG["attack_epsilon"], bnds, msk)
        adv_p, adv_prob = predict_model(model_obj, X_adv)
        metrics = compute_metrics(y_eval, adv_p, adv_prob)
        pert    = attack_perturbation_stats(X_ev, X_adv, y_eval, clean_p, adv_p)
        rows.append({"method": name, "attack": "FGSM", "seed": RUN_SEED, **metrics, **pert})
        print(f"  FGSM {name}: acc={metrics['accuracy']:.4f}  ASR={pert['attack_success_rate']:.4f}")
    fgsm_df = pd.DataFrame(rows)
    atomic_save_df(fgsm_df, fgsm_csv)
    seed_SM.mark_done("fgsm_eval")

# ── 6. PGD Evaluation (100 steps) ─────────────────────────────────────────────
pgd_csv = ATTACK_DIR / "pgd.csv"
if seed_SM.report("pgd_eval") and pgd_csv.exists():
    pgd_df = pd.read_csv(pgd_csv)
else:
    rows = []
    for model_obj, name, X_ev, bnds, msk, clean_p in [
        (baseline,    "Baseline",     X_eval_paper, paper_bounds, paper_mask, bl_clean_p),
        (paper_model, "Paper Defense",X_eval_paper, paper_bounds, paper_mask, pp_clean_p),
        (v9_model,    "V9 Defense",   X_eval_ext,   ext_bounds,   ext_mask,   v9_clean_p),
    ]:
        model_obj.eval()
        X_adv = pgd_attack(model_obj, X_ev, y_eval,
                           CONFIG["attack_epsilon"], CONFIG["pgd_steps_eval"],
                           CONFIG["pgd_restarts"], bnds, msk, RUN_SEED)
        adv_p, adv_prob = predict_model(model_obj, X_adv)
        metrics = compute_metrics(y_eval, adv_p, adv_prob)
        pert    = attack_perturbation_stats(X_ev, X_adv, y_eval, clean_p, adv_p)
        rows.append({"method": name, "attack": "PGD", "seed": RUN_SEED, **metrics, **pert})
        print(f"  PGD {name}: acc={metrics['accuracy']:.4f}  ASR={pert['attack_success_rate']:.4f}")
    pgd_df = pd.DataFrame(rows)
    atomic_save_df(pgd_df, pgd_csv)
    seed_SM.mark_done("pgd_eval")

# ── 7. DeepFool Evaluation (unconstrained) ───────────────────────────────────
df_ids   = eval_ids[:CONFIG["deepfool_eval_n"]]
X_df_p   = X_test_paper[df_ids]
X_df_e   = X_test_ext[df_ids]
y_df     = y_test[df_ids]

deepfool_csv = ATTACK_DIR / "deepfool.csv"
if seed_SM.report("deepfool_eval") and deepfool_csv.exists():
    deepfool_df = pd.read_csv(deepfool_csv)
else:
    rows = []
    for model_obj, name, X_ev in [
        (baseline,    "Baseline",     X_df_p),
        (paper_model, "Paper Defense",X_df_p),
        (v9_model,    "V9 Defense",   X_df_e),
    ]:
        try:
            model_obj.eval()
            X_adv     = deepfool_via_art(model_obj, X_ev)
            adv_p, adv_prob = predict_model(model_obj, X_adv)
            clean_p   = predict_model(model_obj, X_ev)[0]
            metrics   = compute_metrics(y_df, adv_p, adv_prob)
            pert      = attack_perturbation_stats(X_ev, X_adv, y_df, clean_p, adv_p)
            rows.append({"method": name, "attack": "DeepFool (unconstrained)",
                          "seed": RUN_SEED, **metrics, **pert,
                          "note": "unconstrained minimum-perturbation — do not compare directly with bounded attacks"})
            print(f"  DeepFool {name}: acc={metrics['accuracy']:.4f}")
        except Exception as e:
            print(f"  [WARN] DeepFool {name} skipped: {e}")
    deepfool_df = pd.DataFrame(rows)
    atomic_save_df(deepfool_df, deepfool_csv)
    seed_SM.mark_done("deepfool_eval")

# ── 8. Fast Vectorized Square-style Attack Evaluation ─────────────────────────
sq_ids = eval_ids[:CONFIG["square_eval_n"]]
X_sq_p = X_test_paper[sq_ids]
X_sq_e = X_test_ext[sq_ids]
y_sq   = y_test[sq_ids]

sq_csv = ATTACK_DIR / "square_style.csv"
if seed_SM.report("square_eval") and sq_csv.exists():
    sq_df = pd.read_csv(sq_csv)
else:
    rows = []
    for model_obj, name, X_ev, bnds in [
        (baseline,    "Baseline",     X_sq_p, paper_bounds),
        (paper_model, "Paper Defense",X_sq_p, paper_bounds),
        (v9_model,    "V9 Defense",   X_sq_e, ext_bounds),
    ]:
        model_obj.eval()
        t_sq = time.perf_counter()
        X_adv     = square_style_attack(model_obj, X_ev, CONFIG["attack_epsilon"],
                                        bounds=bnds, n_iters=CONFIG["square_iterations"], seed=RUN_SEED)
        clean_p   = predict_model(model_obj, X_ev)[0]
        adv_p, adv_prob = predict_model(model_obj, X_adv)
        metrics   = compute_metrics(y_sq, adv_p, adv_prob)
        pert      = attack_perturbation_stats(X_ev, X_adv, y_sq, clean_p, adv_p)
        rows.append({"method": name, "attack": "Square-style Attack",
                      "seed": RUN_SEED, **metrics, **pert,
                      "note": "custom score-based gradient-free heuristic (NOT canonical Square Attack)"})
        print(f"  Square-style {name} ({time.perf_counter()-t_sq:.1f}s): acc={metrics['accuracy']:.4f}  ASR={pert['attack_success_rate']:.4f}")
    sq_df = pd.DataFrame(rows)
    atomic_save_df(sq_df, sq_csv)
    seed_SM.mark_done("square_eval")

# ── 9. Strictly-Within-Epsilon Label-Only Boundary Search Diagnostic ──────────
bnd_ids = eval_ids[:CONFIG["boundary_n"]]
X_bnd_p = X_test_paper[bnd_ids]
X_bnd_e = X_test_ext[bnd_ids]
y_bnd   = y_test[bnd_ids]

bnd_csv = ATTACK_DIR / "boundary_search.csv"
if seed_SM.report("boundary_eval") and bnd_csv.exists():
    bnd_df = pd.read_csv(bnd_csv)
else:
    rows = []
    for model_obj, name, X_ev, bnds, msk in [
        (baseline,    "Baseline",     X_bnd_p, paper_bounds, paper_mask),
        (paper_model, "Paper Defense",X_bnd_p, paper_bounds, paper_mask),
        (v9_model,    "V9 Defense",   X_bnd_e, ext_bounds,   ext_mask),
    ]:
        model_obj.eval()
        t_bnd = time.perf_counter()
        X_adv_out, rec_df, bnd_stats = label_only_boundary_search_fixed(
            model_obj, X_ev, y_bnd, CONFIG["attack_epsilon"], bnds, msk, RUN_SEED,
            init_trials=CONFIG["boundary_init_trials"], refine_steps=CONFIG["boundary_refine_steps"],
            X_ref_pool=X_train_paper if name != "V9 Defense" else X_train_ext,
            y_ref_pool=y_train_bal
        )
        atomic_save_df(rec_df, ATTACK_DIR / f"boundary_search_records_{name.lower().replace(' ', '_')}.csv")
        
        rows.append({"method": name, "attack": "Label-Only Boundary Search",
                      "seed": RUN_SEED, **bnd_stats,
                      "note": "label-only boundary-search diagnostic probe strictly within epsilon (NOT an attack benchmark)"})
        print(f"  Boundary {name} ({time.perf_counter()-t_bnd:.1f}s): clean_corr={bnd_stats['n_clean_correct']}  within_eps={bnd_stats['n_within_epsilon']}  init_rate={bnd_stats['initialization_success_rate']:.3f}")
    bnd_df = pd.DataFrame(rows)
    atomic_save_df(bnd_df, bnd_csv)
    seed_SM.mark_done("boundary_eval")

# ── 10. Transfer Attack (Target-Aligned Dedicated Surrogates) ─────────────────
tr_ids = eval_ids[:CONFIG["transfer_n"]]
y_tr   = y_test[tr_ids]

transfer_csv = ATTACK_DIR / "transfer.csv"
if seed_SM.report("transfer_eval") and transfer_csv.exists():
    transfer_df = pd.read_csv(transfer_csv)
else:
    rows = []
    set_seed(RUN_SEED + 50_000)
    
    # Surrogate A (Trained on Paper Feature Space -> attacks Baseline & Paper Defense)
    surrogate_paper = WiderNet(len(paper_idx_local))
    surrogate_paper, _ = train_with_resume(
        surrogate_paper, MODEL_DIR / "surrogate_paper",
        X_train_paper, y_train_bal, X_dev_paper, y_dev,
        CONFIG["surrogate_epochs"], RUN_SEED+50_000, "Surrogate (Paper)", CONFIG
    )
    surrogate_paper.eval()
    X_tr_clean_paper = X_test_paper[tr_ids]
    X_tr_adv_paper   = pgd_attack(surrogate_paper, X_tr_clean_paper, y_tr,
                                   CONFIG["attack_epsilon"], CONFIG["pgd_steps_eval"],
                                   CONFIG["pgd_restarts"], paper_bounds, paper_mask, RUN_SEED+50_000)

    # Surrogate B (Trained on V9 Extension Feature Space -> attacks V9 Defense)
    surrogate_v9 = WiderNet(len(extension_idx_local))
    surrogate_v9, _ = train_with_resume(
        surrogate_v9, MODEL_DIR / "surrogate_v9",
        X_train_ext, y_train_bal, X_dev_ext, y_dev,
        CONFIG["surrogate_epochs"], RUN_SEED+60_000, "Surrogate (V9)", CONFIG
    )
    surrogate_v9.eval()
    X_tr_clean_v9 = X_test_ext[tr_ids]
    X_tr_adv_v9   = pgd_attack(surrogate_v9, X_tr_clean_v9, y_tr,
                                CONFIG["attack_epsilon"], CONFIG["pgd_steps_eval"],
                                CONFIG["pgd_restarts"], ext_bounds, ext_mask, RUN_SEED+60_000)

    for model_obj, name, X_ev_clean, X_ev_adv in [
        (baseline,    "Baseline",     X_tr_clean_paper, X_tr_adv_paper),
        (paper_model, "Paper Defense",X_tr_clean_paper, X_tr_adv_paper),
        (v9_model,    "V9 Defense",   X_tr_clean_v9,    X_tr_adv_v9),
    ]:
        model_obj.eval()
        clean_p   = predict_model(model_obj, X_ev_clean)[0]
        adv_p, adv_prob = predict_model(model_obj, X_ev_adv)
        metrics   = compute_metrics(y_tr, adv_p, adv_prob)
        pert      = attack_perturbation_stats(X_ev_clean, X_ev_adv, y_tr, clean_p, adv_p)
        rows.append({"target": name, "attack": "Transfer PGD", "seed": RUN_SEED,
                      "transfer_asr": pert["attack_success_rate"],
                      **metrics, **pert})
        print(f"  Transfer->{name}: ASR={pert['attack_success_rate']:.4f}  acc={metrics['accuracy']:.4f}")

    transfer_df = pd.DataFrame(rows)
    atomic_save_df(transfer_df, transfer_csv)
    seed_SM.mark_done("transfer_eval")
    del surrogate_paper, surrogate_v9
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── 11. PGD Strength Sweep ────────────────────────────────────────────────────
pgd_sweep_csv = ATTACK_DIR / "pgd_strength_sweep.csv"
if seed_SM.report("pgd_sweep") and pgd_sweep_csv.exists():
    pgd_sweep_df = pd.read_csv(pgd_sweep_csv)
else:
    rows = []
    for n_steps in CONFIG["pgd_sweep_steps"]:
        for model_obj, name, X_ev, bnds, msk in [
            (baseline,    "Baseline",     X_eval_paper, paper_bounds, paper_mask),
            (paper_model, "Paper Defense",X_eval_paper, paper_bounds, paper_mask),
            (v9_model,    "V9 Defense",   X_eval_ext,   ext_bounds,   ext_mask),
        ]:
            model_obj.eval()
            X_adv = pgd_attack(model_obj, X_ev, y_eval, CONFIG["attack_epsilon"],
                               n_steps, 1, bnds, msk, RUN_SEED)
            adv_p = predict_model(model_obj, X_adv)[0]
            metrics = compute_metrics(y_eval, adv_p, None)
            rows.append({"method": name, "pgd_steps": n_steps, "seed": RUN_SEED,
                          "accuracy": metrics["accuracy"]})
    pgd_sweep_df = pd.DataFrame(rows)
    atomic_save_df(pgd_sweep_df, pgd_sweep_csv)
    seed_SM.mark_done("pgd_sweep")

# ── 12. Epsilon Sweep ─────────────────────────────────────────────────────────
eps_csv = ATTACK_DIR / "epsilon_sweep.csv"
if seed_SM.report("eps_sweep") and eps_csv.exists():
    eps_df = pd.read_csv(eps_csv)
else:
    rows = []
    for eps_val in CONFIG["eps_sweep"]:
        for model_obj, name, X_ev, bnds, msk in [
            (baseline,    "Baseline",     X_eval_paper, paper_bounds, paper_mask),
            (paper_model, "Paper Defense",X_eval_paper, paper_bounds, paper_mask),
            (v9_model,    "V9 Defense",   X_eval_ext,   ext_bounds,   ext_mask),
        ]:
            model_obj.eval()
            X_adv = pgd_attack(model_obj, X_ev, y_eval, eps_val,
                               CONFIG["pgd_steps_eval"], 1, bnds, msk, RUN_SEED)
            adv_p = predict_model(model_obj, X_adv)[0]
            metrics = compute_metrics(y_eval, adv_p, None)
            rows.append({"method": name, "epsilon": eps_val, "seed": RUN_SEED,
                          "accuracy": metrics["accuracy"]})
    eps_df = pd.DataFrame(rows)
    atomic_save_df(eps_df, eps_csv)
    seed_SM.mark_done("eps_sweep")

# ── 13. Gradient-Masking Diagnostic ───────────────────────────────────────────
def run_gradient_masking_diagnostic(seed_dir, cfg):
    seed_dir   = Path(seed_dir)
    atk_dir    = seed_dir / "attacks"
    indicators = []

    p = atk_dir / "pgd_strength_sweep.csv"
    if p.exists():
        sw = pd.read_csv(p)
        v9 = sw[sw["method"]=="V9 Defense"].sort_values("pgd_steps")
        if len(v9) >= 2:
            drop = v9["accuracy"].iloc[0] - v9["accuracy"].iloc[-1]
            if drop < 0.01:
                indicators.append(("PGD_sweep", "POSSIBLE MASKING SIGNAL", f"drop={drop:.4f}"))
            else:
                indicators.append(("PGD_sweep", "NO CLEAR EVIDENCE", f"drop={drop:.4f}"))

    p = atk_dir / "epsilon_sweep.csv"
    if p.exists():
        sw = pd.read_csv(p)
        v9 = sw[sw["method"]=="V9 Defense"].sort_values("epsilon")
        if len(v9) >= 2:
            drop = v9["accuracy"].iloc[0] - v9["accuracy"].iloc[-1]
            if drop < 0.02:
                indicators.append(("Epsilon_sweep", "POSSIBLE MASKING SIGNAL", f"drop={drop:.4f}"))
            else:
                indicators.append(("Epsilon_sweep", "NO CLEAR EVIDENCE", f"drop={drop:.4f}"))

    tp = atk_dir / "transfer.csv"; pp = atk_dir / "pgd.csv"
    if tp.exists() and pp.exists():
        tr  = pd.read_csv(tp); pg = pd.read_csv(pp)
        v9t = tr[tr.get("target","") == "V9 Defense"]["transfer_asr"].mean() if "transfer_asr" in tr.columns else None
        v9p = pg[pg["method"] == "V9 Defense"]["attack_success_rate"].mean() if "attack_success_rate" in pg.columns else None
        if v9t is not None and v9p is not None:
            if v9p > 0.05 and v9t < v9p * 0.5:
                indicators.append(("Transfer_vs_whitebox", "EVIDENCE CONSISTENT WITH ROBUSTNESS", f"whitebox={v9p:.3f} >> transfer={v9t:.3f}"))
            else:
                indicators.append(("Transfer_vs_whitebox", "NO CLEAR EVIDENCE", f"whitebox={v9p:.3f} transfer={v9t:.3f}"))

    masking = sum(1 for _, v, _ in indicators if "MASKING" in v)
    if masking >= 3:
        verdict = "POSSIBLE MASKING SIGNAL (multiple indicators)"
    elif masking >= 1:
        verdict = "WEAK MASKING SIGNAL (some indicators)"
    else:
        verdict = "NO CLEAR EVIDENCE OF GRADIENT MASKING"

    print("\nGRADIENT-MASKING DIAGNOSTIC (HEURISTIC EVIDENCE)")
    print("=" * 60)
    for name, v, note in indicators:
        print(f"  [{v}] {name}: {note}")
    print(f"\nDIAGNOSTIC VERDICT: {verdict}")
    print("=" * 60)
    return {"indicators": [(n, v, nt) for n, v, nt in indicators], "verdict": verdict}

diag_result = run_gradient_masking_diagnostic(SEED_DIR, CONFIG)
atomic_save_json(diag_result, SEED_DIR / "gradient_masking_diagnostic.json")

# ── 14. Aggregate & Save Seed Results ─────────────────────────────────────────
seed_rows = []
for df_obj in [fgsm_df, pgd_df, deepfool_df, sq_df, bnd_df, transfer_df]:
    if df_obj is not None and len(df_obj) > 0:
        seed_rows.append(df_obj)

clean_rows = []
for model_obj, name, X_ev, y_ev in [
    (baseline,    "Baseline",     X_test_paper, y_test),
    (paper_model, "Paper Defense",X_test_paper, y_test),
    (v9_model,    "V9 Defense",   X_test_ext,   y_test),
]:
    preds, probs = predict_model(model_obj, X_ev)
    m = compute_metrics(y_ev, preds, probs)
    clean_rows.append({"method": name, "attack": "Clean", "seed": RUN_SEED, **m})
seed_rows.insert(0, pd.DataFrame(clean_rows))

all_seed_df = pd.concat(seed_rows, ignore_index=True)
all_seed_df["seed"] = RUN_SEED
atomic_save_df(all_seed_df, SEED_DIR / "seed_results.csv")
print(f"  [SAVE] seed_results.csv ({len(all_seed_df)} rows)")

budget = {
    "seed":                   RUN_SEED,
    "baseline_epochs":        CONFIG["baseline_epochs"],
    "paper_epochs":           CONFIG["paper_epochs"],
    "v9_clean_epochs":        CONFIG["v9_clean_epochs"],
    "v9_online_epochs":       CONFIG["v9_online_epochs"],
    "batch_size":             CONFIG["batch_size"],
    "pgd_steps_eval":         CONFIG["pgd_steps_eval"],
    "pgd_steps_online":       CONFIG["pgd_steps_online"],
    "online_adv_weight":      CONFIG["online_adv_weight"],
    "device":                 str(DEVICE),
    "total_seconds":          time.perf_counter() - t_seed_start,
    "note":                   "V9 computational overhead includes dynamic per-batch PGD online gradient generation"
}
atomic_save_json(budget, SEED_DIR / "training_budget.json")
atomic_save_json({"seed": RUN_SEED, "status": "complete", "duration_sec": time.perf_counter() - t_seed_start}, SEED_DIR / "seed_complete.json")

t_seed_total = time.perf_counter() - t_seed_start
print(f"\n[DONE] Seed {RUN_SEED} completed in {t_seed_total/3600:.2f}h")

In [ ]:
# ======================================================================
# SEED 123 — FULLY SELF-CONTAINED INDEPENDENT EXECUTION
# ======================================================================
RUN_SEED = 123
set_seed(RUN_SEED)
t_seed_start = time.perf_counter()
print(f"\n======================================================================")
print(f"  RUNNING SEED 123")
print(f"======================================================================\n")

SEED_DIR   = OUTPUT_DIR / "seeds" / str(RUN_SEED)
ATTACK_DIR = SEED_DIR / "attacks"
MODEL_DIR  = SEED_DIR / "models"
LOG_DIR    = SEED_DIR / "logs"
for _d in [SEED_DIR, ATTACK_DIR, MODEL_DIR, LOG_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

atomic_save_json({"seed": RUN_SEED, "status": "started", "start_time": time.time()}, SEED_DIR / "seed_start.json")
atomic_save_json(CONFIG, SEED_DIR / "config.json")
_env_seed = {
    "python": sys.version, "torch": torch.__version__,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type=="cuda" else "cpu",
    "os": platform.platform(),
}
atomic_save_json(_env_seed, SEED_DIR / "environment.json")
seed_SM = StageManager(SEED_DIR)

# ── 1. Baseline Training ──────────────────────────────────────────────────────
set_seed(RUN_SEED)
baseline_dir = MODEL_DIR / "baseline"
if seed_SM.report("baseline_train") and (baseline_dir / "best.pt").exists():
    baseline = IDSNet(len(paper_idx_local)).to(DEVICE)
    baseline.load_state_dict(
        torch.load(baseline_dir / "best.pt", map_location=DEVICE, weights_only=False)["model_state_dict"]
    )
else:
    t_b = time.perf_counter()
    baseline = IDSNet(len(paper_idx_local))
    baseline, hist_baseline = train_with_resume(
        baseline, baseline_dir,
        X_train_paper, y_train_bal,
        X_dev_paper,   y_dev,
        CONFIG["baseline_epochs"], RUN_SEED, "Baseline", CONFIG
    )
    hist_baseline.to_csv(baseline_dir / "history.csv", index=False)
    seed_SM.mark_done("baseline_train", {"seconds": time.perf_counter()-t_b})
    print(f"  [OK] Baseline trained in {time.perf_counter()-t_b:.1f}s")

baseline.eval()
bl_preds, bl_probs = predict_model(baseline, X_test_paper)
bl_metrics = compute_metrics(y_test, bl_preds, bl_probs)
print(f"  [RESULT] Baseline clean acc={bl_metrics['accuracy']:.4f}  F1={bl_metrics['f1_weighted']:.4f}")

# ── 2. Paper Defense Training (Static FGSM + PGD + DeepFool Augmentation) ────
set_seed(RUN_SEED + 10_000)
paper_dir = MODEL_DIR / "paper_defense"
if seed_SM.report("paper_defense_train") and (paper_dir / "best.pt").exists():
    paper_model = IDSNet(len(paper_idx_local)).to(DEVICE)
    paper_model.load_state_dict(
        torch.load(paper_dir / "best.pt", map_location=DEVICE, weights_only=False)["model_state_dict"]
    )
else:
    t_p = time.perf_counter()
    aug_fgsm_n = min(CONFIG["paper_fgsm_n"], len(X_train_paper))
    aug_pgd_n  = min(CONFIG["paper_pgd_n"], len(X_train_paper))
    aug_df_n   = min(CONFIG["paper_deepfool_n"], len(X_train_paper))
    
    aug_model = IDSNet(len(paper_idx_local)).to(DEVICE)
    _warm_loader = make_loader_safe(X_train_paper[:aug_fgsm_n], y_train_bal[:aug_fgsm_n], CONFIG["batch_size"])
    _warm_opt    = torch.optim.Adam(aug_model.parameters(), lr=CONFIG["learning_rate"])
    aug_model.train()
    for _ in range(5):
        for _xb, _yb in _warm_loader:
            _xb, _yb = _xb.to(DEVICE, non_blocking=True), _yb.to(DEVICE, non_blocking=True)
            _warm_opt.zero_grad()
            nn.CrossEntropyLoss()(aug_model(_xb), _yb).backward()
            _warm_opt.step()
    aug_model.eval()
    
    # Generate static triad: FGSM + PGD + DeepFool
    X_aug_fgsm = fgsm_attack(aug_model, X_train_paper[:aug_fgsm_n], y_train_bal[:aug_fgsm_n],
                             CONFIG["attack_epsilon"], paper_bounds, paper_mask)
    X_aug_pgd  = pgd_attack(aug_model, X_train_paper[:aug_pgd_n], y_train_bal[:aug_pgd_n],
                            CONFIG["attack_epsilon"], 10, 1, paper_bounds, paper_mask, seed=RUN_SEED+10_000)
    try:
        X_aug_df = deepfool_via_art(aug_model, X_train_paper[:aug_df_n])
        X_aug_df = np.clip(X_aug_df, paper_bounds[0], paper_bounds[1])
    except Exception as e:
        print(f"  [WARN] DeepFool static aug skipped: {e}")
        X_aug_df = X_train_paper[:0]
        aug_df_n = 0
    
    X_tr_aug = np.vstack([X_train_paper, X_aug_fgsm, X_aug_pgd, X_aug_df])
    y_tr_aug = np.concatenate([y_train_bal, y_train_bal[:aug_fgsm_n], y_train_bal[:aug_pgd_n], y_train_bal[:aug_df_n]])
    
    del aug_model, X_aug_fgsm, X_aug_pgd, X_aug_df
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    paper_model = IDSNet(len(paper_idx_local))
    paper_model, hist_paper = train_with_resume(
        paper_model, paper_dir,
        X_tr_aug, y_tr_aug,
        X_dev_paper, y_dev,
        CONFIG["paper_epochs"], RUN_SEED+10_000, "Paper Defense (FGSM+PGD+DF Aug)", CONFIG
    )
    hist_paper.to_csv(paper_dir / "history.csv", index=False)
    seed_SM.mark_done("paper_defense_train", {"seconds": time.perf_counter()-t_p})
    print(f"  [OK] Paper Defense trained in {time.perf_counter()-t_p:.1f}s")

paper_model.eval()
pp_preds, pp_probs = predict_model(paper_model, X_test_paper)
pp_metrics = compute_metrics(y_test, pp_preds, pp_probs)
print(f"  [RESULT] Paper Defense clean acc={pp_metrics['accuracy']:.4f}  F1={pp_metrics['f1_weighted']:.4f}")

# ── 3. V9 Online Adversarial Training (Dynamic Per-Batch PGD on GPU) ──────────
set_seed(RUN_SEED + 20_000)
v9_dir = MODEL_DIR / "v9_defense"
if seed_SM.report("v9_train") and (v9_dir / "best.pt").exists():
    v9_model = IDSNet(len(extension_idx_local)).to(DEVICE)
    v9_model.load_state_dict(
        torch.load(v9_dir / "best.pt", map_location=DEVICE, weights_only=False)["model_state_dict"]
    )
else:
    t_v = time.perf_counter()
    v9_model = IDSNet(len(extension_idx_local))
    # Phase 1: clean pretraining
    v9_model, hist_v9_clean = train_with_resume(
        v9_model, v9_dir / "phase1_clean",
        X_train_ext, y_train_bal,
        X_dev_ext, y_dev,
        CONFIG["v9_clean_epochs"], RUN_SEED+20_000, "V9 Phase1 Clean", CONFIG
    )
    # Phase 2: online PGD training (pure GPU acceleration)
    v9_model, hist_v9_online = train_online_pgd_with_resume(
        v9_model, v9_dir,
        X_train_ext, y_train_bal,
        X_dev_ext, y_dev,
        CONFIG["v9_online_epochs"], RUN_SEED+20_000, "V9 Online PGD", CONFIG,
        ext_bounds, ext_mask
    )
    hist_v9_online.to_csv(v9_dir / "history_online.csv", index=False)
    seed_SM.mark_done("v9_train", {"seconds": time.perf_counter()-t_v})
    print(f"  [OK] V9 Defense trained in {time.perf_counter()-t_v:.1f}s")

v9_model.eval()
v9_preds, v9_probs = predict_model(v9_model, X_test_ext)
v9_metrics = compute_metrics(y_test, v9_preds, v9_probs)
print(f"  [RESULT] V9 Defense clean acc={v9_metrics['accuracy']:.4f}  F1={v9_metrics['f1_weighted']:.4f}")

# ── 4. Fixed Evaluation Sample Selection (shared across all models) ───────────
eval_ids_path = SEED_DIR / "eval_ids.npz"
if eval_ids_path.exists():
    eval_ids = np.load(eval_ids_path)["eval_ids"]
else:
    rng = np.random.RandomState(RUN_SEED)
    eval_ids = rng.choice(len(X_test_mi), min(CONFIG["eval_n"], len(X_test_mi)), replace=False)
    atomic_save_numpy(eval_ids_path, eval_ids=eval_ids)

X_eval_paper = X_test_paper[eval_ids]
X_eval_ext   = X_test_ext[eval_ids]
y_eval       = y_test[eval_ids]

bl_clean_p, _ = predict_model(baseline, X_eval_paper)
pp_clean_p, _ = predict_model(paper_model, X_eval_paper)
v9_clean_p, _ = predict_model(v9_model, X_eval_ext)

# ── 5. FGSM Evaluation ───────────────────────────────────────────────────────
fgsm_csv = ATTACK_DIR / "fgsm.csv"
if seed_SM.report("fgsm_eval") and fgsm_csv.exists():
    fgsm_df = pd.read_csv(fgsm_csv)
else:
    rows = []
    for model_obj, name, X_ev, bnds, msk, clean_p in [
        (baseline,    "Baseline",     X_eval_paper, paper_bounds, paper_mask, bl_clean_p),
        (paper_model, "Paper Defense",X_eval_paper, paper_bounds, paper_mask, pp_clean_p),
        (v9_model,    "V9 Defense",   X_eval_ext,   ext_bounds,   ext_mask,   v9_clean_p),
    ]:
        model_obj.eval()
        X_adv = fgsm_attack(model_obj, X_ev, y_eval, CONFIG["attack_epsilon"], bnds, msk)
        adv_p, adv_prob = predict_model(model_obj, X_adv)
        metrics = compute_metrics(y_eval, adv_p, adv_prob)
        pert    = attack_perturbation_stats(X_ev, X_adv, y_eval, clean_p, adv_p)
        rows.append({"method": name, "attack": "FGSM", "seed": RUN_SEED, **metrics, **pert})
        print(f"  FGSM {name}: acc={metrics['accuracy']:.4f}  ASR={pert['attack_success_rate']:.4f}")
    fgsm_df = pd.DataFrame(rows)
    atomic_save_df(fgsm_df, fgsm_csv)
    seed_SM.mark_done("fgsm_eval")

# ── 6. PGD Evaluation (100 steps) ─────────────────────────────────────────────
pgd_csv = ATTACK_DIR / "pgd.csv"
if seed_SM.report("pgd_eval") and pgd_csv.exists():
    pgd_df = pd.read_csv(pgd_csv)
else:
    rows = []
    for model_obj, name, X_ev, bnds, msk, clean_p in [
        (baseline,    "Baseline",     X_eval_paper, paper_bounds, paper_mask, bl_clean_p),
        (paper_model, "Paper Defense",X_eval_paper, paper_bounds, paper_mask, pp_clean_p),
        (v9_model,    "V9 Defense",   X_eval_ext,   ext_bounds,   ext_mask,   v9_clean_p),
    ]:
        model_obj.eval()
        X_adv = pgd_attack(model_obj, X_ev, y_eval,
                           CONFIG["attack_epsilon"], CONFIG["pgd_steps_eval"],
                           CONFIG["pgd_restarts"], bnds, msk, RUN_SEED)
        adv_p, adv_prob = predict_model(model_obj, X_adv)
        metrics = compute_metrics(y_eval, adv_p, adv_prob)
        pert    = attack_perturbation_stats(X_ev, X_adv, y_eval, clean_p, adv_p)
        rows.append({"method": name, "attack": "PGD", "seed": RUN_SEED, **metrics, **pert})
        print(f"  PGD {name}: acc={metrics['accuracy']:.4f}  ASR={pert['attack_success_rate']:.4f}")
    pgd_df = pd.DataFrame(rows)
    atomic_save_df(pgd_df, pgd_csv)
    seed_SM.mark_done("pgd_eval")

# ── 7. DeepFool Evaluation (unconstrained) ───────────────────────────────────
df_ids   = eval_ids[:CONFIG["deepfool_eval_n"]]
X_df_p   = X_test_paper[df_ids]
X_df_e   = X_test_ext[df_ids]
y_df     = y_test[df_ids]

deepfool_csv = ATTACK_DIR / "deepfool.csv"
if seed_SM.report("deepfool_eval") and deepfool_csv.exists():
    deepfool_df = pd.read_csv(deepfool_csv)
else:
    rows = []
    for model_obj, name, X_ev in [
        (baseline,    "Baseline",     X_df_p),
        (paper_model, "Paper Defense",X_df_p),
        (v9_model,    "V9 Defense",   X_df_e),
    ]:
        try:
            model_obj.eval()
            X_adv     = deepfool_via_art(model_obj, X_ev)
            adv_p, adv_prob = predict_model(model_obj, X_adv)
            clean_p   = predict_model(model_obj, X_ev)[0]
            metrics   = compute_metrics(y_df, adv_p, adv_prob)
            pert      = attack_perturbation_stats(X_ev, X_adv, y_df, clean_p, adv_p)
            rows.append({"method": name, "attack": "DeepFool (unconstrained)",
                          "seed": RUN_SEED, **metrics, **pert,
                          "note": "unconstrained minimum-perturbation — do not compare directly with bounded attacks"})
            print(f"  DeepFool {name}: acc={metrics['accuracy']:.4f}")
        except Exception as e:
            print(f"  [WARN] DeepFool {name} skipped: {e}")
    deepfool_df = pd.DataFrame(rows)
    atomic_save_df(deepfool_df, deepfool_csv)
    seed_SM.mark_done("deepfool_eval")

# ── 8. Fast Vectorized Square-style Attack Evaluation ─────────────────────────
sq_ids = eval_ids[:CONFIG["square_eval_n"]]
X_sq_p = X_test_paper[sq_ids]
X_sq_e = X_test_ext[sq_ids]
y_sq   = y_test[sq_ids]

sq_csv = ATTACK_DIR / "square_style.csv"
if seed_SM.report("square_eval") and sq_csv.exists():
    sq_df = pd.read_csv(sq_csv)
else:
    rows = []
    for model_obj, name, X_ev, bnds in [
        (baseline,    "Baseline",     X_sq_p, paper_bounds),
        (paper_model, "Paper Defense",X_sq_p, paper_bounds),
        (v9_model,    "V9 Defense",   X_sq_e, ext_bounds),
    ]:
        model_obj.eval()
        t_sq = time.perf_counter()
        X_adv     = square_style_attack(model_obj, X_ev, CONFIG["attack_epsilon"],
                                        bounds=bnds, n_iters=CONFIG["square_iterations"], seed=RUN_SEED)
        clean_p   = predict_model(model_obj, X_ev)[0]
        adv_p, adv_prob = predict_model(model_obj, X_adv)
        metrics   = compute_metrics(y_sq, adv_p, adv_prob)
        pert      = attack_perturbation_stats(X_ev, X_adv, y_sq, clean_p, adv_p)
        rows.append({"method": name, "attack": "Square-style Attack",
                      "seed": RUN_SEED, **metrics, **pert,
                      "note": "custom score-based gradient-free heuristic (NOT canonical Square Attack)"})
        print(f"  Square-style {name} ({time.perf_counter()-t_sq:.1f}s): acc={metrics['accuracy']:.4f}  ASR={pert['attack_success_rate']:.4f}")
    sq_df = pd.DataFrame(rows)
    atomic_save_df(sq_df, sq_csv)
    seed_SM.mark_done("square_eval")

# ── 9. Strictly-Within-Epsilon Label-Only Boundary Search Diagnostic ──────────
bnd_ids = eval_ids[:CONFIG["boundary_n"]]
X_bnd_p = X_test_paper[bnd_ids]
X_bnd_e = X_test_ext[bnd_ids]
y_bnd   = y_test[bnd_ids]

bnd_csv = ATTACK_DIR / "boundary_search.csv"
if seed_SM.report("boundary_eval") and bnd_csv.exists():
    bnd_df = pd.read_csv(bnd_csv)
else:
    rows = []
    for model_obj, name, X_ev, bnds, msk in [
        (baseline,    "Baseline",     X_bnd_p, paper_bounds, paper_mask),
        (paper_model, "Paper Defense",X_bnd_p, paper_bounds, paper_mask),
        (v9_model,    "V9 Defense",   X_bnd_e, ext_bounds,   ext_mask),
    ]:
        model_obj.eval()
        t_bnd = time.perf_counter()
        X_adv_out, rec_df, bnd_stats = label_only_boundary_search_fixed(
            model_obj, X_ev, y_bnd, CONFIG["attack_epsilon"], bnds, msk, RUN_SEED,
            init_trials=CONFIG["boundary_init_trials"], refine_steps=CONFIG["boundary_refine_steps"],
            X_ref_pool=X_train_paper if name != "V9 Defense" else X_train_ext,
            y_ref_pool=y_train_bal
        )
        atomic_save_df(rec_df, ATTACK_DIR / f"boundary_search_records_{name.lower().replace(' ', '_')}.csv")
        
        rows.append({"method": name, "attack": "Label-Only Boundary Search",
                      "seed": RUN_SEED, **bnd_stats,
                      "note": "label-only boundary-search diagnostic probe strictly within epsilon (NOT an attack benchmark)"})
        print(f"  Boundary {name} ({time.perf_counter()-t_bnd:.1f}s): clean_corr={bnd_stats['n_clean_correct']}  within_eps={bnd_stats['n_within_epsilon']}  init_rate={bnd_stats['initialization_success_rate']:.3f}")
    bnd_df = pd.DataFrame(rows)
    atomic_save_df(bnd_df, bnd_csv)
    seed_SM.mark_done("boundary_eval")

# ── 10. Transfer Attack (Target-Aligned Dedicated Surrogates) ─────────────────
tr_ids = eval_ids[:CONFIG["transfer_n"]]
y_tr   = y_test[tr_ids]

transfer_csv = ATTACK_DIR / "transfer.csv"
if seed_SM.report("transfer_eval") and transfer_csv.exists():
    transfer_df = pd.read_csv(transfer_csv)
else:
    rows = []
    set_seed(RUN_SEED + 50_000)
    
    # Surrogate A (Trained on Paper Feature Space -> attacks Baseline & Paper Defense)
    surrogate_paper = WiderNet(len(paper_idx_local))
    surrogate_paper, _ = train_with_resume(
        surrogate_paper, MODEL_DIR / "surrogate_paper",
        X_train_paper, y_train_bal, X_dev_paper, y_dev,
        CONFIG["surrogate_epochs"], RUN_SEED+50_000, "Surrogate (Paper)", CONFIG
    )
    surrogate_paper.eval()
    X_tr_clean_paper = X_test_paper[tr_ids]
    X_tr_adv_paper   = pgd_attack(surrogate_paper, X_tr_clean_paper, y_tr,
                                   CONFIG["attack_epsilon"], CONFIG["pgd_steps_eval"],
                                   CONFIG["pgd_restarts"], paper_bounds, paper_mask, RUN_SEED+50_000)

    # Surrogate B (Trained on V9 Extension Feature Space -> attacks V9 Defense)
    surrogate_v9 = WiderNet(len(extension_idx_local))
    surrogate_v9, _ = train_with_resume(
        surrogate_v9, MODEL_DIR / "surrogate_v9",
        X_train_ext, y_train_bal, X_dev_ext, y_dev,
        CONFIG["surrogate_epochs"], RUN_SEED+60_000, "Surrogate (V9)", CONFIG
    )
    surrogate_v9.eval()
    X_tr_clean_v9 = X_test_ext[tr_ids]
    X_tr_adv_v9   = pgd_attack(surrogate_v9, X_tr_clean_v9, y_tr,
                                CONFIG["attack_epsilon"], CONFIG["pgd_steps_eval"],
                                CONFIG["pgd_restarts"], ext_bounds, ext_mask, RUN_SEED+60_000)

    for model_obj, name, X_ev_clean, X_ev_adv in [
        (baseline,    "Baseline",     X_tr_clean_paper, X_tr_adv_paper),
        (paper_model, "Paper Defense",X_tr_clean_paper, X_tr_adv_paper),
        (v9_model,    "V9 Defense",   X_tr_clean_v9,    X_tr_adv_v9),
    ]:
        model_obj.eval()
        clean_p   = predict_model(model_obj, X_ev_clean)[0]
        adv_p, adv_prob = predict_model(model_obj, X_ev_adv)
        metrics   = compute_metrics(y_tr, adv_p, adv_prob)
        pert      = attack_perturbation_stats(X_ev_clean, X_ev_adv, y_tr, clean_p, adv_p)
        rows.append({"target": name, "attack": "Transfer PGD", "seed": RUN_SEED,
                      "transfer_asr": pert["attack_success_rate"],
                      **metrics, **pert})
        print(f"  Transfer->{name}: ASR={pert['attack_success_rate']:.4f}  acc={metrics['accuracy']:.4f}")

    transfer_df = pd.DataFrame(rows)
    atomic_save_df(transfer_df, transfer_csv)
    seed_SM.mark_done("transfer_eval")
    del surrogate_paper, surrogate_v9
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── 11. PGD Strength Sweep ────────────────────────────────────────────────────
pgd_sweep_csv = ATTACK_DIR / "pgd_strength_sweep.csv"
if seed_SM.report("pgd_sweep") and pgd_sweep_csv.exists():
    pgd_sweep_df = pd.read_csv(pgd_sweep_csv)
else:
    rows = []
    for n_steps in CONFIG["pgd_sweep_steps"]:
        for model_obj, name, X_ev, bnds, msk in [
            (baseline,    "Baseline",     X_eval_paper, paper_bounds, paper_mask),
            (paper_model, "Paper Defense",X_eval_paper, paper_bounds, paper_mask),
            (v9_model,    "V9 Defense",   X_eval_ext,   ext_bounds,   ext_mask),
        ]:
            model_obj.eval()
            X_adv = pgd_attack(model_obj, X_ev, y_eval, CONFIG["attack_epsilon"],
                               n_steps, 1, bnds, msk, RUN_SEED)
            adv_p = predict_model(model_obj, X_adv)[0]
            metrics = compute_metrics(y_eval, adv_p, None)
            rows.append({"method": name, "pgd_steps": n_steps, "seed": RUN_SEED,
                          "accuracy": metrics["accuracy"]})
    pgd_sweep_df = pd.DataFrame(rows)
    atomic_save_df(pgd_sweep_df, pgd_sweep_csv)
    seed_SM.mark_done("pgd_sweep")

# ── 12. Epsilon Sweep ─────────────────────────────────────────────────────────
eps_csv = ATTACK_DIR / "epsilon_sweep.csv"
if seed_SM.report("eps_sweep") and eps_csv.exists():
    eps_df = pd.read_csv(eps_csv)
else:
    rows = []
    for eps_val in CONFIG["eps_sweep"]:
        for model_obj, name, X_ev, bnds, msk in [
            (baseline,    "Baseline",     X_eval_paper, paper_bounds, paper_mask),
            (paper_model, "Paper Defense",X_eval_paper, paper_bounds, paper_mask),
            (v9_model,    "V9 Defense",   X_eval_ext,   ext_bounds,   ext_mask),
        ]:
            model_obj.eval()
            X_adv = pgd_attack(model_obj, X_ev, y_eval, eps_val,
                               CONFIG["pgd_steps_eval"], 1, bnds, msk, RUN_SEED)
            adv_p = predict_model(model_obj, X_adv)[0]
            metrics = compute_metrics(y_eval, adv_p, None)
            rows.append({"method": name, "epsilon": eps_val, "seed": RUN_SEED,
                          "accuracy": metrics["accuracy"]})
    eps_df = pd.DataFrame(rows)
    atomic_save_df(eps_df, eps_csv)
    seed_SM.mark_done("eps_sweep")

# ── 13. Gradient-Masking Diagnostic ───────────────────────────────────────────
def run_gradient_masking_diagnostic(seed_dir, cfg):
    seed_dir   = Path(seed_dir)
    atk_dir    = seed_dir / "attacks"
    indicators = []

    p = atk_dir / "pgd_strength_sweep.csv"
    if p.exists():
        sw = pd.read_csv(p)
        v9 = sw[sw["method"]=="V9 Defense"].sort_values("pgd_steps")
        if len(v9) >= 2:
            drop = v9["accuracy"].iloc[0] - v9["accuracy"].iloc[-1]
            if drop < 0.01:
                indicators.append(("PGD_sweep", "POSSIBLE MASKING SIGNAL", f"drop={drop:.4f}"))
            else:
                indicators.append(("PGD_sweep", "NO CLEAR EVIDENCE", f"drop={drop:.4f}"))

    p = atk_dir / "epsilon_sweep.csv"
    if p.exists():
        sw = pd.read_csv(p)
        v9 = sw[sw["method"]=="V9 Defense"].sort_values("epsilon")
        if len(v9) >= 2:
            drop = v9["accuracy"].iloc[0] - v9["accuracy"].iloc[-1]
            if drop < 0.02:
                indicators.append(("Epsilon_sweep", "POSSIBLE MASKING SIGNAL", f"drop={drop:.4f}"))
            else:
                indicators.append(("Epsilon_sweep", "NO CLEAR EVIDENCE", f"drop={drop:.4f}"))

    tp = atk_dir / "transfer.csv"; pp = atk_dir / "pgd.csv"
    if tp.exists() and pp.exists():
        tr  = pd.read_csv(tp); pg = pd.read_csv(pp)
        v9t = tr[tr.get("target","") == "V9 Defense"]["transfer_asr"].mean() if "transfer_asr" in tr.columns else None
        v9p = pg[pg["method"] == "V9 Defense"]["attack_success_rate"].mean() if "attack_success_rate" in pg.columns else None
        if v9t is not None and v9p is not None:
            if v9p > 0.05 and v9t < v9p * 0.5:
                indicators.append(("Transfer_vs_whitebox", "EVIDENCE CONSISTENT WITH ROBUSTNESS", f"whitebox={v9p:.3f} >> transfer={v9t:.3f}"))
            else:
                indicators.append(("Transfer_vs_whitebox", "NO CLEAR EVIDENCE", f"whitebox={v9p:.3f} transfer={v9t:.3f}"))

    masking = sum(1 for _, v, _ in indicators if "MASKING" in v)
    if masking >= 3:
        verdict = "POSSIBLE MASKING SIGNAL (multiple indicators)"
    elif masking >= 1:
        verdict = "WEAK MASKING SIGNAL (some indicators)"
    else:
        verdict = "NO CLEAR EVIDENCE OF GRADIENT MASKING"

    print("\nGRADIENT-MASKING DIAGNOSTIC (HEURISTIC EVIDENCE)")
    print("=" * 60)
    for name, v, note in indicators:
        print(f"  [{v}] {name}: {note}")
    print(f"\nDIAGNOSTIC VERDICT: {verdict}")
    print("=" * 60)
    return {"indicators": [(n, v, nt) for n, v, nt in indicators], "verdict": verdict}

diag_result = run_gradient_masking_diagnostic(SEED_DIR, CONFIG)
atomic_save_json(diag_result, SEED_DIR / "gradient_masking_diagnostic.json")

# ── 14. Aggregate & Save Seed Results ─────────────────────────────────────────
seed_rows = []
for df_obj in [fgsm_df, pgd_df, deepfool_df, sq_df, bnd_df, transfer_df]:
    if df_obj is not None and len(df_obj) > 0:
        seed_rows.append(df_obj)

clean_rows = []
for model_obj, name, X_ev, y_ev in [
    (baseline,    "Baseline",     X_test_paper, y_test),
    (paper_model, "Paper Defense",X_test_paper, y_test),
    (v9_model,    "V9 Defense",   X_test_ext,   y_test),
]:
    preds, probs = predict_model(model_obj, X_ev)
    m = compute_metrics(y_ev, preds, probs)
    clean_rows.append({"method": name, "attack": "Clean", "seed": RUN_SEED, **m})
seed_rows.insert(0, pd.DataFrame(clean_rows))

all_seed_df = pd.concat(seed_rows, ignore_index=True)
all_seed_df["seed"] = RUN_SEED
atomic_save_df(all_seed_df, SEED_DIR / "seed_results.csv")
print(f"  [SAVE] seed_results.csv ({len(all_seed_df)} rows)")

budget = {
    "seed":                   RUN_SEED,
    "baseline_epochs":        CONFIG["baseline_epochs"],
    "paper_epochs":           CONFIG["paper_epochs"],
    "v9_clean_epochs":        CONFIG["v9_clean_epochs"],
    "v9_online_epochs":       CONFIG["v9_online_epochs"],
    "batch_size":             CONFIG["batch_size"],
    "pgd_steps_eval":         CONFIG["pgd_steps_eval"],
    "pgd_steps_online":       CONFIG["pgd_steps_online"],
    "online_adv_weight":      CONFIG["online_adv_weight"],
    "device":                 str(DEVICE),
    "total_seconds":          time.perf_counter() - t_seed_start,
    "note":                   "V9 computational overhead includes dynamic per-batch PGD online gradient generation"
}
atomic_save_json(budget, SEED_DIR / "training_budget.json")
atomic_save_json({"seed": RUN_SEED, "status": "complete", "duration_sec": time.perf_counter() - t_seed_start}, SEED_DIR / "seed_complete.json")

t_seed_total = time.perf_counter() - t_seed_start
print(f"\n[DONE] Seed {RUN_SEED} completed in {t_seed_total/3600:.2f}h")

In [ ]:
# ======================================================================
# SEED 2024 — FULLY SELF-CONTAINED INDEPENDENT EXECUTION
# ======================================================================
RUN_SEED = 2024
set_seed(RUN_SEED)
t_seed_start = time.perf_counter()
print(f"\n======================================================================")
print(f"  RUNNING SEED 2024")
print(f"======================================================================\n")

SEED_DIR   = OUTPUT_DIR / "seeds" / str(RUN_SEED)
ATTACK_DIR = SEED_DIR / "attacks"
MODEL_DIR  = SEED_DIR / "models"
LOG_DIR    = SEED_DIR / "logs"
for _d in [SEED_DIR, ATTACK_DIR, MODEL_DIR, LOG_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

atomic_save_json({"seed": RUN_SEED, "status": "started", "start_time": time.time()}, SEED_DIR / "seed_start.json")
atomic_save_json(CONFIG, SEED_DIR / "config.json")
_env_seed = {
    "python": sys.version, "torch": torch.__version__,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type=="cuda" else "cpu",
    "os": platform.platform(),
}
atomic_save_json(_env_seed, SEED_DIR / "environment.json")
seed_SM = StageManager(SEED_DIR)

# ── 1. Baseline Training ──────────────────────────────────────────────────────
set_seed(RUN_SEED)
baseline_dir = MODEL_DIR / "baseline"
if seed_SM.report("baseline_train") and (baseline_dir / "best.pt").exists():
    baseline = IDSNet(len(paper_idx_local)).to(DEVICE)
    baseline.load_state_dict(
        torch.load(baseline_dir / "best.pt", map_location=DEVICE, weights_only=False)["model_state_dict"]
    )
else:
    t_b = time.perf_counter()
    baseline = IDSNet(len(paper_idx_local))
    baseline, hist_baseline = train_with_resume(
        baseline, baseline_dir,
        X_train_paper, y_train_bal,
        X_dev_paper,   y_dev,
        CONFIG["baseline_epochs"], RUN_SEED, "Baseline", CONFIG
    )
    hist_baseline.to_csv(baseline_dir / "history.csv", index=False)
    seed_SM.mark_done("baseline_train", {"seconds": time.perf_counter()-t_b})
    print(f"  [OK] Baseline trained in {time.perf_counter()-t_b:.1f}s")

baseline.eval()
bl_preds, bl_probs = predict_model(baseline, X_test_paper)
bl_metrics = compute_metrics(y_test, bl_preds, bl_probs)
print(f"  [RESULT] Baseline clean acc={bl_metrics['accuracy']:.4f}  F1={bl_metrics['f1_weighted']:.4f}")

# ── 2. Paper Defense Training (Static FGSM + PGD + DeepFool Augmentation) ────
set_seed(RUN_SEED + 10_000)
paper_dir = MODEL_DIR / "paper_defense"
if seed_SM.report("paper_defense_train") and (paper_dir / "best.pt").exists():
    paper_model = IDSNet(len(paper_idx_local)).to(DEVICE)
    paper_model.load_state_dict(
        torch.load(paper_dir / "best.pt", map_location=DEVICE, weights_only=False)["model_state_dict"]
    )
else:
    t_p = time.perf_counter()
    aug_fgsm_n = min(CONFIG["paper_fgsm_n"], len(X_train_paper))
    aug_pgd_n  = min(CONFIG["paper_pgd_n"], len(X_train_paper))
    aug_df_n   = min(CONFIG["paper_deepfool_n"], len(X_train_paper))
    
    aug_model = IDSNet(len(paper_idx_local)).to(DEVICE)
    _warm_loader = make_loader_safe(X_train_paper[:aug_fgsm_n], y_train_bal[:aug_fgsm_n], CONFIG["batch_size"])
    _warm_opt    = torch.optim.Adam(aug_model.parameters(), lr=CONFIG["learning_rate"])
    aug_model.train()
    for _ in range(5):
        for _xb, _yb in _warm_loader:
            _xb, _yb = _xb.to(DEVICE, non_blocking=True), _yb.to(DEVICE, non_blocking=True)
            _warm_opt.zero_grad()
            nn.CrossEntropyLoss()(aug_model(_xb), _yb).backward()
            _warm_opt.step()
    aug_model.eval()
    
    # Generate static triad: FGSM + PGD + DeepFool
    X_aug_fgsm = fgsm_attack(aug_model, X_train_paper[:aug_fgsm_n], y_train_bal[:aug_fgsm_n],
                             CONFIG["attack_epsilon"], paper_bounds, paper_mask)
    X_aug_pgd  = pgd_attack(aug_model, X_train_paper[:aug_pgd_n], y_train_bal[:aug_pgd_n],
                            CONFIG["attack_epsilon"], 10, 1, paper_bounds, paper_mask, seed=RUN_SEED+10_000)
    try:
        X_aug_df = deepfool_via_art(aug_model, X_train_paper[:aug_df_n])
        X_aug_df = np.clip(X_aug_df, paper_bounds[0], paper_bounds[1])
    except Exception as e:
        print(f"  [WARN] DeepFool static aug skipped: {e}")
        X_aug_df = X_train_paper[:0]
        aug_df_n = 0
    
    X_tr_aug = np.vstack([X_train_paper, X_aug_fgsm, X_aug_pgd, X_aug_df])
    y_tr_aug = np.concatenate([y_train_bal, y_train_bal[:aug_fgsm_n], y_train_bal[:aug_pgd_n], y_train_bal[:aug_df_n]])
    
    del aug_model, X_aug_fgsm, X_aug_pgd, X_aug_df
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    paper_model = IDSNet(len(paper_idx_local))
    paper_model, hist_paper = train_with_resume(
        paper_model, paper_dir,
        X_tr_aug, y_tr_aug,
        X_dev_paper, y_dev,
        CONFIG["paper_epochs"], RUN_SEED+10_000, "Paper Defense (FGSM+PGD+DF Aug)", CONFIG
    )
    hist_paper.to_csv(paper_dir / "history.csv", index=False)
    seed_SM.mark_done("paper_defense_train", {"seconds": time.perf_counter()-t_p})
    print(f"  [OK] Paper Defense trained in {time.perf_counter()-t_p:.1f}s")

paper_model.eval()
pp_preds, pp_probs = predict_model(paper_model, X_test_paper)
pp_metrics = compute_metrics(y_test, pp_preds, pp_probs)
print(f"  [RESULT] Paper Defense clean acc={pp_metrics['accuracy']:.4f}  F1={pp_metrics['f1_weighted']:.4f}")

# ── 3. V9 Online Adversarial Training (Dynamic Per-Batch PGD on GPU) ──────────
set_seed(RUN_SEED + 20_000)
v9_dir = MODEL_DIR / "v9_defense"
if seed_SM.report("v9_train") and (v9_dir / "best.pt").exists():
    v9_model = IDSNet(len(extension_idx_local)).to(DEVICE)
    v9_model.load_state_dict(
        torch.load(v9_dir / "best.pt", map_location=DEVICE, weights_only=False)["model_state_dict"]
    )
else:
    t_v = time.perf_counter()
    v9_model = IDSNet(len(extension_idx_local))
    # Phase 1: clean pretraining
    v9_model, hist_v9_clean = train_with_resume(
        v9_model, v9_dir / "phase1_clean",
        X_train_ext, y_train_bal,
        X_dev_ext, y_dev,
        CONFIG["v9_clean_epochs"], RUN_SEED+20_000, "V9 Phase1 Clean", CONFIG
    )
    # Phase 2: online PGD training (pure GPU acceleration)
    v9_model, hist_v9_online = train_online_pgd_with_resume(
        v9_model, v9_dir,
        X_train_ext, y_train_bal,
        X_dev_ext, y_dev,
        CONFIG["v9_online_epochs"], RUN_SEED+20_000, "V9 Online PGD", CONFIG,
        ext_bounds, ext_mask
    )
    hist_v9_online.to_csv(v9_dir / "history_online.csv", index=False)
    seed_SM.mark_done("v9_train", {"seconds": time.perf_counter()-t_v})
    print(f"  [OK] V9 Defense trained in {time.perf_counter()-t_v:.1f}s")

v9_model.eval()
v9_preds, v9_probs = predict_model(v9_model, X_test_ext)
v9_metrics = compute_metrics(y_test, v9_preds, v9_probs)
print(f"  [RESULT] V9 Defense clean acc={v9_metrics['accuracy']:.4f}  F1={v9_metrics['f1_weighted']:.4f}")

# ── 4. Fixed Evaluation Sample Selection (shared across all models) ───────────
eval_ids_path = SEED_DIR / "eval_ids.npz"
if eval_ids_path.exists():
    eval_ids = np.load(eval_ids_path)["eval_ids"]
else:
    rng = np.random.RandomState(RUN_SEED)
    eval_ids = rng.choice(len(X_test_mi), min(CONFIG["eval_n"], len(X_test_mi)), replace=False)
    atomic_save_numpy(eval_ids_path, eval_ids=eval_ids)

X_eval_paper = X_test_paper[eval_ids]
X_eval_ext   = X_test_ext[eval_ids]
y_eval       = y_test[eval_ids]

bl_clean_p, _ = predict_model(baseline, X_eval_paper)
pp_clean_p, _ = predict_model(paper_model, X_eval_paper)
v9_clean_p, _ = predict_model(v9_model, X_eval_ext)

# ── 5. FGSM Evaluation ───────────────────────────────────────────────────────
fgsm_csv = ATTACK_DIR / "fgsm.csv"
if seed_SM.report("fgsm_eval") and fgsm_csv.exists():
    fgsm_df = pd.read_csv(fgsm_csv)
else:
    rows = []
    for model_obj, name, X_ev, bnds, msk, clean_p in [
        (baseline,    "Baseline",     X_eval_paper, paper_bounds, paper_mask, bl_clean_p),
        (paper_model, "Paper Defense",X_eval_paper, paper_bounds, paper_mask, pp_clean_p),
        (v9_model,    "V9 Defense",   X_eval_ext,   ext_bounds,   ext_mask,   v9_clean_p),
    ]:
        model_obj.eval()
        X_adv = fgsm_attack(model_obj, X_ev, y_eval, CONFIG["attack_epsilon"], bnds, msk)
        adv_p, adv_prob = predict_model(model_obj, X_adv)
        metrics = compute_metrics(y_eval, adv_p, adv_prob)
        pert    = attack_perturbation_stats(X_ev, X_adv, y_eval, clean_p, adv_p)
        rows.append({"method": name, "attack": "FGSM", "seed": RUN_SEED, **metrics, **pert})
        print(f"  FGSM {name}: acc={metrics['accuracy']:.4f}  ASR={pert['attack_success_rate']:.4f}")
    fgsm_df = pd.DataFrame(rows)
    atomic_save_df(fgsm_df, fgsm_csv)
    seed_SM.mark_done("fgsm_eval")

# ── 6. PGD Evaluation (100 steps) ─────────────────────────────────────────────
pgd_csv = ATTACK_DIR / "pgd.csv"
if seed_SM.report("pgd_eval") and pgd_csv.exists():
    pgd_df = pd.read_csv(pgd_csv)
else:
    rows = []
    for model_obj, name, X_ev, bnds, msk, clean_p in [
        (baseline,    "Baseline",     X_eval_paper, paper_bounds, paper_mask, bl_clean_p),
        (paper_model, "Paper Defense",X_eval_paper, paper_bounds, paper_mask, pp_clean_p),
        (v9_model,    "V9 Defense",   X_eval_ext,   ext_bounds,   ext_mask,   v9_clean_p),
    ]:
        model_obj.eval()
        X_adv = pgd_attack(model_obj, X_ev, y_eval,
                           CONFIG["attack_epsilon"], CONFIG["pgd_steps_eval"],
                           CONFIG["pgd_restarts"], bnds, msk, RUN_SEED)
        adv_p, adv_prob = predict_model(model_obj, X_adv)
        metrics = compute_metrics(y_eval, adv_p, adv_prob)
        pert    = attack_perturbation_stats(X_ev, X_adv, y_eval, clean_p, adv_p)
        rows.append({"method": name, "attack": "PGD", "seed": RUN_SEED, **metrics, **pert})
        print(f"  PGD {name}: acc={metrics['accuracy']:.4f}  ASR={pert['attack_success_rate']:.4f}")
    pgd_df = pd.DataFrame(rows)
    atomic_save_df(pgd_df, pgd_csv)
    seed_SM.mark_done("pgd_eval")

# ── 7. DeepFool Evaluation (unconstrained) ───────────────────────────────────
df_ids   = eval_ids[:CONFIG["deepfool_eval_n"]]
X_df_p   = X_test_paper[df_ids]
X_df_e   = X_test_ext[df_ids]
y_df     = y_test[df_ids]

deepfool_csv = ATTACK_DIR / "deepfool.csv"
if seed_SM.report("deepfool_eval") and deepfool_csv.exists():
    deepfool_df = pd.read_csv(deepfool_csv)
else:
    rows = []
    for model_obj, name, X_ev in [
        (baseline,    "Baseline",     X_df_p),
        (paper_model, "Paper Defense",X_df_p),
        (v9_model,    "V9 Defense",   X_df_e),
    ]:
        try:
            model_obj.eval()
            X_adv     = deepfool_via_art(model_obj, X_ev)
            adv_p, adv_prob = predict_model(model_obj, X_adv)
            clean_p   = predict_model(model_obj, X_ev)[0]
            metrics   = compute_metrics(y_df, adv_p, adv_prob)
            pert      = attack_perturbation_stats(X_ev, X_adv, y_df, clean_p, adv_p)
            rows.append({"method": name, "attack": "DeepFool (unconstrained)",
                          "seed": RUN_SEED, **metrics, **pert,
                          "note": "unconstrained minimum-perturbation — do not compare directly with bounded attacks"})
            print(f"  DeepFool {name}: acc={metrics['accuracy']:.4f}")
        except Exception as e:
            print(f"  [WARN] DeepFool {name} skipped: {e}")
    deepfool_df = pd.DataFrame(rows)
    atomic_save_df(deepfool_df, deepfool_csv)
    seed_SM.mark_done("deepfool_eval")

# ── 8. Fast Vectorized Square-style Attack Evaluation ─────────────────────────
sq_ids = eval_ids[:CONFIG["square_eval_n"]]
X_sq_p = X_test_paper[sq_ids]
X_sq_e = X_test_ext[sq_ids]
y_sq   = y_test[sq_ids]

sq_csv = ATTACK_DIR / "square_style.csv"
if seed_SM.report("square_eval") and sq_csv.exists():
    sq_df = pd.read_csv(sq_csv)
else:
    rows = []
    for model_obj, name, X_ev, bnds in [
        (baseline,    "Baseline",     X_sq_p, paper_bounds),
        (paper_model, "Paper Defense",X_sq_p, paper_bounds),
        (v9_model,    "V9 Defense",   X_sq_e, ext_bounds),
    ]:
        model_obj.eval()
        t_sq = time.perf_counter()
        X_adv     = square_style_attack(model_obj, X_ev, CONFIG["attack_epsilon"],
                                        bounds=bnds, n_iters=CONFIG["square_iterations"], seed=RUN_SEED)
        clean_p   = predict_model(model_obj, X_ev)[0]
        adv_p, adv_prob = predict_model(model_obj, X_adv)
        metrics   = compute_metrics(y_sq, adv_p, adv_prob)
        pert      = attack_perturbation_stats(X_ev, X_adv, y_sq, clean_p, adv_p)
        rows.append({"method": name, "attack": "Square-style Attack",
                      "seed": RUN_SEED, **metrics, **pert,
                      "note": "custom score-based gradient-free heuristic (NOT canonical Square Attack)"})
        print(f"  Square-style {name} ({time.perf_counter()-t_sq:.1f}s): acc={metrics['accuracy']:.4f}  ASR={pert['attack_success_rate']:.4f}")
    sq_df = pd.DataFrame(rows)
    atomic_save_df(sq_df, sq_csv)
    seed_SM.mark_done("square_eval")

# ── 9. Strictly-Within-Epsilon Label-Only Boundary Search Diagnostic ──────────
bnd_ids = eval_ids[:CONFIG["boundary_n"]]
X_bnd_p = X_test_paper[bnd_ids]
X_bnd_e = X_test_ext[bnd_ids]
y_bnd   = y_test[bnd_ids]

bnd_csv = ATTACK_DIR / "boundary_search.csv"
if seed_SM.report("boundary_eval") and bnd_csv.exists():
    bnd_df = pd.read_csv(bnd_csv)
else:
    rows = []
    for model_obj, name, X_ev, bnds, msk in [
        (baseline,    "Baseline",     X_bnd_p, paper_bounds, paper_mask),
        (paper_model, "Paper Defense",X_bnd_p, paper_bounds, paper_mask),
        (v9_model,    "V9 Defense",   X_bnd_e, ext_bounds,   ext_mask),
    ]:
        model_obj.eval()
        t_bnd = time.perf_counter()
        X_adv_out, rec_df, bnd_stats = label_only_boundary_search_fixed(
            model_obj, X_ev, y_bnd, CONFIG["attack_epsilon"], bnds, msk, RUN_SEED,
            init_trials=CONFIG["boundary_init_trials"], refine_steps=CONFIG["boundary_refine_steps"],
            X_ref_pool=X_train_paper if name != "V9 Defense" else X_train_ext,
            y_ref_pool=y_train_bal
        )
        atomic_save_df(rec_df, ATTACK_DIR / f"boundary_search_records_{name.lower().replace(' ', '_')}.csv")
        
        rows.append({"method": name, "attack": "Label-Only Boundary Search",
                      "seed": RUN_SEED, **bnd_stats,
                      "note": "label-only boundary-search diagnostic probe strictly within epsilon (NOT an attack benchmark)"})
        print(f"  Boundary {name} ({time.perf_counter()-t_bnd:.1f}s): clean_corr={bnd_stats['n_clean_correct']}  within_eps={bnd_stats['n_within_epsilon']}  init_rate={bnd_stats['initialization_success_rate']:.3f}")
    bnd_df = pd.DataFrame(rows)
    atomic_save_df(bnd_df, bnd_csv)
    seed_SM.mark_done("boundary_eval")

# ── 10. Transfer Attack (Target-Aligned Dedicated Surrogates) ─────────────────
tr_ids = eval_ids[:CONFIG["transfer_n"]]
y_tr   = y_test[tr_ids]

transfer_csv = ATTACK_DIR / "transfer.csv"
if seed_SM.report("transfer_eval") and transfer_csv.exists():
    transfer_df = pd.read_csv(transfer_csv)
else:
    rows = []
    set_seed(RUN_SEED + 50_000)
    
    # Surrogate A (Trained on Paper Feature Space -> attacks Baseline & Paper Defense)
    surrogate_paper = WiderNet(len(paper_idx_local))
    surrogate_paper, _ = train_with_resume(
        surrogate_paper, MODEL_DIR / "surrogate_paper",
        X_train_paper, y_train_bal, X_dev_paper, y_dev,
        CONFIG["surrogate_epochs"], RUN_SEED+50_000, "Surrogate (Paper)", CONFIG
    )
    surrogate_paper.eval()
    X_tr_clean_paper = X_test_paper[tr_ids]
    X_tr_adv_paper   = pgd_attack(surrogate_paper, X_tr_clean_paper, y_tr,
                                   CONFIG["attack_epsilon"], CONFIG["pgd_steps_eval"],
                                   CONFIG["pgd_restarts"], paper_bounds, paper_mask, RUN_SEED+50_000)

    # Surrogate B (Trained on V9 Extension Feature Space -> attacks V9 Defense)
    surrogate_v9 = WiderNet(len(extension_idx_local))
    surrogate_v9, _ = train_with_resume(
        surrogate_v9, MODEL_DIR / "surrogate_v9",
        X_train_ext, y_train_bal, X_dev_ext, y_dev,
        CONFIG["surrogate_epochs"], RUN_SEED+60_000, "Surrogate (V9)", CONFIG
    )
    surrogate_v9.eval()
    X_tr_clean_v9 = X_test_ext[tr_ids]
    X_tr_adv_v9   = pgd_attack(surrogate_v9, X_tr_clean_v9, y_tr,
                                CONFIG["attack_epsilon"], CONFIG["pgd_steps_eval"],
                                CONFIG["pgd_restarts"], ext_bounds, ext_mask, RUN_SEED+60_000)

    for model_obj, name, X_ev_clean, X_ev_adv in [
        (baseline,    "Baseline",     X_tr_clean_paper, X_tr_adv_paper),
        (paper_model, "Paper Defense",X_tr_clean_paper, X_tr_adv_paper),
        (v9_model,    "V9 Defense",   X_tr_clean_v9,    X_tr_adv_v9),
    ]:
        model_obj.eval()
        clean_p   = predict_model(model_obj, X_ev_clean)[0]
        adv_p, adv_prob = predict_model(model_obj, X_ev_adv)
        metrics   = compute_metrics(y_tr, adv_p, adv_prob)
        pert      = attack_perturbation_stats(X_ev_clean, X_ev_adv, y_tr, clean_p, adv_p)
        rows.append({"target": name, "attack": "Transfer PGD", "seed": RUN_SEED,
                      "transfer_asr": pert["attack_success_rate"],
                      **metrics, **pert})
        print(f"  Transfer->{name}: ASR={pert['attack_success_rate']:.4f}  acc={metrics['accuracy']:.4f}")

    transfer_df = pd.DataFrame(rows)
    atomic_save_df(transfer_df, transfer_csv)
    seed_SM.mark_done("transfer_eval")
    del surrogate_paper, surrogate_v9
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── 11. PGD Strength Sweep ────────────────────────────────────────────────────
pgd_sweep_csv = ATTACK_DIR / "pgd_strength_sweep.csv"
if seed_SM.report("pgd_sweep") and pgd_sweep_csv.exists():
    pgd_sweep_df = pd.read_csv(pgd_sweep_csv)
else:
    rows = []
    for n_steps in CONFIG["pgd_sweep_steps"]:
        for model_obj, name, X_ev, bnds, msk in [
            (baseline,    "Baseline",     X_eval_paper, paper_bounds, paper_mask),
            (paper_model, "Paper Defense",X_eval_paper, paper_bounds, paper_mask),
            (v9_model,    "V9 Defense",   X_eval_ext,   ext_bounds,   ext_mask),
        ]:
            model_obj.eval()
            X_adv = pgd_attack(model_obj, X_ev, y_eval, CONFIG["attack_epsilon"],
                               n_steps, 1, bnds, msk, RUN_SEED)
            adv_p = predict_model(model_obj, X_adv)[0]
            metrics = compute_metrics(y_eval, adv_p, None)
            rows.append({"method": name, "pgd_steps": n_steps, "seed": RUN_SEED,
                          "accuracy": metrics["accuracy"]})
    pgd_sweep_df = pd.DataFrame(rows)
    atomic_save_df(pgd_sweep_df, pgd_sweep_csv)
    seed_SM.mark_done("pgd_sweep")

# ── 12. Epsilon Sweep ─────────────────────────────────────────────────────────
eps_csv = ATTACK_DIR / "epsilon_sweep.csv"
if seed_SM.report("eps_sweep") and eps_csv.exists():
    eps_df = pd.read_csv(eps_csv)
else:
    rows = []
    for eps_val in CONFIG["eps_sweep"]:
        for model_obj, name, X_ev, bnds, msk in [
            (baseline,    "Baseline",     X_eval_paper, paper_bounds, paper_mask),
            (paper_model, "Paper Defense",X_eval_paper, paper_bounds, paper_mask),
            (v9_model,    "V9 Defense",   X_eval_ext,   ext_bounds,   ext_mask),
        ]:
            model_obj.eval()
            X_adv = pgd_attack(model_obj, X_ev, y_eval, eps_val,
                               CONFIG["pgd_steps_eval"], 1, bnds, msk, RUN_SEED)
            adv_p = predict_model(model_obj, X_adv)[0]
            metrics = compute_metrics(y_eval, adv_p, None)
            rows.append({"method": name, "epsilon": eps_val, "seed": RUN_SEED,
                          "accuracy": metrics["accuracy"]})
    eps_df = pd.DataFrame(rows)
    atomic_save_df(eps_df, eps_csv)
    seed_SM.mark_done("eps_sweep")

# ── 13. Gradient-Masking Diagnostic ───────────────────────────────────────────
def run_gradient_masking_diagnostic(seed_dir, cfg):
    seed_dir   = Path(seed_dir)
    atk_dir    = seed_dir / "attacks"
    indicators = []

    p = atk_dir / "pgd_strength_sweep.csv"
    if p.exists():
        sw = pd.read_csv(p)
        v9 = sw[sw["method"]=="V9 Defense"].sort_values("pgd_steps")
        if len(v9) >= 2:
            drop = v9["accuracy"].iloc[0] - v9["accuracy"].iloc[-1]
            if drop < 0.01:
                indicators.append(("PGD_sweep", "POSSIBLE MASKING SIGNAL", f"drop={drop:.4f}"))
            else:
                indicators.append(("PGD_sweep", "NO CLEAR EVIDENCE", f"drop={drop:.4f}"))

    p = atk_dir / "epsilon_sweep.csv"
    if p.exists():
        sw = pd.read_csv(p)
        v9 = sw[sw["method"]=="V9 Defense"].sort_values("epsilon")
        if len(v9) >= 2:
            drop = v9["accuracy"].iloc[0] - v9["accuracy"].iloc[-1]
            if drop < 0.02:
                indicators.append(("Epsilon_sweep", "POSSIBLE MASKING SIGNAL", f"drop={drop:.4f}"))
            else:
                indicators.append(("Epsilon_sweep", "NO CLEAR EVIDENCE", f"drop={drop:.4f}"))

    tp = atk_dir / "transfer.csv"; pp = atk_dir / "pgd.csv"
    if tp.exists() and pp.exists():
        tr  = pd.read_csv(tp); pg = pd.read_csv(pp)
        v9t = tr[tr.get("target","") == "V9 Defense"]["transfer_asr"].mean() if "transfer_asr" in tr.columns else None
        v9p = pg[pg["method"] == "V9 Defense"]["attack_success_rate"].mean() if "attack_success_rate" in pg.columns else None
        if v9t is not None and v9p is not None:
            if v9p > 0.05 and v9t < v9p * 0.5:
                indicators.append(("Transfer_vs_whitebox", "EVIDENCE CONSISTENT WITH ROBUSTNESS", f"whitebox={v9p:.3f} >> transfer={v9t:.3f}"))
            else:
                indicators.append(("Transfer_vs_whitebox", "NO CLEAR EVIDENCE", f"whitebox={v9p:.3f} transfer={v9t:.3f}"))

    masking = sum(1 for _, v, _ in indicators if "MASKING" in v)
    if masking >= 3:
        verdict = "POSSIBLE MASKING SIGNAL (multiple indicators)"
    elif masking >= 1:
        verdict = "WEAK MASKING SIGNAL (some indicators)"
    else:
        verdict = "NO CLEAR EVIDENCE OF GRADIENT MASKING"

    print("\nGRADIENT-MASKING DIAGNOSTIC (HEURISTIC EVIDENCE)")
    print("=" * 60)
    for name, v, note in indicators:
        print(f"  [{v}] {name}: {note}")
    print(f"\nDIAGNOSTIC VERDICT: {verdict}")
    print("=" * 60)
    return {"indicators": [(n, v, nt) for n, v, nt in indicators], "verdict": verdict}

diag_result = run_gradient_masking_diagnostic(SEED_DIR, CONFIG)
atomic_save_json(diag_result, SEED_DIR / "gradient_masking_diagnostic.json")

# ── 14. Aggregate & Save Seed Results ─────────────────────────────────────────
seed_rows = []
for df_obj in [fgsm_df, pgd_df, deepfool_df, sq_df, bnd_df, transfer_df]:
    if df_obj is not None and len(df_obj) > 0:
        seed_rows.append(df_obj)

clean_rows = []
for model_obj, name, X_ev, y_ev in [
    (baseline,    "Baseline",     X_test_paper, y_test),
    (paper_model, "Paper Defense",X_test_paper, y_test),
    (v9_model,    "V9 Defense",   X_test_ext,   y_test),
]:
    preds, probs = predict_model(model_obj, X_ev)
    m = compute_metrics(y_ev, preds, probs)
    clean_rows.append({"method": name, "attack": "Clean", "seed": RUN_SEED, **m})
seed_rows.insert(0, pd.DataFrame(clean_rows))

all_seed_df = pd.concat(seed_rows, ignore_index=True)
all_seed_df["seed"] = RUN_SEED
atomic_save_df(all_seed_df, SEED_DIR / "seed_results.csv")
print(f"  [SAVE] seed_results.csv ({len(all_seed_df)} rows)")

budget = {
    "seed":                   RUN_SEED,
    "baseline_epochs":        CONFIG["baseline_epochs"],
    "paper_epochs":           CONFIG["paper_epochs"],
    "v9_clean_epochs":        CONFIG["v9_clean_epochs"],
    "v9_online_epochs":       CONFIG["v9_online_epochs"],
    "batch_size":             CONFIG["batch_size"],
    "pgd_steps_eval":         CONFIG["pgd_steps_eval"],
    "pgd_steps_online":       CONFIG["pgd_steps_online"],
    "online_adv_weight":      CONFIG["online_adv_weight"],
    "device":                 str(DEVICE),
    "total_seconds":          time.perf_counter() - t_seed_start,
    "note":                   "V9 computational overhead includes dynamic per-batch PGD online gradient generation"
}
atomic_save_json(budget, SEED_DIR / "training_budget.json")
atomic_save_json({"seed": RUN_SEED, "status": "complete", "duration_sec": time.perf_counter() - t_seed_start}, SEED_DIR / "seed_complete.json")

t_seed_total = time.perf_counter() - t_seed_start
print(f"\n[DONE] Seed {RUN_SEED} completed in {t_seed_total/3600:.2f}h")

In [ ]:
# Cell 20 — Aggregate Seeds with Strict Completeness Guard
OUTPUT_DIR = Path(CONFIG["output_dir"])
SEEDS      = CONFIG["seeds"]
agg_rows   = []

completed_seeds = []
missing_seeds   = []

for seed in SEEDS:
    result_file = OUTPUT_DIR / "seeds" / str(seed) / "seed_results.csv"
    if result_file.exists():
        df = pd.read_csv(result_file)
        agg_rows.append(df)
        completed_seeds.append(seed)
        print(f"  [LOAD] seed={seed}: {len(df)} rows")
    else:
        missing_seeds.append(seed)

is_complete = (set(completed_seeds) == set(SEEDS))

if not is_complete:
    print("\n" + "!" * 70)
    print(f"  FINAL RESULTS BLOCKED: Missing seeds: {missing_seeds}")
    print(f"  Completed seeds ({len(completed_seeds)}/{len(SEEDS)}): {completed_seeds}")
    print("!" * 70 + "\n")
    if agg_rows:
        partial_results = pd.concat(agg_rows, ignore_index=True)
        atomic_save_df(partial_results, OUTPUT_DIR / "PARTIAL_RESULTS.csv")
        print(f"[SAVE] PARTIAL_RESULTS.csv — {len(partial_results)} rows saved.")
else:
    print("\n" + "=" * 70)
    print(f"  ALL {len(SEEDS)} SEEDS COMPLETE ({completed_seeds}) — GENERATING FINAL RESULTS")
    print("=" * 70 + "\n")
    all_results = pd.concat(agg_rows, ignore_index=True)
    atomic_save_df(all_results, OUTPUT_DIR / "FINAL_RESULTS.csv")
    print(f"[SAVE] FINAL_RESULTS.csv — {len(all_results)} total rows")

    metric_cols = [c for c in all_results.columns
                   if c not in ("seed","method","attack","split","note","target")]
    numeric_cols = all_results[metric_cols].select_dtypes(include=[np.number]).columns.tolist()
    agg = (all_results.groupby(["method","attack"])[numeric_cols]
           .agg(["mean","std"]).round(4))
    agg.columns = ["_".join(c) for c in agg.columns]
    agg = agg.reset_index()
    atomic_save_df(agg, OUTPUT_DIR / "FINAL_RESULTS_MEAN_STD.csv")
    print(f"[SAVE] FINAL_RESULTS_MEAN_STD.csv")

    show_cols = [c for c in agg.columns if any(k in c for k in ("accuracy_mean", "f1_weighted_mean", "attack_success_rate_mean", "within_epsilon_success_rate_mean", "transfer_asr_mean")) or c in ("method","attack")]
    print("\n=== FINAL AGGREGATE RESULTS (mean across all 3 seeds) ===")
    print(agg[show_cols].to_string(index=False))

In [ ]:
# Cell 21 — Final Sanity Audit (Automated Verification of Artifacts)
OUTPUT_DIR = Path(CONFIG["output_dir"])
SEEDS      = CONFIG["seeds"]

checks = []
def chk(name, fn):
    try:
        ok = fn()
        checks.append((name, ok, None))
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
    except Exception as e:
        checks.append((name, False, str(e)))
        print(f"  [FAIL] {name}: {e}")

all_seeds_done = all((OUTPUT_DIR / "seeds" / str(s) / "seed_results.csv").exists() for s in SEEDS)

if all_seeds_done:
    chk("FINAL_RESULTS.csv exists",          lambda: (OUTPUT_DIR/"FINAL_RESULTS.csv").exists())
    chk("FINAL_RESULTS_MEAN_STD.csv exists", lambda: (OUTPUT_DIR/"FINAL_RESULTS_MEAN_STD.csv").exists())
else:
    print("  [INFO] Incomplete seed run — verifying PARTIAL_RESULTS.csv completeness guard.")
    chk("PARTIAL_RESULTS.csv exists",        lambda: (OUTPUT_DIR/"PARTIAL_RESULTS.csv").exists())

chk("data_summary.json exists",          lambda: (OUTPUT_DIR/"data_summary.json").exists())
chk("mi_features.json exists",           lambda: (OUTPUT_DIR/"artifacts"/"mi_features.json").exists())
chk("shap_features.json exists",         lambda: (OUTPUT_DIR/"artifacts"/"shap_features.json").exists())

for seed in SEEDS:
    sd = OUTPUT_DIR / "seeds" / str(seed)
    if (sd / "seed_results.csv").exists():
        chk(f"seed {seed} results exist",   lambda sd=sd: (sd/"seed_results.csv").exists())
        chk(f"seed {seed} config exists",   lambda sd=sd: (sd/"config.json").exists())
        chk(f"seed {seed} baseline.pt",     lambda sd=sd: (sd/"models"/"baseline"/"best.pt").exists())
        chk(f"seed {seed} paper best.pt",   lambda sd=sd: (sd/"models"/"paper_defense"/"best.pt").exists())
        chk(f"seed {seed} v9 best.pt",      lambda sd=sd: (sd/"models"/"v9_defense"/"best.pt").exists())
        chk(f"seed {seed} fgsm.csv",        lambda sd=sd: (sd/"attacks"/"fgsm.csv").exists())
        chk(f"seed {seed} pgd.csv",         lambda sd=sd: (sd/"attacks"/"pgd.csv").exists())
        chk(f"seed {seed} eval_ids.npz",    lambda sd=sd: (sd/"eval_ids.npz").exists())
        chk(f"seed {seed} results not empty", lambda sd=sd: len(pd.read_csv(sd/"seed_results.csv")) > 0)

for seed in SEEDS:
    sd = OUTPUT_DIR / "seeds" / str(seed)
    for mname in ["baseline", "paper_defense", "v9_defense"]:
        pt = sd / "models" / mname / "best.pt"
        if pt.exists():
            chk(f"seed {seed} {mname} checkpoint valid",
                lambda pt=pt: validate_checkpoint(pt, ["model_state_dict","epoch","seed"]))

failed = [(n, e) for n, ok, e in checks if not ok]
print(f"\n{'='*60}")
print(f"  Checks: {len(checks)}  Passed: {len(checks)-len(failed)}  Failed: {len(failed)}")
print("="*60)

if failed:
    print("\n  FAILURES:")
    for n, e in failed:
        print(f"    [FAIL] {n}" + (f": {e}" if e else ""))
    raise RuntimeError(f"SANITY AUDIT FAILED: {len(failed)} checks failed.")
else:
    print("\n  ALL SANITY CHECKS PASSED")

In [ ]:
# Cell 22 — Final Export and Manifest Generation
OUTPUT_DIR = Path(CONFIG["output_dir"])
SEEDS      = CONFIG["seeds"]

# Training budget compilation
budget_rows = []
for seed in SEEDS:
    bp = OUTPUT_DIR / "seeds" / str(seed) / "training_budget.json"
    if bp.exists():
        with open(bp) as f:
            b = json.load(f)
        budget_rows.append(b)
if budget_rows:
    atomic_save_df(pd.DataFrame(budget_rows), OUTPUT_DIR / "training_budget.csv")
    print("[SAVE] training_budget.csv")

# Ensure data_summary.json at root
dsrc = OUTPUT_DIR / "artifacts" / "data_summary.json"
if dsrc.exists():
    shutil.copy(dsrc, OUTPUT_DIR / "data_summary.json")
    print("[SAVE] data_summary.json")

# config.json at root
atomic_save_json(CONFIG, OUTPUT_DIR / "config.json")
print("[SAVE] config.json")

# environment.json at root
env = {
    "python":  sys.version,
    "torch":   torch.__version__,
    "device":  str(DEVICE) if "DEVICE" in dir() else "unknown",
    "gpu":     torch.cuda.get_device_name(0) if DEVICE.type=="cuda" else "cpu",
    "os":      platform.platform(),
    "created": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
atomic_save_json(env, OUTPUT_DIR / "environment.json")
print("[SAVE] environment.json")

# Experiment manifest
all_seeds_done = all((OUTPUT_DIR / "seeds" / str(s) / "seed_results.csv").exists() for s in SEEDS)
manifest = {
    "experiment_name":  CONFIG["experiment_name"],
    "created_at":       time.strftime("%Y-%m-%dT%H:%M:%S"),
    "seeds":            CONFIG["seeds"],
    "status":           "COMPLETE" if all_seeds_done else "PARTIAL",
    "output_dir":       str(OUTPUT_DIR.resolve()),
    "files": {
        "final_results":          str(OUTPUT_DIR / "FINAL_RESULTS.csv") if all_seeds_done else str(OUTPUT_DIR / "PARTIAL_RESULTS.csv"),
        "final_results_mean_std": str(OUTPUT_DIR / "FINAL_RESULTS_MEAN_STD.csv") if all_seeds_done else None,
        "training_budget":        str(OUTPUT_DIR / "training_budget.csv"),
        "data_summary":           str(OUTPUT_DIR / "data_summary.json"),
        "config":                 str(OUTPUT_DIR / "config.json"),
        "environment":            str(OUTPUT_DIR / "environment.json"),
    },
    "per_seed": {str(s): str(OUTPUT_DIR / "seeds" / str(s)) for s in SEEDS}
}
atomic_save_json(manifest, OUTPUT_DIR / "experiment_manifest.json")
print("[SAVE] experiment_manifest.json")

In [ ]:
# Cell 23 — Extended Unseen Attack Evaluation Suite (MIM, BIM, APGD-CE, CW-Linf)
def mim_attack(model, X, y, eps=0.05, steps=40, decay=1.0, bounds=None, mask=None):
    """Momentum Iterative FGSM (MIM / MI-FGSM) Attack."""
    model.eval()
    x0 = _to_gpu_float(X)
    yt = torch.as_tensor(y, dtype=torch.long, device=DEVICE)
    mt = _to_gpu_float(mask) if mask is not None else torch.ones_like(x0)
    lo = _to_gpu_float(bounds[0]) if bounds is not None else None
    hi = _to_gpu_float(bounds[1]) if bounds is not None else None
    alpha = eps / 10.0
    
    x = x0.clone()
    g = torch.zeros_like(x0)
    
    for _ in range(steps):
        x.requires_grad_(True)
        loss = F.cross_entropy(model(x), yt)
        grad = torch.autograd.grad(loss, x)[0]
        grad_norm = torch.mean(torch.abs(grad), dim=1, keepdim=True) + 1e-12
        g = decay * g + grad / grad_norm
        x = x.detach() + alpha * g.sign() * mt
        x = torch.clamp(x, x0 - eps, x0 + eps)
        if lo is not None and hi is not None:
            x = torch.clamp(x, lo, hi)
        x = (x0 + (x - x0) * mt).detach()
        
    result = x.cpu().numpy().astype(np.float32)
    assert np.max(np.abs(result - X.astype(np.float32))) <= eps + 1e-5
    return result

def bim_attack(model, X, y, eps=0.05, steps=40, bounds=None, mask=None):
    """Basic Iterative Method (BIM / I-FGSM) Attack."""
    model.eval()
    x0 = _to_gpu_float(X)
    yt = torch.as_tensor(y, dtype=torch.long, device=DEVICE)
    mt = _to_gpu_float(mask) if mask is not None else torch.ones_like(x0)
    lo = _to_gpu_float(bounds[0]) if bounds is not None else None
    hi = _to_gpu_float(bounds[1]) if bounds is not None else None
    alpha = eps / 10.0
    
    x = x0.clone()
    for _ in range(steps):
        x.requires_grad_(True)
        loss = F.cross_entropy(model(x), yt)
        grad = torch.autograd.grad(loss, x)[0]
        x = x.detach() + alpha * grad.sign() * mt
        x = torch.clamp(x, x0 - eps, x0 + eps)
        if lo is not None and hi is not None:
            x = torch.clamp(x, lo, hi)
        x = (x0 + (x - x0) * mt).detach()
        
    result = x.cpu().numpy().astype(np.float32)
    assert np.max(np.abs(result - X.astype(np.float32))) <= eps + 1e-5
    return result

def apgd_ce_attack(model, X, y, eps=0.05, steps=50, bounds=None, mask=None, seed=42):
    """Auto-PGD (APGD-CE) with Adaptive Step Size & Momentum (Croce & Hein)."""
    model.eval()
    x0 = _to_gpu_float(X)
    yt = torch.as_tensor(y, dtype=torch.long, device=DEVICE)
    mt = _to_gpu_float(mask) if mask is not None else torch.ones_like(x0)
    lo = _to_gpu_float(bounds[0]) if bounds is not None else None
    hi = _to_gpu_float(bounds[1]) if bounds is not None else None
    
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    x = torch.clamp(x0 + torch.empty_like(x0).uniform_(-eps, eps, generator=gen) * mt, x0 - eps, x0 + eps)
    if lo is not None and hi is not None:
        x = torch.clamp(x, lo, hi)
    x = (x0 + (x - x0) * mt).detach()
    
    x_best = x.clone()
    with torch.no_grad():
        best_loss = F.cross_entropy(model(x), yt, reduction="none")
    
    eta = 2.0 * eps
    x_prev = x0.clone()
    rho = 0.75
    w = max(5, int(steps * 0.22))
    
    for t in range(steps):
        x.requires_grad_(True)
        losses = F.cross_entropy(model(x), yt, reduction="none")
        grad = torch.autograd.grad(losses.sum(), x)[0]
        
        z = x.detach() + eta * grad.sign() * mt
        z = torch.clamp(z, x0 - eps, x0 + eps)
        if lo is not None and hi is not None:
            z = torch.clamp(z, lo, hi)
        z = (x0 + (z - x0) * mt).detach()
        
        if t > 0:
            x_next = x.detach() + rho * (z - x.detach()) + (1 - rho) * (x.detach() - x_prev)
            x_next = torch.clamp(x_next, x0 - eps, x0 + eps)
            if lo is not None and hi is not None:
                x_next = torch.clamp(x_next, lo, hi)
            x_next = (x0 + (x_next - x0) * mt).detach()
        else:
            x_next = z
            
        x_prev = x.clone()
        x = x_next
        
        with torch.no_grad():
            curr_loss = F.cross_entropy(model(x), yt, reduction="none")
        better = curr_loss > best_loss
        x_best[better] = x[better]
        best_loss[better] = curr_loss[better]
        
        if (t + 1) % w == 0:
            eta *= 0.5
            
    result = x_best.cpu().numpy().astype(np.float32)
    assert np.max(np.abs(result - X.astype(np.float32))) <= eps + 1e-5
    return result

def cw_linf_attack(model, X, y, eps=0.05, steps=50, bounds=None, mask=None, seed=42):
    """Carlini-Wagner L-infinity Margin Loss Attack."""
    model.eval()
    x0 = _to_gpu_float(X)
    yt = torch.as_tensor(y, dtype=torch.long, device=DEVICE)
    mt = _to_gpu_float(mask) if mask is not None else torch.ones_like(x0)
    lo = _to_gpu_float(bounds[0]) if bounds is not None else None
    hi = _to_gpu_float(bounds[1]) if bounds is not None else None
    alpha = eps / 10.0
    
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    x = torch.clamp(x0 + torch.empty_like(x0).uniform_(-eps, eps, generator=gen) * mt, x0 - eps, x0 + eps)
    if lo is not None and hi is not None:
        x = torch.clamp(x, lo, hi)
    x = (x0 + (x - x0) * mt).detach()
    
    for _ in range(steps):
        x.requires_grad_(True)
        logits = model(x)
        y_onehot = F.one_hot(yt, num_classes=2).float()
        target_logit = (logits * y_onehot).sum(dim=1)
        other_logit = (logits * (1.0 - y_onehot) - y_onehot * 1e4).max(dim=1)[0]
        cw_loss = (other_logit - target_logit).sum()
        grad = torch.autograd.grad(cw_loss, x)[0]
        
        x = x.detach() + alpha * grad.sign() * mt
        x = torch.clamp(x, x0 - eps, x0 + eps)
        if lo is not None and hi is not None:
            x = torch.clamp(x, lo, hi)
        x = (x0 + (x - x0) * mt).detach()
        
    result = x.cpu().numpy().astype(np.float32)
    assert np.max(np.abs(result - X.astype(np.float32))) <= eps + 1e-5
    return result

print("=" * 70)
print("  EXTENDED UNSEEN ATTACK EVALUATION (MIM, BIM, APGD-CE, CW-Linf)")
print("=" * 70)

unseen_eval_rows = []
OUTPUT_DIR = Path(CONFIG["output_dir"])
SEEDS = CONFIG["seeds"]

for s in SEEDS:
    seed_dir = OUTPUT_DIR / "seeds" / str(s)
    eval_ids_p = seed_dir / "eval_ids.npz"
    if not eval_ids_p.exists():
        print(f"[WARN] eval_ids not found for seed {s}, skipping.")
        continue
    eval_ids_s = np.load(eval_ids_p)["eval_ids"]
    
    X_ev_paper_s = X_test_paper[eval_ids_s]
    X_ev_ext_s   = X_test_ext[eval_ids_s]
    y_ev_s       = y_test[eval_ids_s]
    
    # Load trained models
    bl_m = IDSNet(len(paper_idx_local)).to(DEVICE)
    pp_m = IDSNet(len(paper_idx_local)).to(DEVICE)
    v9_m = IDSNet(len(extension_idx_local)).to(DEVICE)
    
    bl_m.load_state_dict(torch.load(seed_dir / "models" / "baseline" / "best.pt", map_location=DEVICE, weights_only=False)["model_state_dict"])
    pp_m.load_state_dict(torch.load(seed_dir / "models" / "paper_defense" / "best.pt", map_location=DEVICE, weights_only=False)["model_state_dict"])
    v9_m.load_state_dict(torch.load(seed_dir / "models" / "v9_defense" / "best.pt", map_location=DEVICE, weights_only=False)["model_state_dict"])
    
    bl_clean_p = predict_model(bl_m, X_ev_paper_s)[0]
    pp_clean_p = predict_model(pp_m, X_ev_paper_s)[0]
    v9_clean_p = predict_model(v9_m, X_ev_ext_s)[0]
    
    attack_funcs = [
        ("MIM (MI-FGSM)", lambda m, x, y, b, k: mim_attack(m, x, y, CONFIG["attack_epsilon"], 40, 1.0, b, k)),
        ("BIM (I-FGSM)",  lambda m, x, y, b, k: bim_attack(m, x, y, CONFIG["attack_epsilon"], 40, b, k)),
        ("Auto-PGD (APGD-CE)", lambda m, x, y, b, k: apgd_ce_attack(m, x, y, CONFIG["attack_epsilon"], 50, b, k, seed=s)),
        ("CW-Linf Margin Attack", lambda m, x, y, b, k: cw_linf_attack(m, x, y, CONFIG["attack_epsilon"], 50, b, k, seed=s)),
    ]
    
    seed_unseen_rows = []
    for atk_name, atk_fn in attack_funcs:
        for model_obj, name, X_ev, bnds, msk, clean_p in [
            (bl_m, "Baseline",      X_ev_paper_s, paper_bounds, paper_mask, bl_clean_p),
            (pp_m, "Paper Defense", X_ev_paper_s, paper_bounds, paper_mask, pp_clean_p),
            (v9_m, "V9 Defense",    X_ev_ext_s,   ext_bounds,   ext_mask,   v9_clean_p),
        ]:
            t_a = time.perf_counter()
            X_adv = atk_fn(model_obj, X_ev, y_ev_s, bnds, msk)
            adv_p, adv_prob = predict_model(model_obj, X_adv)
            metrics = compute_metrics(y_ev_s, adv_p, adv_prob)
            pert    = attack_perturbation_stats(X_ev, X_adv, y_ev_s, clean_p, adv_p)
            row = {"method": name, "attack": atk_name, "seed": s, **metrics, **pert}
            seed_unseen_rows.append(row)
            unseen_eval_rows.append(row)
            print(f"  [Seed {s}] {atk_name:22s} -> {name:15s} ({time.perf_counter()-t_a:.1f}s): Acc={metrics['accuracy']:.4f}  ASR={pert['attack_success_rate']:.4f}")
            
    # Save seed-specific unseen attack CSV
    atomic_save_df(pd.DataFrame(seed_unseen_rows), seed_dir / "attacks" / "unseen_attacks.csv")

# Save Aggregated Unseen Tables
df_unseen_all = pd.DataFrame(unseen_eval_rows)
atomic_save_df(df_unseen_all, OUTPUT_DIR / "UNSEEN_ATTACKS_RESULTS.csv")

metric_cols = [c for c in df_unseen_all.columns if c not in ("seed","method","attack","split","note","target")]
numeric_cols = df_unseen_all[metric_cols].select_dtypes(include=[np.number]).columns.tolist()
agg_unseen = (df_unseen_all.groupby(["method","attack"])[numeric_cols].agg(["mean","std"]).round(4))
agg_unseen.columns = ["_".join(c) for c in agg_unseen.columns]
agg_unseen = agg_unseen.reset_index()
atomic_save_df(agg_unseen, OUTPUT_DIR / "UNSEEN_ATTACKS_MEAN_STD.csv")

# Combine with Original Final Results into Unified Master Table
final_csv = OUTPUT_DIR / "FINAL_RESULTS.csv"
if final_csv.exists():
    df_orig = pd.read_csv(final_csv)
    df_combined = pd.concat([df_orig, df_unseen_all], ignore_index=True)
    atomic_save_df(df_combined, OUTPUT_DIR / "ALL_ATTACKS_COMBINED_RESULTS.csv")
    
    num_cols_comb = df_combined[metric_cols].select_dtypes(include=[np.number]).columns.tolist()
    agg_comb = (df_combined.groupby(["method","attack"])[num_cols_comb].agg(["mean","std"]).round(4))
    agg_comb.columns = ["_".join(c) for c in agg_comb.columns]
    agg_comb = agg_comb.reset_index()
    atomic_save_df(agg_comb, OUTPUT_DIR / "ALL_ATTACKS_COMBINED_MEAN_STD.csv")
    print(f"\n[SAVE] ALL_ATTACKS_COMBINED_MEAN_STD.csv saved ({len(agg_comb)} method-attack pairs).")

show_cols = [c for c in agg_unseen.columns if any(k in c for k in ("accuracy_mean", "f1_weighted_mean", "attack_success_rate_mean")) or c in ("method","attack")]
print("\n" + "=" * 70)
print("=== EXTENDED UNSEEN ATTACKS (MEAN ACROSS ALL 3 SEEDS) ===")
print("=" * 70)
print(agg_unseen[show_cols].to_string(index=False))

In [ ]:
# Cell 24 — DeepFool Perturbation Budget Analysis & deepfoolbudget.csv Export
def deepfool_gpu_vectorized(model, X, max_iters=50, overshoot=0.02):
    """Pure-GPU vectorized DeepFool unconstrained minimum-perturbation generator."""
    model.eval()
    x_orig = torch.as_tensor(np.ascontiguousarray(X), dtype=torch.float32, device=DEVICE)
    x = x_orig.clone()
    n, d = X.shape
    with torch.no_grad():
        orig_logits = model(x)
        orig_preds = orig_logits.argmax(dim=1)
    active_mask = torch.ones(n, dtype=torch.bool, device=DEVICE)
    for it in range(max_iters):
        if not active_mask.any():
            break
        x_act = x[active_mask].clone().requires_grad_(True)
        logits = model(x_act)
        orig_act = orig_preds[active_mask]
        y_onehot = F.one_hot(orig_act, num_classes=2).float()
        f_orig = (logits * y_onehot).sum(dim=1)
        f_other = (logits * (1.0 - y_onehot)).sum(dim=1)
        diff = f_other - f_orig
        grad = torch.autograd.grad(diff.sum(), x_act)[0]
        grad_norm_sq = (grad ** 2).sum(dim=1, keepdim=True) + 1e-12
        step = (torch.abs(diff.unsqueeze(1)) / grad_norm_sq) * grad * (1.0 + overshoot)
        with torch.no_grad():
            x_act_new = x_act + step
            indices = torch.where(active_mask)[0]
            x[indices] = x_act_new
            new_logits = model(x_act_new)
            new_preds = new_logits.argmax(dim=1)
            still_active = (new_preds == orig_act)
            active_mask[indices] = still_active
    return x.detach().cpu().numpy().astype(np.float32)

print("=" * 70)
print("  DEEPFOOL PERTURBATION BUDGET ANALYSIS (HOW MUCH DISTORTION WAS NEEDED?)")
print("=" * 70)

all_df_budget_rows = []
OUTPUT_DIR = Path(CONFIG["output_dir"])
SEEDS = CONFIG["seeds"]

for s in SEEDS:
    seed_dir = OUTPUT_DIR / "seeds" / str(s)
    eval_ids_p = seed_dir / "eval_ids.npz"
    if not eval_ids_p.exists():
        continue
    eval_ids_df = np.load(eval_ids_p)["eval_ids"][:CONFIG["deepfool_eval_n"]]
    X_df_p = X_test_paper[eval_ids_df]
    X_df_e = X_test_ext[eval_ids_df]
    y_df   = y_test[eval_ids_df]
    
    bl_m = IDSNet(len(paper_idx_local)).to(DEVICE)
    pp_m = IDSNet(len(paper_idx_local)).to(DEVICE)
    v9_m = IDSNet(len(extension_idx_local)).to(DEVICE)
    
    bl_m.load_state_dict(torch.load(seed_dir / "models" / "baseline" / "best.pt", map_location=DEVICE, weights_only=False)["model_state_dict"])
    pp_m.load_state_dict(torch.load(seed_dir / "models" / "paper_defense" / "best.pt", map_location=DEVICE, weights_only=False)["model_state_dict"])
    v9_m.load_state_dict(torch.load(seed_dir / "models" / "v9_defense" / "best.pt", map_location=DEVICE, weights_only=False)["model_state_dict"])
    
    for model_obj, name, X_ev in [
        (bl_m, "Baseline", X_df_p),
        (pp_m, "Paper Defense", X_df_p),
        (v9_m, "V9 Defense", X_df_e),
    ]:
        clean_p = predict_model(model_obj, X_ev)[0]
        X_adv = deepfool_gpu_vectorized(model_obj, X_ev, max_iters=CONFIG["deepfool_steps"])
        adv_p = predict_model(model_obj, X_adv)[0]
        
        delta = np.abs(X_adv - X_ev)
        linf_all = delta.max(axis=1)
        l2_all = np.linalg.norm(delta, axis=1)
        
        clean_correct = (clean_p == y_df)
        successful_flips = clean_correct & (adv_p != y_df)
        
        n_attempted = len(X_ev)
        n_clean_correct = int(clean_correct.sum())
        n_flipped = int(successful_flips.sum())
        asr = float(n_flipped / max(1, n_clean_correct))
        
        linf_flips = linf_all[successful_flips]
        l2_flips   = l2_all[successful_flips]
        
        pct_within_eps_005 = float((linf_flips <= 0.05).mean()) if len(linf_flips) > 0 else 0.0
        pct_exceed_eps_005 = float((linf_flips > 0.05).mean()) if len(linf_flips) > 0 else 0.0
        pct_exceed_eps_010 = float((linf_flips > 0.10).mean()) if len(linf_flips) > 0 else 0.0
        pct_exceed_eps_020 = float((linf_flips > 0.20).mean()) if len(linf_flips) > 0 else 0.0
        pct_exceed_eps_050 = float((linf_flips > 0.50).mean()) if len(linf_flips) > 0 else 0.0
        
        row_b = {
            "seed": s, "method": name, "n_samples": n_attempted,
            "n_clean_correct": n_clean_correct, "n_successful_flips": n_flipped,
            "deepfool_asr": asr,
            "mean_Linf_flips": float(linf_flips.mean()) if len(linf_flips) > 0 else 0.0,
            "median_Linf_flips": float(np.median(linf_flips)) if len(linf_flips) > 0 else 0.0,
            "p10_Linf_flips": float(np.percentile(linf_flips, 10)) if len(linf_flips) > 0 else 0.0,
            "p75_Linf_flips": float(np.percentile(linf_flips, 75)) if len(linf_flips) > 0 else 0.0,
            "p90_Linf_flips": float(np.percentile(linf_flips, 90)) if len(linf_flips) > 0 else 0.0,
            "max_Linf_flips": float(linf_flips.max()) if len(linf_flips) > 0 else 0.0,
            "mean_L2_flips": float(l2_flips.mean()) if len(l2_flips) > 0 else 0.0,
            "median_L2_flips": float(np.median(l2_flips)) if len(l2_flips) > 0 else 0.0,
            "pct_within_eps_0.05": pct_within_eps_005,
            "pct_exceeding_eps_0.05": pct_exceed_eps_005,
            "pct_exceeding_eps_0.10": pct_exceed_eps_010,
            "pct_exceeding_eps_0.20": pct_exceed_eps_020,
            "pct_exceeding_eps_0.50": pct_exceed_eps_050,
        }
        all_df_budget_rows.append(row_b)
        print(f"  [Seed {s:4d}] {name:15s} | ASR: {asr*100:5.1f}% | Mean Linf: {row_b['mean_Linf_flips']:.4f} | Median Linf: {row_b['median_Linf_flips']:.4f} | Max Linf: {row_b['max_Linf_flips']:.4f} | % > eps(0.05): {pct_exceed_eps_005*100:5.1f}%")

df_df_budget = pd.DataFrame(all_df_budget_rows)
metric_cols_num = [c for c in df_df_budget.columns if c not in ("seed", "method")]
agg_budget = df_df_budget.groupby("method")[metric_cols_num].agg(["mean", "std"]).round(4)
agg_budget.columns = ["_".join(c) for c in agg_budget.columns]
agg_budget = agg_budget.reset_index()

df_df_budget["record_type"] = "seed_run"
agg_budget["record_type"] = "aggregate_mean_std"
agg_budget["seed"] = "All (42, 123, 2024)"

full_deepfool_budget = pd.concat([agg_budget, df_df_budget], ignore_index=True)
atomic_save_df(full_deepfool_budget, OUTPUT_DIR / "deepfoolbudget.csv")
atomic_save_df(full_deepfool_budget, Path("deepfoolbudget.csv"))
print(f"\n[SAVE] deepfoolbudget.csv saved ({len(full_deepfool_budget)} rows).")

cols_show = ["method", "deepfool_asr_mean", "mean_Linf_flips_mean", "median_Linf_flips_mean", "max_Linf_flips_mean", "mean_L2_flips_mean", "pct_exceeding_eps_0.05_mean", "pct_within_eps_0.05_mean"]
print("\n" + "=" * 70)
print("=== DEEPFOOL PERTURBATION BUDGET SUMMARY (MEAN ACROSS 3 SEEDS) ===")
print("=" * 70)
print(agg_budget[[c for c in cols_show if c in agg_budget.columns]].to_string(index=False))

In [ ]:
# ============================================================
# FINAL CELL — IEEE-READY ANALYTICS SUITE
# Naming: Baseline Defense / Reference Defense / Proposed Defense
# Palette: Classy Steel Blue / Burnt Orange / Forest Green
# Figures: combined panels, big fonts (IEEE 2-column compatible)
# Output:  outputs/v9_final/analytics_ieee/
# ============================================================
%matplotlib inline
import IPython.display as ipd
from IPython.display import Image as IpyImage


import json
from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib
# matplotlib backend set by Jupyter (%matplotlib inline)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (confusion_matrix, roc_curve, auc,
                              precision_recall_curve, average_precision_score)
import pyarrow.parquet as pq

# -- Directories --------------------------------------------------------------
BASE_DIR     = Path("outputs/v9_final")
ANALYTICS_DIR = BASE_DIR / "analytics_ieee"
ART_DIR      = BASE_DIR / "artifacts"
SEEDS_DIR    = BASE_DIR / "seeds"
SUBDIRS = {
    "dataset":           ANALYTICS_DIR / "dataset",
    "features":          ANALYTICS_DIR / "features",
    "performance":       ANALYTICS_DIR / "performance",
    "robustness":        ANALYTICS_DIR / "robustness",
    "confusion_matrices":ANALYTICS_DIR / "confusion_matrices",
    "diagnostics":       ANALYTICS_DIR / "diagnostics",
    "training":          ANALYTICS_DIR / "training",
    "tables":            ANALYTICS_DIR / "tables",
}
for d in SUBDIRS.values():
    d.mkdir(parents=True, exist_ok=True)

# -- Model display names -------------------------------------------------------
LABEL = {
    "Baseline":      "Baseline Defense",
    "Paper Defense": "Reference Defense",
    "V9 Defense":    "Proposed Defense",
}
# Short label for x-ticks
SHORT = {
    "Baseline":      "Baseline\nDefense",
    "Paper Defense": "Reference\nDefense",
    "V9 Defense":    "Proposed\nDefense",
}
# CSV key -> display name (for seed_results.csv)
METHODS_2 = ["Baseline", "V9 Defense"]          # exclude Paper from comparisons
METHODS_3 = ["Baseline", "Paper Defense", "V9 Defense"]

# -- Classy, simple palette ----------------------------------------------------
# Three principal colors + neutral accents — suitable for IEEE B&W printing too
C_BASE  = "#4472C4"   # Classic IEEE steel blue  (Baseline Defense)
C_PAPER = "#ED7D31"   # Soft burnt orange        (Reference Defense)
C_V9    = "#375623"   # Deep forest green        (Proposed Defense)

C_PAIR  = [C_BASE, C_V9]        # 2-model palette
C_THREE = [C_BASE, C_PAPER, C_V9]  # 3-model palette

C_BAR_BLUE   = "#BDD7EE"   # Light steel blue fill
C_BAR_GREEN  = "#C6EFCE"   # Light mint fill
C_BAR_ORANGE = "#FCE4D6"   # Light peach fill
C_EDGE = "#1F3864"          # Dark navy edge

# -- Global IEEE font settings -------------------------------------------------
FS = dict(
    title  = 15,
    label  = 14,
    tick   = 12,
    legend = 12,
    annot  = 11,
)
plt.rcParams.update({
    'font.family':          'DejaVu Sans',
    'font.size':            FS['tick'],
    'axes.titlesize':       FS['title'],
    'axes.labelsize':       FS['label'],
    'xtick.labelsize':      FS['tick'],
    'ytick.labelsize':      FS['tick'],
    'legend.fontsize':      FS['legend'],
    'figure.dpi':           150,
    'savefig.dpi':          300,
    'axes.grid':            True,
    'grid.alpha':           0.25,
    'grid.linestyle':       '--',
    'axes.spines.top':      False,
    'axes.spines.right':    False,
    # --- WHITE BACKGROUND (explicit) ---
    'figure.facecolor':     'white',
    'axes.facecolor':       'white',
    'savefig.facecolor':    'white',
    'savefig.edgecolor':    'white',
    'text.color':           '#1F1F1F',
    'axes.labelcolor':      '#1F1F1F',
    'xtick.color':          '#1F1F1F',
    'ytick.color':          '#1F1F1F',
})

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -- Model Architecture --------------------------------------------------------
class IDSNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim,128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.30),
            nn.Linear(128, 64),       nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.30),
            nn.Linear(64,  32),       nn.BatchNorm1d(32),  nn.ReLU(), nn.Dropout(0.30),
            nn.Linear(32, 2)
        )
    def forward(self, x): return self.network(x)

# -- Save helper ---------------------------------------------------------------
def save_fig(fig, name, subfolder):
    for p in [ANALYTICS_DIR / name, SUBDIRS[subfolder] / name]:
        fig.savefig(p, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"  [SAVED] {name}")

def bar_label(ax, bars, fmt="{:.2f}%", scale=100, dy_frac=0.02):
    ylim = ax.get_ylim()
    dy = (ylim[1] - ylim[0]) * dy_frac
    for b in bars:
        h = b.get_height()
        ax.text(b.get_x()+b.get_width()/2, h+dy,
                fmt.format(h*scale), ha='center', va='bottom',
                fontsize=FS['annot'], fontweight='bold')

# =============================================================================
def main():
    print("="*60)
    print("IEEE 2-COLUMN ANALYTICS FIGURE GENERATION")
    print(f"->  {ANALYTICS_DIR.resolve()}")
    print("="*60)

    # -- Load CSVs -------------------------------------------------------------
    print("\n[A] Loading saved CSV result files...")
    df_comb     = pd.read_csv(BASE_DIR / "ALL_ATTACKS_COMBINED_MEAN_STD.csv")
    df_eps      = pd.read_csv(BASE_DIR / "epsilon_sweep_summary.csv")
    df_pgd_sw   = pd.read_csv(BASE_DIR / "pgd_step_sweep_summary.csv")
    df_deepfool = pd.read_csv(BASE_DIR / "deepfoolbudget.csv")
    df_budget   = pd.read_csv(BASE_DIR / "training_budget.csv")
    df_shap     = pd.read_csv(ART_DIR  / "shap_importance.csv")
    with open(ART_DIR / "shap_features.json") as f:
        sf = json.load(f)
    paper_idx, ext_idx = sf["paper_idx"], sf["extension_idx"]
    seed_dfs = {s: pd.read_csv(SEEDS_DIR/str(s)/"seed_results.csv") for s in [42,123,2024]}

    # Map method names in CSVs -> display labels
    df_comb["method_label"]  = df_comb["method"].map(LABEL)
    for s in [42,123,2024]:
        seed_dfs[s]["method_label"] = seed_dfs[s]["method"].map(LABEL)

    # -- Load PyTorch models + eval tensors ------------------------------------
    print("[B] Loading models & eval tensors...")
    def load_model(path, dim):
        m = IDSNet(dim).to(DEVICE)
        m.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=False)["model_state_dict"])
        m.eval(); return m

    model_base  = load_model(SEEDS_DIR/"42"/"models"/"baseline"/"best.pt",     15)
    model_paper = load_model(SEEDS_DIR/"42"/"models"/"paper_defense"/"best.pt", 15)
    model_v9    = load_model(SEEDS_DIR/"42"/"models"/"v9_defense"/"best.pt",    27)

    eval_idx = np.load(SEEDS_DIR/"42"/"eval_ids.npz")["eval_ids"]
    sc       = np.load(ART_DIR/"scaled.npz")
    X_eval   = sc["X_test_all"][eval_idx]
    tt       = np.load(ART_DIR/"train_test_split.npz")
    tbl      = pq.read_table(ART_DIR/"data_clean.parquet", columns=["Label"]).to_pandas()
    y_eval   = (tbl["Label"].iloc[tt["test_idx"]].values[eval_idx] != "Benign").astype(int)

    Xb  = torch.as_tensor(X_eval[:,paper_idx], dtype=torch.float32, device=DEVICE)
    Xv9 = torch.as_tensor(X_eval[:,ext_idx],   dtype=torch.float32, device=DEVICE)
    Yt  = torch.as_tensor(y_eval, dtype=torch.long, device=DEVICE)

    # Clean predictions
    with torch.no_grad():
        pb   = torch.softmax(model_base( Xb),  dim=1)[:,1].cpu().numpy()
        pp   = torch.softmax(model_paper(Xb),  dim=1)[:,1].cpu().numpy()
        pv9  = torch.softmax(model_v9(   Xv9), dim=1)[:,1].cpu().numpy()
    yhat_b  = (pb  >= 0.5).astype(int)
    yhat_p  = (pp  >= 0.5).astype(int)
    yhat_v9 = (pv9 >= 0.5).astype(int)

    # PGD-20 adversarial predictions
    eps   = 0.05
    alpha = eps * 0.10
    cl_v9_min = torch.as_tensor(sc["X_train_all"][:,ext_idx].min(0), dtype=torch.float32, device=DEVICE)
    cl_v9_max = torch.as_tensor(sc["X_train_all"][:,ext_idx].max(0), dtype=torch.float32, device=DEVICE)
    cl_b_min  = torch.as_tensor(sc["X_train_all"][:,paper_idx].min(0), dtype=torch.float32, device=DEVICE)
    cl_b_max  = torch.as_tensor(sc["X_train_all"][:,paper_idx].max(0), dtype=torch.float32, device=DEVICE)

    def pgd20(model, X0, Yref, clip_lo, clip_hi):
        xadv = X0.clone().detach()
        for _ in range(20):
            xadv.requires_grad_(True)
            loss = F.cross_entropy(model(xadv), Yref)
            g    = torch.autograd.grad(loss, xadv)[0]
            xadv = (xadv.detach() + alpha*g.sign())
            xadv = torch.clamp(xadv, X0-eps, X0+eps)
            xadv = torch.clamp(xadv, clip_lo, clip_hi).detach()
        with torch.no_grad():
            return torch.softmax(model(xadv), dim=1)[:,1].cpu().numpy()

    pv9_pgd = pgd20(model_v9,   Xv9, Yt, cl_v9_min, cl_v9_max)
    pb_pgd  = pgd20(model_base, Xb,  Yt, cl_b_min,  cl_b_max)

    xadv_fgsm = Xv9.clone().detach().requires_grad_(True)
    lf = F.cross_entropy(model_v9(xadv_fgsm), Yt)
    gf = torch.autograd.grad(lf, xadv_fgsm)[0]
    xadv_fgsm = torch.clamp(xadv_fgsm.detach()+eps*gf.sign(), cl_v9_min, cl_v9_max)
    with torch.no_grad():
        pv9_fgsm = torch.softmax(model_v9(xadv_fgsm), dim=1)[:,1].cpu().numpy()

    yhat_v9_pgd  = (pv9_pgd  >= 0.5).astype(int)
    yhat_v9_fgsm = (pv9_fgsm >= 0.5).astype(int)

    # =========================================================================
    # FIGURE 01: Dataset Volume Pipeline  (single panel, wide)
    # =========================================================================
    print("\n[01] Dataset pipeline...")
    fig, ax = plt.subplots(figsize=(9, 4.2))
    stages = ["Raw\nIngestion","Cleaned\nFlows","Train\nPool","Core-\nTrain","Dev\nSplit","Test\nSet","SMOTE\nBalanced"]
    vals   = [8.28, 6.32, 5.06, 4.04, 0.51, 1.26, 7.19]
    clrs   = ["#D9E1F2","#BDD7EE","#9DC3E6","#2E75B6","#1F4E79","#ED7D31","#375623"]
    bars   = ax.bar(range(len(stages)), vals, color=clrs, edgecolor=C_EDGE, linewidth=0.8, width=0.65)
    ax.set_xticks(range(len(stages))); ax.set_xticklabels(stages)
    for b in bars:
        h = b.get_height()
        ax.text(b.get_x()+b.get_width()/2, h+0.15, f"{h:.2f}M",
                ha='center', fontsize=FS['annot']+1, fontweight='bold')
    ax.set_ylabel("Flows (Millions)", fontsize=FS['label'])
    ax.set_title("Dataset Volume Pipeline — CSE-CIC-IDS2018", fontsize=FS['title'])
    ax.set_ylim(0, 9.8)
    save_fig(fig, "01_dataset_pipeline.png", "dataset")

    # =========================================================================
    # FIGURE 02: Class Balance — Before & After SMOTE  (2-panel combined)
    # =========================================================================
    print("[02] Class balance before/after SMOTE...")
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.0))
    for ax, (b_cnt, a_cnt, ttl) in zip(axes, [
            (3595795, 449005,  "Before SMOTE (Imbalanced ~8:1)"),
            (3595795, 3595795, "After SMOTE (Balanced 1:1)")]):
        total = b_cnt + a_cnt
        bars = ax.bar(["Benign", "Attack"],
                      [b_cnt/1e6, a_cnt/1e6],
                      color=["#BDD7EE","#FCE4D6"], edgecolor=C_EDGE, linewidth=0.9, width=0.48)
        for bar, cnt in zip(bars,[b_cnt,a_cnt]):
            h = bar.get_height()
            ax.text(bar.get_x()+bar.get_width()/2, h+0.08,
                    f"{h:.2f}M\n({cnt/total*100:.1f}%)",
                    ha='center', fontsize=FS['annot'], fontweight='bold')
        ax.set_ylabel("Flows (Millions)", fontsize=FS['label'])
        ax.set_title(ttl, fontsize=FS['title'])
        ax.set_ylim(0, 4.5)
    fig.suptitle("Core-Train Class Distribution", fontsize=FS['title']+1, fontweight='bold', y=1.02)
    save_fig(fig, "02_class_balance.png", "dataset")

    # =========================================================================
    # FIGURE 03: Feature Dimensionality Reduction Pipeline
    # =========================================================================
    print("[03] Feature pipeline...")
    fig, ax = plt.subplots(figsize=(8, 4.0))
    f_stages = ["Raw\n(80)","Cleaned\n(70)","One-Hot\n(69)","MI Filter\n(40)","Proposed\nSHAP (27)"]
    f_vals   = [80, 70, 69, 40, 27]
    f_clrs   = ["#D9E1F2","#BDD7EE","#9DC3E6","#ED7D31","#375623"]
    bars = ax.bar(range(5), f_vals, color=f_clrs, edgecolor=C_EDGE, linewidth=0.9, width=0.55)
    ax.set_xticks(range(5)); ax.set_xticklabels(f_stages)
    for b in bars:
        h = b.get_height()
        ax.text(b.get_x()+b.get_width()/2, h+1.5, f"{int(h)}",
                ha='center', fontsize=FS['label'], fontweight='bold')
    ax.set_ylabel("Feature Dimensions", fontsize=FS['label'])
    ax.set_title("Feature Dimensionality Reduction Pipeline", fontsize=FS['title'])
    ax.set_ylim(0, 100)
    save_fig(fig, "03_feature_pipeline.png", "dataset")

    # =========================================================================
    # FIGURE 04: SHAP Importance — Clean vs Adversarial (top 15)
    # =========================================================================
    print("[04] SHAP importance...")
    top15 = df_shap.sort_values("clean_imp", ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(9, 5.5))
    y_pos = np.arange(len(top15))
    hb = 0.38
    ax.barh(y_pos-hb/2, top15["clean_imp"], height=hb, color="#BDD7EE",
            edgecolor=C_EDGE, linewidth=0.7, label="Clean SHAP")
    ax.barh(y_pos+hb/2, top15["adv_imp"],   height=hb, color="#FCE4D6",
            edgecolor=C_EDGE, linewidth=0.7, label="Adversarial SHAP (PGD)")
    ax.set_yticks(y_pos); ax.set_yticklabels(top15["feature"], fontsize=FS['tick'])
    ax.invert_yaxis()
    ax.set_xlabel("Mean |SHAP Value|", fontsize=FS['label'])
    ax.set_title("Top-15 Features: Clean vs Adversarial SHAP Importance", fontsize=FS['title'])
    ax.legend(fontsize=FS['legend'], loc='lower right')
    save_fig(fig, "04_shap_importance.png", "features")

    # =========================================================================
    # FIGURE 05: Clean Performance — Accuracy & F1 (2-panel combined)
    # =========================================================================
    print("[05] Clean performance...")
    clean_sub = df_comb[df_comb["attack"]=="Clean"].set_index("method")
    m2_keys   = ["Baseline","V9 Defense"]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    for ax, (col, ytitle, ylim) in zip(axes, [
            ("accuracy",    "Accuracy",          (0.88, 0.98)),
            ("f1_weighted", "Weighted F1-Score",  (0.88, 0.98))]):
        means = [clean_sub.loc[m, f"{col}_mean"] for m in m2_keys]
        stds  = [clean_sub.loc[m, f"{col}_std"]  for m in m2_keys]
        labels = [LABEL[m] for m in m2_keys]
        bars = ax.bar(labels, means, yerr=stds, capsize=6,
                      color=["#BDD7EE","#C6EFCE"], edgecolor=C_EDGE, linewidth=0.9, width=0.48)
        for b, mv, sv in zip(bars, means, stds):
            h = b.get_height()
            ax.text(b.get_x()+b.get_width()/2, h+0.003,
                    f"{h*100:.2f}%\n+-{sv*100:.2f}%",
                    ha='center', fontsize=FS['annot'], fontweight='bold')
        ax.set_ylabel(ytitle, fontsize=FS['label'])
        ax.set_ylim(ylim)
        ax.tick_params(axis='x', labelsize=FS['tick'])
    axes[0].set_title("Clean Accuracy", fontsize=FS['title'])
    axes[1].set_title("Clean Weighted F1-Score", fontsize=FS['title'])
    fig.suptitle("Clean Traffic Utility", fontsize=FS['title']+1, fontweight='bold', y=1.02)
    save_fig(fig, "05_clean_performance.png", "performance")

    # =========================================================================
    # FIGURE 06: Robust Accuracy & ASR across attacks (2-panel combined)
    # =========================================================================
    print("[06] Multi-threat robustness...")
    att_sub  = df_comb[~df_comb["attack"].isin(["Clean","Label-Only Boundary Search"])].copy()
    attacks  = sorted(att_sub["attack"].unique())
    x        = np.arange(len(attacks))
    w        = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.0))
    for ax, (col, ytitle) in zip(axes, [
            ("accuracy",           "Robust Accuracy"),
            ("attack_success_rate","Attack Success Rate (ASR)")]):
        for i,(m,c,fc) in enumerate(zip(
                ["Baseline","V9 Defense"],
                [C_BASE,    C_V9],
                ["#BDD7EE", "#C6EFCE"])):
            md    = att_sub[att_sub["method"]==m].set_index("attack").reindex(attacks)
            means = md[f"{col}_mean"].fillna(0).values
            stds  = md[f"{col}_std"].fillna(0).values
            ax.bar(x+(i-0.5)*w, means, yerr=stds, capsize=3,
                   width=w, label=LABEL[m], color=fc, edgecolor=c, linewidth=0.9)
        ax.set_xticks(x)
        ax.set_xticklabels(attacks, rotation=30, ha='right', fontsize=FS['tick']-1)
        ax.set_ylabel(ytitle, fontsize=FS['label'])
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=FS['legend'])
        ax.set_title(ytitle, fontsize=FS['title'])
    fig.suptitle("Multi-Threat Evaluation: Baseline Defense vs Proposed Defense",
                 fontsize=FS['title']+1, fontweight='bold', y=1.01)
    save_fig(fig, "06_attack_robustness.png", "robustness")

    # =========================================================================
    # FIGURE 07: Precision & Recall summary (5 attack conditions, 2-panel)
    # =========================================================================
    print("[07] Precision-Recall comparison...")
    threat_subset = ["Clean","FGSM","PGD","Auto-PGD (APGD-CE)","CW-Linf Margin Attack"]
    x_t  = np.arange(len(threat_subset))
    w_t  = 0.20
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    for ax, col, ytitle in zip(axes, ["precision","recall"], ["Precision","Recall"]):
        for i,(m,fc,ec) in enumerate(zip(
                ["Baseline","V9 Defense"],
                ["#BDD7EE", "#C6EFCE"],
                [C_BASE,    C_V9])):
            md    = df_comb[df_comb["method"]==m].set_index("attack").reindex(threat_subset)
            means = md[f"{col}_mean"].fillna(0).values
            ax.bar(x_t+(i-0.5)*w_t, means, width=w_t,
                   label=LABEL[m], color=fc, edgecolor=ec, linewidth=0.9)
        ax.set_xticks(x_t)
        ax.set_xticklabels(threat_subset, rotation=22, ha='right', fontsize=FS['tick'])
        ax.set_ylabel(ytitle, fontsize=FS['label'])
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=FS['legend'])
        ax.set_title(ytitle, fontsize=FS['title'])
    fig.suptitle("Precision & Recall Across Attack Conditions",
                 fontsize=FS['title']+1, fontweight='bold', y=1.01)
    save_fig(fig, "07_precision_recall.png", "performance")

    # =========================================================================
    # HELPER: Confusion Matrix Plot (light pastel fill)
    # =========================================================================
    def plot_cm(cm, title, fname):
        tn, fp, fn, tp = cm.ravel()
        total = cm.sum()
        fpr   = fp/(fp+tn)*100
        tpr   = tp/(tp+fn)*100
        acc   = (tp+tn)/total*100

        fig, ax = plt.subplots(figsize=(5.8, 5.0))
        # Light pastel colormap
        cmap = matplotlib.colors.LinearSegmentedColormap.from_list(
            "pastel_blue", ["#FFFFFF","#BDD7EE","#2E75B6"])
        im = ax.imshow(cm, cmap=cmap, interpolation='nearest')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_xticks([0,1])
        ax.set_yticks([0,1])
        ax.set_xticklabels(["Pred: Benign","Pred: Attack"], fontsize=FS['tick'])
        ax.set_yticklabels(["True: Benign","True: Attack"], fontsize=FS['tick'])
        ax.set_xlabel("Predicted Label", fontsize=FS['label'])
        ax.set_ylabel("True Label",      fontsize=FS['label'])
        thresh = cm.max() / 2.
        cell_labels = [
            [f"TN = {tn:,}\n{tn/total*100:.1f}%",
             f"FP = {fp:,}\n{fp/total*100:.1f}%\nFPR = {fpr:.1f}%"],
            [f"FN = {fn:,}\n{fn/total*100:.1f}%",
             f"TP = {tp:,}\n{tp/total*100:.1f}%\nTPR = {tpr:.1f}%"]
        ]
        for r in range(2):
            for c in range(2):
                clr = "white" if cm[r,c] > thresh else "#1F3864"
                ax.text(c, r, cell_labels[r][c],
                        ha='center', va='center', fontsize=FS['tick']+1,
                        fontweight='bold', color=clr)
        ax.set_title(f"{title}\nAcc = {acc:.2f}%  |  FPR = {fpr:.2f}%  |  TPR = {tpr:.2f}%",
                     fontsize=FS['title'], pad=10)
        save_fig(fig, fname, "confusion_matrices")

    # =========================================================================
    # FIGURE 08-10: Confusion Matrices — Clean (Baseline / Reference / Proposed)
    # =========================================================================
    print("[08-10] Confusion matrices (clean)...")
    plot_cm(confusion_matrix(y_eval, yhat_b),   "Baseline Defense — Clean",   "08_cm_baseline_clean.png")
    plot_cm(confusion_matrix(y_eval, yhat_p),   "Reference Defense — Clean",  "09_cm_reference_clean.png")
    plot_cm(confusion_matrix(y_eval, yhat_v9),  "Proposed Defense — Clean",   "10_cm_proposed_clean.png")

    # =========================================================================
    # FIGURE 11-12: Confusion Matrices — Proposed Defense under attack
    # =========================================================================
    print("[11-12] Confusion matrices (adversarial)...")
    plot_cm(confusion_matrix(y_eval, yhat_v9_pgd),  "Proposed Defense — Under PGD",  "11_cm_proposed_pgd.png")
    plot_cm(confusion_matrix(y_eval, yhat_v9_fgsm), "Proposed Defense — Under FGSM", "12_cm_proposed_fgsm.png")

    # =========================================================================
    # FIGURE 13: ROC Curves — Clean & PGD (2-panel combined)
    # =========================================================================
    print("[13] ROC curves...")
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.0))
    for ax, (title, prob_pairs) in zip(axes, [
        ("Clean Traffic ROC",
         [("Baseline Defense", pb,  C_BASE),
          ("Proposed Defense", pv9, C_V9)]),
        ("Under PGD Attack ROC",
         [("Baseline Defense", pb_pgd,  C_BASE),
          ("Proposed Defense", pv9_pgd, C_V9)]),
    ]):
        for label, probs, clr in prob_pairs:
            fpr_, tpr_, _ = roc_curve(y_eval, probs)
            roc_auc = auc(fpr_, tpr_)
            ax.plot(fpr_, tpr_, lw=2.0, color=clr,
                    label=f"{label} (AUC={roc_auc:.4f})")
        ax.plot([0,1],[0,1],'--', color='#A0A0A0', lw=1.2)
        ax.set_xlabel("False Positive Rate", fontsize=FS['label'])
        ax.set_ylabel("True Positive Rate",  fontsize=FS['label'])
        ax.set_title(title, fontsize=FS['title'])
        ax.legend(fontsize=FS['legend'], loc='lower right')
    fig.suptitle("Receiver Operating Characteristic (ROC) Curves",
                 fontsize=FS['title']+1, fontweight='bold', y=1.01)
    save_fig(fig, "13_roc_curves.png", "performance")

    # =========================================================================
    # FIGURE 14: Precision-Recall Curves — Clean & PGD (2-panel combined)
    # =========================================================================
    print("[14] PR curves...")
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.0))
    for ax, (title, prob_pairs) in zip(axes, [
        ("Clean Traffic PR",
         [("Baseline Defense", pb,  C_BASE),
          ("Proposed Defense", pv9, C_V9)]),
        ("Under PGD Attack PR",
         [("Baseline Defense", pb_pgd,  C_BASE),
          ("Proposed Defense", pv9_pgd, C_V9)]),
    ]):
        for label, probs, clr in prob_pairs:
            prec_, rec_, _ = precision_recall_curve(y_eval, probs)
            ap = average_precision_score(y_eval, probs)
            ax.plot(rec_, prec_, lw=2.0, color=clr,
                    label=f"{label} (AP={ap:.4f})")
        ax.set_xlabel("Recall",    fontsize=FS['label'])
        ax.set_ylabel("Precision", fontsize=FS['label'])
        ax.set_title(title, fontsize=FS['title'])
        ax.legend(fontsize=FS['legend'], loc='lower left')
    fig.suptitle("Precision-Recall Curves",
                 fontsize=FS['title']+1, fontweight='bold', y=1.01)
    save_fig(fig, "14_pr_curves.png", "performance")

    # =========================================================================
    # FIGURE 15: Epsilon & PGD-Step Sweeps (2-panel combined)
    # =========================================================================
    print("[15] Epsilon + PGD-step sweeps...")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    for ax, df_sw, xcol, xticks, xlabel, title in zip(axes,
        [df_eps,    df_pgd_sw],
        ["epsilon", "pgd_steps"],
        [[0.01,0.03,0.05,0.10],[20,50,100,200]],
        ["Perturbation Budget eps","PGD Iterations (Steps)"],
        ["Robustness vs eps (Epsilon Sweep)","Stability vs PGD Depth"]
    ):
        for m, clr in zip(["Baseline","V9 Defense"],[C_BASE,C_V9]):
            sub = df_sw[df_sw["method"]==m].sort_values(xcol)
            ax.plot(sub[xcol], sub["mean"], marker='o', lw=2.0, color=clr,
                    label=LABEL[m])
            ax.fill_between(sub[xcol], sub["mean"]-sub["std"], sub["mean"]+sub["std"],
                            alpha=0.12, color=clr)
        ax.set_xlabel(xlabel,   fontsize=FS['label'])
        ax.set_ylabel("Accuracy",fontsize=FS['label'])
        ax.set_xticks(xticks)
        ax.legend(fontsize=FS['legend'])
        ax.set_title(title, fontsize=FS['title'])
    fig.suptitle("Attack Strength Sensitivity Analysis",
                 fontsize=FS['title']+1, fontweight='bold', y=1.01)
    save_fig(fig, "15_attack_sweeps.png", "robustness")

    # =========================================================================
    # FIGURE 16: DeepFool + Boundary Search Diagnostics (2-panel combined)
    # =========================================================================
    print("[16] DeepFool + Boundary search...")
    sub_df = df_deepfool.dropna(subset=["median_Linf_flips_mean"]).set_index("method")
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.6))

    # Left: DeepFool L_inf percentiles
    ax = axes[0]
    m2_keys_diag = ["Baseline","V9 Defense"]
    p10 = [float(sub_df.loc[m,"p10_Linf_flips_mean"])    for m in m2_keys_diag]
    med = [float(sub_df.loc[m,"median_Linf_flips_mean"]) for m in m2_keys_diag]
    p90 = [float(sub_df.loc[m,"p90_Linf_flips_mean"])    for m in m2_keys_diag]
    xlbls = [LABEL[m] for m in m2_keys_diag]
    x_d  = np.arange(2)
    ax.bar(x_d, med, width=0.45, color=["#BDD7EE","#C6EFCE"], edgecolor=C_EDGE, lw=0.9, label="Median Linf")
    ax.errorbar(x_d, med,
                yerr=[np.array(med)-np.array(p10), np.array(p90)-np.array(med)],
                fmt='none', color='#1F3864', capsize=7, lw=1.8, label="10th-90th Pct")
    ax.axhline(0.05, color='#C00000', lw=1.8, linestyle='--', label="Nominal eps = 0.05")
    ax.set_xticks(x_d); ax.set_xticklabels(xlbls, fontsize=FS['tick'])
    ax.set_ylabel("L_inf Distortion", fontsize=FS['label'])
    ax.set_title("DeepFool Perturbation Distance\nto Decision Boundary", fontsize=FS['title'], pad=10)
    ax.legend(fontsize=FS['legend'])

    # Right: Label-Only Boundary Search
    ax = axes[1]
    b_rows = []
    for s in [42,123,2024]:
        fp = SEEDS_DIR/str(s)/"attacks"/"boundary_search.csv"
        if fp.exists(): b_rows.append(pd.read_csv(fp))
    df_bs   = pd.concat(b_rows, ignore_index=True)
    agg_bs  = df_bs.groupby("method")[["initialization_success_rate","within_epsilon_success_rate"]].mean()
    init_r  = [float(agg_bs.loc[m,"initialization_success_rate"])*100  for m in m2_keys_diag]
    weps_r  = [float(agg_bs.loc[m,"within_epsilon_success_rate"])*100  for m in m2_keys_diag]
    wb = 0.30
    ax.bar(x_d-wb/2, init_r, width=wb, color="#BDD7EE", edgecolor=C_EDGE, lw=0.9, label="Init. Success Rate")
    ax.bar(x_d+wb/2, weps_r, width=wb, color="#C6EFCE", edgecolor=C_EDGE, lw=0.9, label="Within-eps Rate")
    ax.set_xticks(x_d); ax.set_xticklabels(xlbls, fontsize=FS['tick'])
    ax.set_ylabel("Success Rate (%)", fontsize=FS['label'])
    ax.set_title("Label-Only Boundary Search\nDiagnostic", fontsize=FS['title'], pad=10)
    ax.legend(fontsize=FS['legend'])

    fig.suptitle("Geometric Decision Boundary Analysis",
                 fontsize=FS['title']+1, fontweight='bold')
    fig.tight_layout(rect=[0, 0, 1, 0.93])   # leave top 7% for suptitle
    save_fig(fig, "16_boundary_diagnostics.png", "diagnostics")

    # =========================================================================
    # FIGURE 17: Transfer ASR
    # =========================================================================
    print("[17] Transfer ASR...")
    t_rows = []
    for s in [42,123,2024]:
        tp = SEEDS_DIR/str(s)/"attacks"/"transfer.csv"
        if tp.exists():
            df_t = pd.read_csv(tp)
            for _,r in df_t.iterrows():
                t_rows.append({"target":r["target"],"asr":r["transfer_asr"]})
    df_tr  = pd.DataFrame(t_rows)
    grp    = df_tr.groupby("target")["asr"].agg(["mean","std"])
    t_means= [grp.loc[m,"mean"]*100 for m in m2_keys_diag]
    t_stds = [grp.loc[m,"std"]*100  for m in m2_keys_diag]
    fig, ax = plt.subplots(figsize=(6.0, 4.5))
    bars = ax.bar(xlbls, t_means, yerr=t_stds, capsize=6,
                  color=["#BDD7EE","#C6EFCE"], edgecolor=C_EDGE, lw=0.9, width=0.48)
    for b, mv, sv in zip(bars, t_means, t_stds):
        h = b.get_height()
        ax.text(b.get_x()+b.get_width()/2, h+0.5, f"{mv:.2f}%\n+-{sv:.2f}%",
                ha='center', fontsize=FS['annot'], fontweight='bold')
    ax.set_ylabel("Transfer ASR (%)", fontsize=FS['label'])
    ax.set_title("Black-Box Transferability from\nSurrogate WiderNet", fontsize=FS['title'])
    ax.set_ylim(0, max(t_means)*1.4)
    save_fig(fig, "17_transfer_asr.png", "diagnostics")

    # =========================================================================
    # FIGURE 18: Cross-Seed Replication Variability
    # =========================================================================
    print("[18] Seed variability...")
    seeds  = [42, 123, 2024]
    x_s    = np.arange(3)
    ws     = 0.20
    fig, ax = plt.subplots(figsize=(9, 4.8))
    fill_cleans = ["#BDD7EE","#C6EFCE"]
    fill_pgds   = ["#DEEAF1","#E2EFDA"]
    for i, m in enumerate(["Baseline","V9 Defense"]):
        clean_accs = [seed_dfs[s][(seed_dfs[s]["method"]==m)&(seed_dfs[s]["attack"]=="Clean")]["accuracy"].values[0]*100 for s in seeds]
        pgd_accs   = [seed_dfs[s][(seed_dfs[s]["method"]==m)&(seed_dfs[s]["attack"]=="PGD")]["accuracy"].values[0]*100   for s in seeds]
        ax.bar(x_s+(i*2-1.5)*ws, clean_accs, width=ws,
               label=f"{LABEL[m]} — Clean",
               color=fill_cleans[i], edgecolor=C_THREE[i*2], lw=0.9)
        ax.bar(x_s+(i*2-0.5)*ws, pgd_accs,   width=ws,
               label=f"{LABEL[m]} — PGD-100",
               color=fill_pgds[i],   edgecolor=C_THREE[i*2], lw=0.9, hatch='//')
    ax.set_xticks(x_s); ax.set_xticklabels([f"Seed {s}" for s in seeds], fontsize=FS['tick'])
    ax.set_ylabel("Accuracy (%)", fontsize=FS['label'])
    ax.set_title("Cross-Seed Replication Variability (Seeds 42, 123, 2024)", fontsize=FS['title'])
    ax.set_ylim(60, 102)
    ax.legend(fontsize=FS['legend'], ncol=2)
    save_fig(fig, "18_seed_variability.png", "training")

    # =========================================================================
    # FIGURE 19: Training Budget & Phase Breakdown (2-panel combined)
    # =========================================================================
    print("[19] Training budget...")
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    # Left: Epoch budget
    ax = axes[0]
    bar_labels = [LABEL["Baseline"], LABEL["V9 Defense"]]
    epochs = [50, 50]
    bars = ax.bar(bar_labels, epochs, color=["#BDD7EE","#C6EFCE"],
                  edgecolor=C_EDGE, lw=0.9, width=0.48)
    for b in bars:
        h = b.get_height()
        ax.text(b.get_x()+b.get_width()/2, h+1, f"{int(h)} Epochs",
                ha='center', fontsize=FS['label'], fontweight='bold')
    ax.set_ylabel("Training Epochs", fontsize=FS['label'])
    ax.set_title("Standardized Epoch Budget", fontsize=FS['title'])
    ax.set_ylim(0, 65)

    # Right: Proposed Defense phase breakdown
    ax = axes[1]
    phases = ["Phase 1\n(10 Clean Epochs)", "Phase 2\n(40 Online PGD Epochs)"]
    phase_vals = [10, 40]
    phase_clrs = ["#BDD7EE","#375623"]
    bars = ax.bar(phases, phase_vals, color=phase_clrs, edgecolor=C_EDGE, lw=0.9, width=0.48)
    for b in bars:
        h = b.get_height()
        ax.text(b.get_x()+b.get_width()/2, h+0.8,
                f"{int(h)} Ep ({h/50*100:.0f}%)",
                ha='center', fontsize=FS['label'], fontweight='bold',
                color='white' if h==40 else C_EDGE)
    ax.set_ylabel("Epoch Allocation", fontsize=FS['label'])
    ax.set_title("Proposed Defense: Two-Phase\nTraining Strategy", fontsize=FS['title'])
    ax.set_ylim(0, 55)

    fig.suptitle("Training Configuration Summary",
                 fontsize=FS['title']+1, fontweight='bold', y=1.01)
    save_fig(fig, "19_training_budget.png", "training")

    # =========================================================================
    # FIGURE 20: ASR — 4 Representative Attacks x 3 Models (Grouped Bar Chart)
    # FGSM (Single-Step) | PGD-100 (Iterative) | Square (Black-Box) | Transfer PGD (Surrogate)
    # =========================================================================
    print("[20] ASR — 4 attacks x 3 models...")

    methods_3 = ["Baseline", "Paper Defense", "V9 Defense"]
    fill_3    = ["#BDD7EE",  "#FCE4D6",       "#C6EFCE"]
    edge_3    = [C_BASE,     C_PAPER,          C_V9]
    label_3   = [LABEL["Baseline"], LABEL["Paper Defense"], LABEL["V9 Defense"]]

    # --- Build Transfer PGD means/stds from transfer.csv files ---
    t_rows = []
    for s in [42, 123, 2024]:
        tf = SEEDS_DIR/str(s)/"attacks"/"transfer.csv"
        if tf.exists():
            t_rows.append(pd.read_csv(tf))
    df_tr = pd.concat(t_rows, ignore_index=True)
    tr_grp = df_tr.groupby("target")["transfer_asr"].agg(["mean","std"])

    # --- 4 attack definitions: (display label, csv_attack_key, use_transfer) ---
    attack_cfg = [
        ("FGSM\n(Single-Step)",       "FGSM",               False),
        ("PGD-100\n(Iterative)",       "PGD",                False),
        ("Square\n(Black-Box)",        "Square-style Attack", False),
        ("Transfer PGD\n(Surrogate)",  None,                 True),
    ]
    xtick_labels = [a[0] for a in attack_cfg]
    n_att = len(attack_cfg)
    w_grp = 0.22
    x_pos = np.arange(n_att)
    asr_idx = df_comb.set_index(["method","attack"])

    fig, ax = plt.subplots(figsize=(10, 5.8))

    for i, (m, fc, ec, lbl) in enumerate(zip(methods_3, fill_3, edge_3, label_3)):
        offset = (i - 1) * w_grp
        means, stds = [], []
        for _, csv_key, is_transfer in attack_cfg:
            if is_transfer:
                try:
                    means.append(float(tr_grp.loc[m, "mean"]))
                    stds.append( float(tr_grp.loc[m, "std"]))
                except KeyError:
                    means.append(0.0); stds.append(0.0)
            else:
                try:
                    row = asr_idx.loc[(m, csv_key)]
                    means.append(float(row["attack_success_rate_mean"]))
                    stds.append( float(row["attack_success_rate_std"]))
                except KeyError:
                    means.append(0.0); stds.append(0.0)

        bars = ax.bar(x_pos + offset, means, width=w_grp,
                      color=fc, edgecolor=ec, linewidth=1.0,
                      label=lbl, yerr=stds, capsize=5,
                      error_kw=dict(elinewidth=1.4, ecolor=ec))
        # Bold percentage labels above each bar
        for b, mv, sv in zip(bars, means, stds):
            ax.text(b.get_x() + b.get_width()/2,
                    b.get_height() + sv + 0.012,
                    f"{mv*100:.1f}%",
                    ha='center', va='bottom',
                    fontsize=FS['annot'], fontweight='bold', color=ec)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(xtick_labels, fontsize=FS['tick']+1)
    ax.set_ylabel("Attack Success Rate (ASR)", fontsize=FS['label'])
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
    ax.set_ylim(0, 0.42)
    ax.set_title("Attack Success Rate: Representative Threat Categories",
                 fontsize=FS['title'], pad=12)
    ax.legend(fontsize=FS['legend'], loc='upper right',
              framealpha=0.9, edgecolor='#CCCCCC')
    # Reference line at 10%
    ax.axhline(0.10, color='#888888', lw=1.2, linestyle=':', alpha=0.7)
    ax.text(n_att - 0.05, 0.103, "10% ASR threshold",
            fontsize=FS['annot']-1, color='#666666', va='bottom', ha='right')
    fig.tight_layout()
    save_fig(fig, "20_asr_all_models.png", "robustness")


    # -- Export tables --------------------------------------------------------
    df_comb.to_csv(     SUBDIRS["tables"]/"table_all_attacks.csv",       index=False)
    df_eps.to_csv(      SUBDIRS["tables"]/"table_epsilon_sweep.csv",     index=False)
    df_pgd_sw.to_csv(   SUBDIRS["tables"]/"table_pgd_step_sweep.csv",    index=False)
    df_deepfool.to_csv( SUBDIRS["tables"]/"table_deepfool_budget.csv",   index=False)
    df_budget.to_csv(   SUBDIRS["tables"]/"table_training_budget.csv",   index=False)
    df_shap.to_csv(     SUBDIRS["tables"]/"table_shap_importance.csv",   index=False)

    print("\n" + "="*60)
    print("ALL 20 IEEE-READY FIGURES GENERATED")
    print(f"-> {ANALYTICS_DIR.resolve()}")
    print("="*60)

if __name__ == "__main__":
    main()

# ============================================================
# INLINE DISPLAY — show all saved figures in this cell
# ============================================================
import os
from IPython.display import display as ipy_display, Image as IpyImage

print("\n=== INLINE FIGURES ===")
for subfolder in ["dataset","features","performance","robustness",
                  "confusion_matrices","diagnostics","training"]:
    sub_path = ANALYTICS_DIR / subfolder
    if sub_path.exists():
        pngs = sorted(sub_path.glob("*.png"))
        if pngs:
            print(f"\n-- {subfolder} --")
            for p in pngs:
                print(f"  {p.name}")
                ipy_display(IpyImage(str(p), width=700))
print("\nAll figures displayed above.")
